# PKG Attrition v13 · Model 5

**Payment Knowledge Graph (PKG) · Treasury Management · DSI**

Model 4, rerun on customer metrics v3, is the base: same target, rows, folds, models, list and checks. Model 5 asks
three new questions.

| | Question | How |
|---|---|---|
| 1 | Do the client's **counterparties** add warning? | four feature families from the payment network (§3b): the health of the client's external counterparties (`pkg_cpty_metrics_monthly`); the metrics and deposits of its PNC counterparties, including whether their money has left; its dollars through **payment processors, brokerages and banks** (by name outside PNC, by NAICS inside); the **NAICS mix** of the PNC customers it pays (2-, 3-, 4- and 6-digit). `pkg_attr_cpty_features_monthly` joins too when it exists |
| 2 | Does the model hold on **recent months**? | the rolling folds continue past the last fully labelled month into 2026; each month's models see only outcomes known at that month; outcomes so far are compared with the test months cut to the **same follow-up** |
| 3 | **Which bank** does the money go to? | a ranking model over the client's other banks (`pkg_cust_fi_month`): trained on earlier departures, checked on the test-period departures, applied to the current list |

The network features are judged by **ablation**: a tree on model 4's features and a tree on all features, fitted in
the same folds on the same rows. Ids in the report are power ids (`SHOW_PWR_IDS`).

Sources: the deposit table, `pkg_cust_metrics_monthly_v3`, `pkg_cust_pair_month_v3`, `pkg_cpty_metrics_monthly`,
`pkg_cust_dim`, `pkg_cust_fi_month`, optionally `pkg_attr_cpty_features_monthly`. No table is written. The monthly
network features are cached as files under the cdi_tm location (`_tmp_pkg_attr_m5_cache`) with a signature, so a
rerun reuses them; the first run reads the pair table once per month.

### Pre-registered predictions — scored in §12

| # | Prediction | Confidence |
|---|---|---|
| P1 | Pooled AUC of the full tree ≥ the model-4-features tree + 0.005 | 45% |
| P2 | Dollars reached at 1,000 calls, α = 0.5: full tree ≥ 1.03 × model-4-features tree | 40% |
| P3 | At least one network feature in the top 25 (tree SHAP) | 75% |
| P4 | A processor or NAICS-mix feature ranks leavers on its own with AUC ≥ 0.55 | 45% |
| P5 | 2026 precision (1,000 calls, α = 0.5) within ±30% of the test months' precision at the same follow-up | 60% |
| P6 | Destination bank named first correctly for ≥ 50% of departures one month before the exit (departures with an identified destination) | 50% |
| P7 | The destination model beats "the client's current top bank" by ≥ 5 points (top-1), three months before the exit | 45% |
| P8 | ≥ 40% of the departing normal balance in the test period has an identifiable destination bank | 40% |

In [ ]:
# =====================================================================
# 0 · CONFIG
# =====================================================================
DB         = "bdahd01p_dlcdi1_cdi_tm"
DEP_TABLE  = "dsihd01p_dsi.lap_dsi_universe_optimized"          # daily: account × leg × business day
MET_TABLE  = f"{DB}.pkg_cust_metrics_monthly_v3"
DICT_TABLE = f"{DB}.pkg_cust_metrics_dictionary_v3"
PAIR_TABLE = f"{DB}.pkg_cust_pair_month_v3"        # customer × dir × scope × counterparty (ext: ent_id of the cpty table; int: mdm_id)
CPTY_TABLE = f"{DB}.pkg_cpty_metrics_monthly"      # counterparty's view: _in = received from PNC customers
CPTY_DICT  = f"{DB}.pkg_cpty_metrics_dictionary"   # labels only; optional
DIM_TABLE  = f"{DB}.pkg_cust_dim"                  # one row per mdm_id: power id, family, NAICS
FI_TABLE   = f"{DB}.pkg_cust_fi_month"             # TM customer × dir × canonical bank (destination model)
CPF_TABLE  = f"{DB}.pkg_attr_cpty_features_monthly"   # optional (pkg_attr_cpty_features.ipynb); skipped when absent
OUT_DIR    = "../metrics/attrition/v13/m5"
DATE_START, DATE_END = "2024-01-01", "2026-08-31"                # nothing before 2024 is trusted
SHUFFLE_PARTITIONS = 800

# ── status codes: two vocabularies in one column (settled in v2) ─────
STATUS_CLASS = {"01": "open", "99": "open", "09": "open", "O": "open",
                "03": "inactive", "05": "dormant", "06": "escheat", "12": "closing",
                "07": "closed", "08": "closed", "C": "closed"}
LIVE_CLASSES = {"open", "inactive", "dormant", "escheat", "closing"}
PRICING = ["earnings_credit_rate", "curr_int_rate"]              # used if present

# ── definitions ──────────────────────────────────────────────────────
H          = 6          # horizon: does the money leave within the next 6 months?
CONFIRM_M  = 2          # a full exit is confirmed after 2 observed non-live months
NORM_WIN, NORM_MIN_OBS = 12, 3    # normal balance: median of up to 12 prior live months, needs 3
MIN_HIST   = 3          # months observed before a client-month can be scored
MSH_FRAC, MSH_FLOOR = 0.50, 10_000     # money still here: ≥ half of normal, normal ≥ $10k
BAL_EXIT_FRAC, BAL_EXIT_HOLD = 0.05, 3 # balance exit: < 5% of normal for 3 months, accounts open
CHURN_WARM = 6          # months of payment history before the churn-state metrics are used

# ── network features (§3b) ───────────────────────────────────────────
NET_VERSION = "m5-net-v1"   # part of the cache signature: bump when the monthly network features change
NET_CACHE   = None          # folder for the monthly cache; None = <cdi_tm location>/_tmp_pkg_attr_m5_cache
USE_CPF     = True          # join pkg_attr_cpty_features_monthly when it exists
NB_BASE_M   = 6             # PNC counterparties weighted by the client's dollars with them over the previous 6 months
KNOWN_LAG   = max(CONFIRM_M, BAL_EXIT_HOLD) - 1   # a counterparty's exit counts only once it is confirmed (no look-ahead)
NAICS_TOPK  = {2: 10, 3: 8, 4: 8, 6: 8}           # NAICS groups tracked per level (out-dollars over the labelled months)
NAICS_IN_LEVELS = (2,)      # levels also tracked for money received
NEW_NAICS_M = 6             # "new industry" = a NAICS-4 group the client paid nothing in the previous 6 months
# counterparties outside PNC by name (ent_name, upper case, punctuation → space); PNC counterparties by NAICS prefix
PROC_RE = {
    "mm":      r"\b(PAYPAL|STRIPE|TRANSFERWISE|WISE US|PAYONEER|BILL COM|MELIO|TIPALTI|AVIDXCHANGE|AVID XCHANGE|DWOLLA|VENMO|"
               r"SQUARE INC|BLOCK INC|CASH APP|COINBASE|KRAKEN|CIRCLE INTERNET|PLAID|MODERN TREASURY|AIRWALLEX|REMITLY|"
               r"WESTERN UNION|MONEYGRAM|FLYWIRE|PAYMENTUS|BLUESNAP|AUTHORIZE NET|INTUIT PAYMENTS|QUICKBOOKS|BREX|"
               r"MERCURY TECHNOLOGIES|RELAY FINANCIAL)\b",
    "payroll": r"\b(ADP|PAYCHEX|GUSTO|PAYLOCITY|PAYCOR|DAYFORCE|CERIDIAN|PAYCOM|TRINET|JUSTWORKS|INSPERITY|RIPPLING|ISOLVED|UKG)\b",
    "card":    r"\b(VISA|MASTERCARD|MASTER CARD|AMERICAN EXPRESS|AMEX|DISCOVER|FISERV|FIRST DATA|CLOVER|WORLDPAY|ELAVON|"
               r"GLOBAL PAYMENTS|HEARTLAND PAYMENT|TSYS|PAYMENTECH|ADYEN|SHIFT4|NUVEI|MERCHANT SERVICES|MERCHANT SVCS)\b",
    "invest":  r"\b(SCHWAB|FIDELITY INVESTMENTS|FIDELITY BROKERAGE|NATIONAL FINANCIAL SERVICES|VANGUARD|MORGAN STANLEY|MERRILL|"
               r"E ?TRADE|AMERITRADE|INTERACTIVE BROKERS|ROBINHOOD|PERSHING|RAYMOND JAMES|EDWARD JONES|LPL FINANCIAL|"
               r"GOLDMAN SACHS|BLACKROCK|JANNEY|STIFEL|APEX CLEARING|TREASURY ?DIRECT|TREAS DRCT)\b",
}                           # "bank": counterparties the name typer classed as a bank (pair o_class)
PROC_NAICS = {"mm": ("522320", "522390"), "payroll": ("541214",), "invest": ("523",), "bank": ("5221",)}
CP_SIDE_COLS = ["retention", "lost_amt_share", "new_amt_share", "rec_broken_amt_share", "dep_wavg", "top1_share", "n_cust"]
CP_FULL_COLS = ["hub_tier", "top_fi_share", "fi_lost", "two_way_ratio"]
NB_SIDE_COLS = ["retention", "lost_amt_share", "rec_broken_amt_share"]                 # PNC counterparties' own metrics, matching side
NB_FIX_COLS  = ["selfpay_share_out", "fi_rec_broken_net_share_out", "pnc_fi_share_out"]
CPF_RE   = r"^(own_ext_amt|own_ext_n|own_known_n|mig_n|mig_amt|mig_share|mig_dual_n|rb_|std_obs_amt_share|own_chg|partner_chg|padj|nbr_)"
CPF_SKIP = {"window_months", "mig_lookback_months", "n_mdm"}

# ── training ─────────────────────────────────────────────────────────
NEG_RATE   = 0.15       # share of non-leaving client-months kept for training (weight 1/NEG_RATE)
SEED       = 13
L2_GRID    = [1.0, 10.0, 100.0, 1000.0]   # ridge penalty (summed log-likelihood), chosen on the first fold's validation
CLIP_Q     = (0.005, 0.995)
CORR_MAX   = 0.95       # drop the later of two features correlated above this (model-4 features are kept first)
TEST_FIRST = "2025-04"  # first test month; the last is the last month with a fully observed outcome
FORWARD    = True       # also score every later month (2026) with models fitted only on outcomes known then
ABLATION   = True       # a second tree on model 4's features only, same folds and rows
ENGINE     = "auto"     # "auto" | "xgboost" | "lightgbm" | "sklearn" | "off"
HOLD_FRAC  = 0.20       # share of clients held out inside the training window (tuning, early stopping, calibration)
CAL_RECENT = 3          # calibrate the tree on held-out clients from the most recent labelled months only
P_CAP      = 0.95       # calibrated probabilities are capped here
GBM_GRID   = [4, 6, 8]  # tree depth tried on the first fold's validation
GBM_ETA, GBM_MAX_TREES, GBM_EARLY = 0.05, 3000, 100

# ── queue and value ──────────────────────────────────────────────────
CALLS      = [100, 250, 500, 1000, 2000]   # calls per month
ALPHAS     = [0.0, 0.5, 1.0]               # score = p × normal balance^α
SAVE_RATES = [0.01, 0.05, 0.10, 0.20]      # illustrative; no call cost anywhere
K_MAIN, A_MAIN = 1000, 0.5
BANDS      = [(0, 1e5, "< $100k"), (1e5, 1e6, "$100k–1m"), (1e6, 1e7, "$1m–10m"), (1e7, float("inf"), "$10m+")]

# ── signal timing and examples ───────────────────────────────────────
N_TIMING   = 30         # features traced month by month before exit
LIFT_BAR   = 1.5        # a signal "fires" when leavers cross the stayers' 10th/90th percentile ≥ 1.5× as often
EVENT_PRE  = 12
MAX_EVENT_CUST = 15_000 # per group, for the event-time pulls
N_SHAP_CUST = 5_000     # per group, for the model view
LATE_FEATS = ["dep_eom_vs_avg", "dep_d1"]   # drain already in progress: late warnings
N_AUDIT    = 15         # largest departures checked, from the whole window and from the test months each
TOP_EXCL   = [1, 5]     # headline dollars also shown without the largest 1 and 5 departing clients
CAP_CREDIT = 250e6      # headline also shown with each client's credit capped here
N_TRIG     = 4          # trigger features shown per example client
STEADY_M   = 12         # months of steady balance (≥ 10% of normal every month) required before an example's alert
SHOW_PWR_IDS = True     # report and tables show power ids (False: one-way hashes, full ids only in *_FULL_IDS.csv)

# ── destination bank (§10b) ──────────────────────────────────────────
DEST_MIN   = 0.10       # a bank is the destination when its excess dollars (last 4 months vs its own months −9…−4) ≥ 10% of normal
DEST_LEADS = (1, 3)     # predicted 1 and 3 months before the money left
DEST_TOPG  = 15         # the largest competitor banks are always candidates
VIA_GROUPS = ("mm", "invest")   # no own-name bank: did outflows to money-movement processors or brokerages rise instead?
N_LIST_SHOW = 25        # rows of the current list shown in the report

In [ ]:
# =====================================================================
# 1 · SETUP — shims, session, helpers
# =====================================================================
import numpy as np, warnings
for _a, _t in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _a): setattr(np, _a, _t)
if "bool" not in np.__dict__: np.bool = np.bool_
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"): pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):    pd.Series.iteritems = pd.Series.items
for _c in (ResourceWarning, DeprecationWarning, FutureWarning, RuntimeWarning):
    warnings.filterwarnings("ignore", category=_c)

import os, re, time, json, base64, hashlib, pathlib, html as _html
from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")            # features are cast to float before collecting
spark.conf.set("spark.sql.execution.arrow.pyspark.fallback.enabled", "true")
DISK = StorageLevel.DISK_ONLY
OUT = pathlib.Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figdata"; FIG.mkdir(exist_ok=True)
def figdata(name, df):
    """Data behind a figure, so the report can be rebuilt later without rerunning."""
    df.to_csv(FIG / f"{name}.csv", index=False)
T0 = time.time()
R = {}          # every number the report uses
CHARTS = {}     # chart name → file
CHECKS = []

# ── one consolidated output per cell ─────────────────────────────────
def _fmt_col(s, kind):
    if kind == "pct":   return s.map(lambda v: "—" if pd.isna(v) else f"{v:.1%}")
    if kind == "money": return s.map(lambda v: "—" if pd.isna(v) else f"${v:,.0f}")
    if pd.api.types.is_integer_dtype(s): return s.map(lambda v: f"{v:,}")
    if pd.api.types.is_float_dtype(s):   return s.map(lambda v: "—" if pd.isna(v) else f"{v:,.3f}")
    return s.map(lambda v: "—" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))

def fmt_df(df, pct=(), money=()):
    d = df.copy()
    for c in d.columns:
        d[c] = _fmt_col(d[c], "pct" if c in pct else "money" if c in money else None)
    return d

class Block:
    """Collects every table and note of a cell and displays them once, as one image-friendly output."""
    def __init__(self, title):
        # tex2jax_ignore / mathjax_ignore: JupyterLab must not read text between two $ signs as a formula
        self.h = [f"<div class='tex2jax_ignore mathjax_ignore' style='font-family:Arial,sans-serif;font-size:12px'>"
                  f"<h3 style='margin:2px 0 6px'>{title}</h3>"]
    def table(self, df, title, pct=(), money=(), save=None, max_rows=80):
        if save: df.to_csv(OUT / f"{save}.csv", index=False)
        d = fmt_df(df.head(max_rows), pct, money)
        self.h.append(f"<p style='margin:8px 0 2px'><b>{title}</b> <span style='color:#777'>[{len(df):,} rows]</span></p>" +
                      d.to_html(index=False, border=0, classes="t").replace(
                          "<table", "<table style='border-collapse:collapse;font-size:11px' cellpadding='3'"))
        return self
    def kv(self, pairs, title):
        return self.table(pd.DataFrame([{"item": k, "value": v if isinstance(v, str) else
                          f"{v:,}" if isinstance(v, (int, np.integer)) else f"{v:,.4f}" if isinstance(v, float) else str(v)}
                                        for k, v in pairs]), title)
    def note(self, s):
        self.h.append(f"<p style='margin:4px 0;color:#333'>{s}</p>"); return self
    def check(self, item, status, detail):
        CHECKS.append({"item": item, "status": status, "detail": detail})
        col = {"READY": "#2e7d32", "FIX": "#e65100", "BLOCKER": "#b71c1c", "INFO": "#555"}[status]
        return self.note(f"<span style='color:{col};font-weight:bold'>[{status}]</span> {item}: {detail}")
    def show(self, t0=None):
        if t0: self.note(f"<span style='color:#999'>wall {time.time() - t0:,.0f}s</span>")
        display(HTML("".join(self.h) + "</div>"))

def savefig(fig, name):
    p = OUT / f"{name}.png"; fig.savefig(p, dpi=120, bbox_inches="tight"); plt.close(fig); CHARTS[name] = p

def topd(df):
    for c, t in df.dtypes:
        if t.startswith("decimal"): df = df.withColumn(c, F.col(c).cast("double"))
    return df.toPandas()

def nz(c):
    s = F.trim(F.col(c).cast("string")); return F.when(s == "", F.lit(None)).otherwise(s)

# ── months: offset t from the first month only ───────────────────────
def ym_idx(ym): y, m = map(int, ym[:7].split("-")); return y * 12 + m - 1
M0 = ym_idx(DATE_START)
def t_of(ym): return ym_idx(ym) - M0
def ym_of(t): i = M0 + int(t); return f"{i // 12:04d}-{i % 12 + 1:02d}"
def t_col(c): return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int") - 1 - F.lit(M0)).cast("int")
assert t_of(DATE_START) == 0 and ym_of(0) == DATE_START[:7]

def cid(x):
    """Display id: one-way hash. Full ids are only ever written to *_FULL_IDS.csv."""
    return hashlib.sha1(str(x).encode()).hexdigest()[:10]

def auc(y, s):
    """Rank AUC with ties averaged; y in {0,1}."""
    y = np.asarray(y, float); s = np.asarray(s, float); ok = ~np.isnan(s); y, s = y[ok], s[ok]
    n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

# ── file system (cache of the monthly network features), signatures ──
HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def rm(p):
    if exists(p): fs_for(p).delete(JPath(p), True)
def mv(src, dst):
    """Move a finished staging folder into place (the destination is replaced)."""
    rm(dst); fs = fs_for(src); fs.mkdirs(JPath(dst).getParent())
    if not fs.rename(JPath(src), JPath(dst)): raise RuntimeError(f"rename failed: {src} -> {dst}")
def write_json(p, obj):
    o = fs_for(p).create(JPath(p), True); o.write(bytearray(json.dumps(obj, sort_keys=True, default=str).encode("utf-8"))); o.close()
def read_json(p):
    if not exists(p): return None
    rd = spark._jvm.java.io.BufferedReader(spark._jvm.java.io.InputStreamReader(fs_for(p).open(JPath(p)), "UTF-8"))
    lines, line = [], rd.readLine()
    while line is not None: lines.append(line); line = rd.readLine()
    rd.close()
    try: return json.loads("\n".join(lines))
    except ValueError: return None
def path_fp(p):
    """Fingerprint of a folder: data files, bytes, latest modification (names starting with _ or . skipped)."""
    if not exists(p): return "missing"
    it, n, size, mt = fs_for(p).listFiles(JPath(p), True), 0, 0, 0
    base = str(fs_for(p).makeQualified(JPath(p)))
    while it.hasNext():
        st = it.next(); rel = str(st.getPath())[len(base):]
        if any(x.startswith(("_", ".")) for x in rel.split("/") if x): continue
        n, size, mt = n + 1, size + st.getLen(), max(mt, st.getModificationTime())
    return f"{n}:{size}:{mt}"
def table_exists(t):
    try: spark.table(t).schema; return True
    except Exception: return False
def table_location(t):
    try:
        for r in spark.sql(f"DESCRIBE FORMATTED {t}").collect():
            if str(r[0]).strip().lower() == "location": return str(r[1]).strip().rstrip("/")
    except Exception: pass
    return None
def part_fp(t, month):
    """Fingerprint of one month partition (standard layout first, then the metastore's partition location)."""
    loc = table_location(t)
    if loc and exists(f"{loc}/month={month}"): return path_fp(f"{loc}/month={month}")
    try:
        for r in spark.sql(f"DESCRIBE FORMATTED {t} PARTITION (month='{month}')").collect():
            if str(r[0]).strip().lower() == "location": return path_fp(str(r[1]).strip())
    except Exception: pass
    return "unknown"
def md5(obj): return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:12]
LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if str(r[0]).lower() == "location"][0].rstrip("/")
def disp(x):
    """Id shown in tables and the report: the power id, or its one-way hash when SHOW_PWR_IDS is off."""
    return str(x) if SHOW_PWR_IDS else cid(x)

# ── tree engine ──────────────────────────────────────────────────────
def _engine():
    order = ["xgboost", "lightgbm", "sklearn"] if ENGINE == "auto" else ([] if ENGINE == "off" else [ENGINE])
    for e in order:
        try:
            importlib.import_module(e if e != "sklearn" else "sklearn.ensemble"); return e
        except Exception: pass
    return None
import importlib
GBM = _engine()
print(f"Spark {spark.version} | window {DATE_START} … {DATE_END} | tree engine: {GBM or 'none (simple model only)'} "
      f"| cpus {os.cpu_count()} | outputs → {OUT.resolve()}")

## §2 Deposit data → monthly customer panel, targets, deposit features

Legs are summed per account-day; balances are rolled to customer-day, then month (average, minimum, maximum,
month-end). Every customer gets a complete month grid from its first month, so a month with no rows reads as
non-live with zero balance. All windows look backwards only.

**Target** (defined and argued in the report, §4): *the money leaves within the next 6 months*, the earlier of
a **full exit** (every account non-live, confirmed by 2 observed non-live months) and a **balance exit** (balance
below 5% of normal for 3 months while accounts stay open).

In [ ]:
# =====================================================================
# 2 · DEPOSIT PANEL, TARGETS, DEPOSIT FEATURES                  [OUTPUT BLOCK 2]
# =====================================================================
t0 = time.time(); B = Block("§2 · Deposit data → monthly customer panel")
raw = spark.table(DEP_TABLE)
REQ = ["acct_full_acct_id", "sub_product_cd", "edw_tda_load_dt", "balance", "acct_status", "deposit_family",
       "opened_dt", "closed_dt", "cust_pwr_id", "rltn_pwr_id"]
miss = [c for c in REQ if c not in raw.columns]
if miss: raise RuntimeError(f"deposit table lacks {miss}")
PRC = [c for c in PRICING if c in raw.columns]
TIEB = [c for c in ["bdh_hdfs_load_ts", "cod_hdfs_load_ts"] if c in raw.columns]
SMAP = F.create_map(*[F.lit(x) for kv_ in STATUS_CLASS.items() for x in kv_])
dep = raw.select(*REQ, *PRC, *TIEB)
for c in ["acct_full_acct_id", "sub_product_cd", "cust_pwr_id", "rltn_pwr_id", "acct_status", "deposit_family"]:
    dep = dep.withColumn(c, nz(c))
for c in ["edw_tda_load_dt", "opened_dt", "closed_dt"]: dep = dep.withColumn(c, F.to_date(F.col(c)))
for c in ["balance"] + PRC: dep = dep.withColumn(c, F.col(c).cast("double"))
dep = (dep.filter(F.col("edw_tda_load_dt").between(F.lit(DATE_START), F.lit(DATE_END)))
          .withColumn("leg", F.coalesce(F.col("sub_product_cd"), F.lit("__NA__")))
          .withColumn("ym", F.date_format("edw_tda_load_dt", "yyyy-MM"))
          .withColumn("status_class", F.coalesce(SMAP[F.col("acct_status")], F.lit("UNMAPPED")))
          .withColumn("live_leg", F.col("status_class").isin(*LIVE_CLASSES).cast("int")))

unm = [r[0] for r in dep.filter("status_class = 'UNMAPPED'").select("acct_status").distinct().limit(20).collect()]
if unm: raise RuntimeError(f"unmapped acct_status codes {unm}: add them to STATUS_CLASS")
n_dup = dep.groupBy("acct_full_acct_id", "leg", "edw_tda_load_dt").count().filter("count > 1").count()
if n_dup:
    _o = [F.col(c).desc_nulls_last() for c in TIEB] + [F.col("balance").desc_nulls_last()]
    dep = dep.withColumn("_rn", F.row_number().over(Window.partitionBy("acct_full_acct_id", "leg", "edw_tda_load_dt")
                                                    .orderBy(*_o))).filter("_rn = 1").drop("_rn")
MC = topd(dep.groupBy("ym").agg(F.countDistinct("edw_tda_load_dt").alias("bdays"),
                                F.countDistinct("cust_pwr_id").alias("customers")).orderBy("ym"))
END_M = MC.loc[MC.bdays >= 15, "ym"].max(); END_T = t_of(END_M)
holes = [ym_of(t) for t in range(END_T + 1) if ym_of(t) not in set(MC.loc[MC.bdays >= 15, "ym"])]
if holes: raise RuntimeError(f"deposit months missing or partial inside the window: {holes}")
dep = dep.withColumn("t", t_col("ym")).filter(F.col("t") <= END_T)

# ── customer-month balances (customer-day first, so daily min / max are real) ──
cday = (dep.filter("cust_pwr_id is not null").groupBy("cust_pwr_id", "edw_tda_load_dt", "t")
           .agg(F.sum("balance").alias("bal_d")))
cbal = cday.groupBy("cust_pwr_id", "t").agg(F.avg("bal_d").alias("bal_avg"), F.min("bal_d").alias("bal_min"),
                                            F.max("bal_d").alias("bal_max"),
                                            F.expr("max_by(bal_d, edw_tda_load_dt)").alias("bal_eom"))
# ── account-month → customer-month counts, product mix, pricing ─────
am = dep.groupBy("acct_full_acct_id", "t", "ym").agg(
    F.max("live_leg").alias("live"),
    F.expr("max_by(cust_pwr_id, edw_tda_load_dt)").alias("cust_pwr_id"),
    F.expr("max_by(rltn_pwr_id, edw_tda_load_dt)").alias("rltn_pwr_id"),
    F.expr("max_by(deposit_family, abs(coalesce(balance, 0)))").alias("deposit_family"),
    (F.sum("balance") / F.countDistinct("edw_tda_load_dt")).alias("acct_bal"),
    F.min("opened_dt").alias("opened_dt"),
    F.max((F.date_format("closed_dt", "yyyy-MM") == F.col("ym")).cast("int")).alias("closed_in_m"),
    *[F.expr(f"max_by({c}, edw_tda_load_dt)").alias(c) for c in PRC])
fam = F.upper(F.coalesce(F.col("deposit_family"), F.lit("")))
pos = F.greatest(F.col("acct_bal"), F.lit(0.0))
fb = lambda cond: F.sum(F.when(cond, pos).otherwise(0.0))
def wavg(c):
    ok = F.col(c).isNotNull() & (F.col("acct_bal") > 0)
    return (F.sum(F.when(ok, F.col(c) * F.col("acct_bal"))) / F.sum(F.when(ok, F.col("acct_bal")))).alias(c)
cacc = am.filter("cust_pwr_id is not null").groupBy("cust_pwr_id", "t").agg(
    F.count("*").alias("n_accts"), F.sum("live").alias("n_live"),
    F.sum((F.date_format("opened_dt", "yyyy-MM") == F.col("ym")).cast("int")).alias("n_opened"),
    F.sum("closed_in_m").alias("n_closed"),
    fb(fam == "IBDDA").alias("fb_ibdda"), fb(fam == "MMDA").alias("fb_mmda"), fb(fam.contains("SWEEP")).alias("fb_sweep"),
    fb(F.lit(True)).alias("fb_all"), F.max("rltn_pwr_id").alias("rltn_pwr_id"), *[wavg(c) for c in PRC])
cm = cbal.join(cacc, ["cust_pwr_id", "t"], "full")

# ── complete month grid ──────────────────────────────────────────────
first = cm.groupBy("cust_pwr_id").agg(F.min("t").alias("first_t"))
grid = first.crossJoin(F.broadcast(spark.range(0, END_T + 1).select(F.col("id").cast("int").alias("t")))) \
            .filter(F.col("t") >= F.col("first_t"))
X = grid.join(cm, ["cust_pwr_id", "t"], "left")
for c in ["n_accts", "n_live", "n_opened", "n_closed"]: X = X.withColumn(c, F.coalesce(F.col(c), F.lit(0)))
for c in ["bal_avg", "bal_min", "bal_max", "bal_eom", "fb_ibdda", "fb_mmda", "fb_sweep", "fb_all"]:
    X = X.withColumn(c, F.coalesce(F.col(c), F.lit(0.0)))
wc = Window.partitionBy("cust_pwr_id"); wo = wc.orderBy("t")
rg = lambda a, b: wo.rangeBetween(a, b)
X = (X.withColumn("rltn_pwr_id", F.last("rltn_pwr_id", ignorenulls=True).over(wo.rowsBetween(Window.unboundedPreceding, 0)))
      .withColumn("bal", F.greatest(F.col("bal_avg"), F.lit(0.0)))
      .withColumn("live", (F.col("n_live") > 0).cast("int"))
      .withColumn("hist_t", F.col("t") - F.col("first_t") + 1)
      .withColumn("last_live_t", F.max(F.when(F.col("live") == 1, F.col("t"))).over(wc))
      .withColumn("exit_t", F.when(F.col("last_live_t") < END_T, F.col("last_live_t") + 1).cast("int"))
      .withColumn("a_t", F.when(F.lit(END_T) - F.col("exit_t") + 1 >= CONFIRM_M, F.col("exit_t")))
      .withColumn("_bl", F.when(F.col("live") == 1, F.col("bal")))
      .withColumn("bal_norm", F.expr("percentile_approx(_bl, 0.5)").over(rg(-NORM_WIN, -1)))
      .withColumn("_nn", F.count("_bl").over(rg(-NORM_WIN, -1)))
      .withColumn("bal_norm", F.when(F.col("_nn") >= NORM_MIN_OBS, F.col("bal_norm")))
      .withColumn("msh", F.coalesce(((F.col("live") == 1) & (F.col("bal_norm") >= MSH_FLOOR)
                                     & (F.col("bal") >= MSH_FRAC * F.col("bal_norm"))).cast("int"), F.lit(0)))
      .withColumn("capped", F.least(F.col("bal"), F.col("bal_norm"))))
thr = F.lit(BAL_EXIT_FRAC) * F.col("bal_norm")
bc = (F.col("live") == 1) & (F.col("bal_norm") > 0) & (F.col("bal") < thr) & (F.col("t") + BAL_EXIT_HOLD - 1 <= END_T)
for k in range(1, BAL_EXIT_HOLD): bc = bc & (F.lead("bal", k).over(wo) < thr)
X = (X.withColumn("b_t", F.min(F.when(F.coalesce(bc, F.lit(False)), F.col("t"))).over(wc).cast("int"))
      .withColumn("ab_t", F.least("a_t", "b_t"))
      .withColumn("exit_type", F.when(F.col("ab_t").isNull(), F.lit(None))
                                .when(F.col("a_t") == F.col("ab_t"), F.lit("full exit")).otherwise(F.lit("balance exit"))))

# ── deposit features (backward-looking only) ─────────────────────────
def chg(col, a, b, min_obs=2):
    return F.when(F.count(col).over(rg(a, b)) >= min_obs, F.col(col) - F.avg(col).over(rg(a, b)))
X = X.withColumn("_lb", F.log1p("bal")).withColumn("_nl", F.col("n_live").cast("double"))
DEP = {}
def add(name, expr, group, label):
    global X; X = X.withColumn(name, expr); DEP[name] = (group, label)
add("dep_log_bal", F.col("_lb"), "balance level", "balance, log")
add("dep_bal_vs_norm", F.col("_lb") - F.log1p("bal_norm"), "balance level", "balance vs normal balance (log ratio)")
add("dep_d1", F.col("_lb") - F.lag("_lb", 1).over(wo), "balance change", "balance change vs last month")
add("dep_d3", chg("_lb", -3, -1), "balance change", "balance change vs the last 3 months")
add("dep_d6", chg("_lb", -6, -4), "balance change", "balance change vs 4–6 months ago")
add("dep_d12", chg("_lb", -12, -10), "balance change", "balance change vs 10–12 months ago")
add("dep_drawdown", F.col("_lb") - F.max("_lb").over(rg(-12, -1)), "balance change", "balance vs its 12-month peak")
add("dep_vol6", F.when(F.count("_lb").over(rg(-5, 0)) >= 3, F.stddev("_lb").over(rg(-5, 0))), "balance volatility",
    "volatility of balance over 6 months")
add("dep_range", F.log1p(F.greatest("bal_max", F.lit(0.0))) - F.log1p(F.greatest("bal_min", F.lit(0.0))),
    "balance volatility", "within-month range of daily balance")
add("dep_eom_vs_avg", F.log1p(F.greatest("bal_eom", F.lit(0.0))) - F.col("_lb"), "balance volatility",
    "month-end balance vs monthly average")
add("dep_n_accts", F.log1p("n_accts"), "accounts", "number of accounts")
add("dep_n_live", F.log1p("n_live"), "accounts", "number of live accounts")
add("dep_live_d6", chg("_nl", -6, -4), "accounts", "change in live accounts vs 4–6 months ago")
add("dep_opened_3m", F.sum("n_opened").over(rg(-2, 0)), "accounts", "accounts opened in the last 3 months")
add("dep_closed_3m", F.sum("n_closed").over(rg(-2, 0)), "accounts", "accounts closed in the last 3 months")
for c, lab in [("fb_ibdda", "interest-bearing DDA"), ("fb_mmda", "money market"), ("fb_sweep", "sweep")]:
    add(f"dep_share_{c[3:]}", F.when(F.col("fb_all") > 0, F.col(c) / F.col("fb_all")), "product mix", f"share of balance in {lab}")
add("dep_tenure", F.log1p("hist_t"), "tenure", "months observed (log)")
add("dep_at_start", (F.col("first_t") == 0).cast("int"), "tenure", "client already present when the data starts")
for c in PRC:
    add(f"dep_{c}", F.col(c), "pricing", c.replace("_", " "))
    add(f"dep_{c}_d6", chg(c, -6, -4), "pricing", f"change in {c.replace('_', ' ')} vs 4–6 months ago")
rl = X.filter("rltn_pwr_id is not null").groupBy("rltn_pwr_id", "t").agg(F.sum("bal").alias("rl_bal"),
                                                                         F.countDistinct("cust_pwr_id").alias("rl_n"))
X = X.join(rl, ["rltn_pwr_id", "t"], "left").withColumn(
    "_lo", F.when(F.col("rl_n") > 1, F.log1p(F.greatest(F.col("rl_bal") - F.col("bal"), F.lit(0.0)))))
add("dep_siblings", F.log1p(F.coalesce(F.col("rl_n"), F.lit(1)) - 1), "relationship", "other customers in the same relationship")
add("dep_sib_d6", chg("_lo", -6, -4), "relationship", "change in the relationship's other balances vs 4–6 months ago")
DEP_FEATS = list(DEP)
X = X.drop("_bl", "_nn", "_nl", "_lo").persist(DISK)
AMP = am.select("acct_full_acct_id", "t", "cust_pwr_id", "live", "acct_bal").persist(DISK)   # for the §7 checks

# ── summary ──────────────────────────────────────────────────────────
g = X.agg(F.count("*").alias("rows"), F.countDistinct("cust_pwr_id").alias("customers"),
          F.sum("live").alias("live_rows"), F.sum("msh").alias("msh_rows")).first().asDict()
lab = topd(X.groupBy("cust_pwr_id").agg(F.first("exit_type").alias("exit_type"), F.first("exit_t").alias("exit_t"),
                                        F.first("a_t").alias("a_t"), F.first("b_t").alias("b_t"))
            .groupBy("exit_type").agg(F.count("*").alias("customers"),
                                      F.sum(F.col("exit_t").isNotNull().cast("int")).alias("also_full_exit_or_provisional"))
            .orderBy("exit_type"))
# is a full exit absorbing?  P(client comes back | ≥ CONFIRM_M observed non-live months)
nl = X.filter((F.col("live") == 0) & (F.col("t") > F.col("first_t")))
nl = nl.withColumn("_g", F.col("t") - F.row_number().over(Window.partitionBy("cust_pwr_id").orderBy("t")))
runs = nl.groupBy("cust_pwr_id", "_g").agg(F.count("*").alias("len"), F.min("t").alias("s"), F.first("last_live_t").alias("llt"))
rr = runs.filter(F.col("len") >= CONFIRM_M).agg(F.count("*").alias("n"), F.sum((F.col("s") < F.col("llt")).cast("int")).alias("back")).first()
R.update(dep_end=END_M, customers=g["customers"], p_return=(rr["back"] or 0) / max(rr["n"], 1), n_dup_keys=n_dup,
         dep_feats=len(DEP_FEATS), pricing_cols=PRC)
B.kv([("deposit window (full months)", f"{DATE_START[:7]} … {END_M} ({END_T + 1} months)"),
      ("duplicated account × leg × day keys (removed)", n_dup), ("customer-months (complete grid)", g["rows"]),
      ("customers", g["customers"]), ("live customer-months", g["live_rows"]),
      ("live months with money still here", f"{g['msh_rows'] / g['live_rows']:.1%}"),
      (f"P(client returns | ≥ {CONFIRM_M} non-live months)", f"{R['p_return']:.2%}  (runs: {rr['n']:,})"),
      ("deposit features", f"{len(DEP_FEATS)} ({', '.join(sorted(set(v[0] for v in DEP.values())))})")], "2a · Panel")
lab["exit_type"] = lab.exit_type.fillna("no exit in window (or not yet confirmed)")
B.table(lab, "2b · Customers by target event (whole window, before any qualification)", save="m1_2b_events")
B.check("deposit panel", "READY", f"{END_M} is the last full month; exits are absorbing ({R['p_return']:.2%} return)"
        if R["p_return"] <= 0.02 else f"{R['p_return']:.2%} of confirmed exits come back — raise CONFIRM_M")
B.show(t0)

## §3 Payment metrics → payment features

Every numeric column of `pkg_cust_metrics_monthly_v3` except the identity columns becomes three features:

| Feature | Meaning |
|---|---|
| `m` | the level: log1p for dollars and counts (signed for net flows), as stored for shares and ratios |
| `m__d6` | change vs the client's own months −6…−4 (a log ratio for dollars and counts, a difference for shares) |
| `m__d3` | change vs the client's own last 3 months |

A client with several `mdm_id`s under one power id takes the row of its largest entity that month. A month with
no row after the client's first row is **zero activity**: dollars and counts 0, shares empty. v3 keeps a row for dark
customers while a standing relationship is broken, so a month counts as **silent** when it has no payments, not when it
has no row. Churn-state metrics
(new / reactivated / retained / lost, recurring, retention) are left empty for the first 6 months of 2024, while
their 12-month lookback is too short to mean anything.

In [ ]:
# =====================================================================
# 3 · PAYMENT METRICS → PAYMENT FEATURES                         [OUTPUT BLOCK 3]
# =====================================================================
t0 = time.time(); B = Block("§3 · Payment metrics → payment features")
MET = spark.table(MET_TABLE).filter(F.col("month").between(DATE_START[:7], END_M))
ID_KNOWN = {"mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type", "naics_cd", "naics_desc",
            "naics2", "naics_status", "is_tm", "pwr_shared", "family_size", "customer_start_dt", "in_neo4j_customer",
            "in_cust_dim", "month", "history_months"}
try:
    DICT = topd(spark.table(DICT_TABLE))
    ID_KNOWN |= set(DICT.loc[DICT.family == "identity", "column"])
    FAMILY = dict(zip(DICT.column, DICT.family)); DEFN = dict(zip(DICT.column, DICT.definition))
except Exception as e:
    FAMILY, DEFN = {}, {}
    B.check("dictionary", "INFO", f"{DICT_TABLE} unreadable ({type(e).__name__}); families default to 'other'")
NUM = ("int", "bigint", "double", "float", "smallint", "tinyint", "long")
MCOLS = [c for c, t in MET.dtypes if c not in ID_KNOWN and (t in NUM or t.startswith("decimal"))]

# ── every month present and populated? ───────────────────────────────
nn = topd(MET.groupBy("month").agg(F.count("*").alias("rows"), *[F.count(c).alias(c) for c in MCOLS]).orderBy("month"))
want = [ym_of(t) for t in range(END_T + 1)]
absent = [m for m in want if m not in set(nn.month)]
pop = (nn[MCOLS] > 0)
usual = pop.mean() >= 0.5                                         # columns populated in most months
CHURN_RE = re.compile(r"^(new_|reactivated_|retained_|lost_|rec_|fi_rec_|retention|fi_retention)")
nt = nn.month.map(t_of).to_numpy()
empty = ~pop.loc[:, usual.values]
lookback = np.array([bool(CHURN_RE.match(c)) for c in empty.columns])
# lookback-based columns are legitimately empty while their history is short (first months of 2024)
bad = empty.loc[:, ~lookback].any(axis=1).to_numpy() & (nt >= 1) | empty.any(axis=1).to_numpy() & (nt >= CHURN_WARM)
stale = nn.loc[bad, "month"].tolist()
B.table(nn[["month", "rows"]].assign(empty_columns=empty.sum(axis=1).values),
        "3a · Metrics by month (empty_columns = columns populated in most months but empty here)", save="m1_3a_months")
if absent or stale:
    B.check("metrics months", "BLOCKER", f"absent {absent}, empty columns in {stale}")
    raise RuntimeError(f"metrics months not ready: absent {absent}, stale {stale}")
B.check("metrics months", "READY", f"all {len(want)} months present, {len(MCOLS)} metric columns populated throughout")

# ── key: deposit cust_pwr_id ↔ metrics customer_pwr_id (format decided from the data) ──
def pk_expr(c, mode):
    s = F.trim(F.col(c).cast("string")); s = F.when(s == "", F.lit(None)).otherwise(s)
    return F.regexp_replace(s, "^0+(?=.)", "") if mode == "strip0" else s
dk = X.select("cust_pwr_id").distinct()
mk = MET.select("customer_pwr_id").distinct()
rates = {m_: dk.select(pk_expr("cust_pwr_id", m_).alias("k")).join(
            mk.select(pk_expr("customer_pwr_id", m_).alias("k")).distinct().withColumn("h", F.lit(1)), "k", "left")
            .agg(F.avg(F.coalesce("h", F.lit(0)).cast("double"))).first()[0] for m_ in ("raw", "strip0")}
KEY = "strip0" if rates["strip0"] > rates["raw"] + 0.01 else "raw"
R["key_match"] = rates[KEY]
if R["key_match"] < 0.3: raise RuntimeError(f"power-id join broken: {rates}")

# ── one row per power id and month: the largest entity ───────────────
flow = sum(F.coalesce(F.col(c).cast("double"), F.lit(0.0)) for c in ["amt_all_in", "amt_all_out"] if c in MCOLS)
ACT = [c for c in ["ntxn_all_out", "ntxn_all_in", "ntxn_self_out", "ntxn_self_in"] if c in MCOLS]
pm = (MET.select(pk_expr("customer_pwr_id", KEY).alias("pwr"), t_col("month").alias("t"), "mdm_id",
                 *[F.col(c).cast("double").alias(c) for c in MCOLS])
         .filter("pwr is not null")
         .withColumn("_f", flow)
         .withColumn("_r", F.row_number().over(Window.partitionBy("pwr", "t").orderBy(F.desc("_f"), "mdm_id")))
         .withColumn("pay_n_mdm", F.count("*").over(Window.partitionBy("pwr", "t")))
         .filter("_r = 1").drop("_r", "_f", "mdm_id")
         .withColumn("_row", F.lit(1))
         .withColumn("_act", (sum(F.coalesce(F.col(c), F.lit(0.0)) for c in ACT) > 0).cast("int") if ACT else F.lit(1)))

def kind(c):
    if re.match(r"^(amt_|net_flow|avg_ticket|selfpay_amt)", c): return "money"
    if re.search(r"(share|hhi|retention|entropy|coverage|ratio)", c): return "bounded"
    return "count"
KIND = {c: kind(c) for c in MCOLS}

Y = X.withColumn("pwr", pk_expr("cust_pwr_id", KEY)).join(pm, ["pwr", "t"], "left")
wp = Window.partitionBy("cust_pwr_id").orderBy("t")
Y = (Y.withColumn("pay_first_t", F.min(F.when(F.col("_row") == 1, F.col("t"))).over(
            wp.rowsBetween(Window.unboundedPreceding, 0)))                    # first row known by month t
      .withColumn("pay_seen", F.col("pay_first_t").isNotNull().cast("int"))
      .withColumn("pay_silent", ((F.col("pay_seen") == 1) & (F.coalesce(F.col("_act"), F.lit(0)) == 0)).cast("int"))   # v3: no activity
      .withColumn("_last_act", F.max(F.when(F.col("_act") == 1, F.col("t"))).over(wp.rowsBetween(Window.unboundedPreceding, 0)))
      .withColumn("pay_months_since_act", F.col("t") - F.col("_last_act"))
      .withColumn("pay_n_mdm", F.when(F.col("pay_seen") == 1, F.coalesce("pay_n_mdm", F.lit(0)))))
# size anchor and raw activity kept before the transforms (silent month = 0 once the client has been seen)
_z = lambda c: F.coalesce(F.col(c), F.lit(0.0)) if c in MCOLS else F.lit(0.0)
Y = (Y.withColumn("pay_log_flow", F.when(F.col("pay_seen") == 1, F.log1p(F.greatest(_z("amt_all_in") + _z("amt_all_out"), F.lit(0.0)))))
      .withColumn("raw_ntx", F.when(F.col("pay_seen") == 1, _z("ntxn_all_in") + _z("ntxn_all_out")))
      .withColumn("raw_selfpay_out", F.when(F.col("pay_seen") == 1, _z("selfpay_amt_out"))))
lv = []
for c in MCOLS:
    v = F.col(c)
    if KIND[c] == "money": e = F.signum(v) * F.log1p(F.abs(v))
    elif KIND[c] == "count": e = F.log1p(F.greatest(v, F.lit(0.0)))
    else: e = v
    if KIND[c] != "bounded": e = F.when(F.col("_row").isNull() & (F.col("pay_seen") == 1), F.lit(0.0)).otherwise(e)
    if CHURN_RE.match(c): e = F.when(F.col("t") < CHURN_WARM, F.lit(None)).otherwise(e)
    lv.append(e.alias(c))
Y = Y.select(*[c for c in Y.columns if c not in MCOLS], *lv)
wr = lambda a, b: wp.rangeBetween(a, b)
ch = []
for c in MCOLS:
    for nm, (a, b) in (("d6", (-6, -4)), ("d3", (-3, -1))):
        ch.append(F.when(F.count(c).over(wr(a, b)) >= 2, F.col(c) - F.avg(c).over(wr(a, b))).alias(f"{c}__{nm}"))
FEAT = Y.select("*", *ch).persist(DISK)
PAY_FEATS = [f for c in MCOLS for f in (c, f"{c}__d6", f"{c}__d3")]
STATUS_FEATS = ["pay_seen", "pay_silent", "pay_months_since_act"]

cov = FEAT.filter("live = 1").agg(F.count("*").alias("n"), F.avg("pay_seen").alias("seen"), F.avg("pay_silent").alias("silent"),
                                  F.sum(F.when(F.col("pay_seen") == 1, F.col("bal"))).alias("bal_seen"),
                                  F.sum("bal").alias("bal_all")).first()
R.update(pay_seen=cov["seen"], pay_seen_usd=cov["bal_seen"] / cov["bal_all"], pay_silent=cov["silent"],
         n_metrics=len(MCOLS), pay_feats=len(PAY_FEATS), key_mode=KEY)
fam_tab = pd.Series({c: FAMILY.get(c, "other") for c in MCOLS}).value_counts().rename_axis("family").reset_index(name="metrics")
B.kv([("power-id key match (deposit customers found in the metrics)", f"{R['key_match']:.1%} (mode {KEY})"),
      ("live customer-months with payment history", f"{cov['seen']:.1%} of months, {R['pay_seen_usd']:.1%} of balance"),
      ("…of which silent (no payments that month)", f"{cov['silent']:.1%} of live months"),
      ("metric columns → features", f"{len(MCOLS)} → {len(PAY_FEATS)} (+ {len(STATUS_FEATS)} activity-status features)")],
     "3b · Payment features")
B.table(fam_tab, "3c · Metric columns by dictionary family")
B.show(t0)

## §3b Network features: counterparties, processors, NAICS mix

Built from the pair table (`pkg_cust_pair_month_v3`), one month at a time, for the deposit clients only. Family
flows (same power id or same master power id) are internal treasury moves and are left out. Every value uses data up
to its own month.

| Family | Features (each direction: `out` = the client pays, `in` = the client receives) |
|---|---|
| counterparty health | dollar-weighted average of the client's **external** counterparties' metrics (`pkg_cpty_metrics_monthly`, eligible counterparties, the side facing the client): retention, lost and new dollar shares, broken recurring dollars, concentration, hub tier, main-bank share, banks lost |
| PNC counterparties | dollar-weighted average of the **PNC** counterparties' own metrics (v3), and from the deposit panel: their balance change, their balance against normal, and the **share of the client's last 6 months' dollars with PNC counterparties whose money has left** (counted only once the exit is confirmed) |
| processors | dollars and share of all dollars with money-movement processors (PayPal, Stripe, Wise, Bill.com …), payroll processors, card networks, brokerages and TreasuryDirect, and banks: by name outside PNC, by NAICS inside (522320 / 522390, 541214, 523, 5221) |
| NAICS mix | share of the client's dollars with PNC customers of known NAICS going to each tracked group (2-, 3-, 4-, 6-digit, chosen by volume over the labelled months), the spread across NAICS-4 groups, and the share going to NAICS-4 groups the client did not pay in the previous 6 months |
| counterparty features v1.1 | `pkg_attr_cpty_features_monthly` (own accounts elsewhere, migrated partners, broken relationships), when the table exists |

Each feature also gets its change against the client's own months −6…−4 (and, for processors and NAICS mix, the
last 3 months). Dollars through processors are compared with size peers like the other payment levels.

The monthly results are cached under the cdi_tm location with a signature of the settings, the customer dimension and
the client set, and a fingerprint of each month's input partitions; a month is rebuilt only when one of them changes.

In [ ]:
# =====================================================================
# 3b · NETWORK FEATURES — counterparties, processors, NAICS mix   [OUTPUT BLOCK 3b]
# =====================================================================
t0 = time.time(); B = Block("§3b · Network features: counterparties, processors, NAICS mix")
for t_ in (PAIR_TABLE, CPTY_TABLE, DIM_TABLE):
    if not table_exists(t_): raise RuntimeError(f"{t_} is not readable from Spark")
_pc, _cc, _dc = (spark.table(t_).columns for t_ in (PAIR_TABLE, CPTY_TABLE, DIM_TABLE))
miss = ([f"pair.{c}" for c in ("cust", "dir", "scope", "other", "amount", "month") if c not in _pc]
        + [f"cpty.{c}" for c in ("ent_id", "ent_name", "month") if c not in _cc]
        + [f"dim.{c}" for c in ("mdm_id", "customer_pwr_id", "naics_cd") if c not in _dc])
if miss: raise RuntimeError(f"columns missing: {miss}")
MCOLS0 = list(MCOLS)                                   # v3 payment metrics (model 4's features)
CACHE = (NET_CACHE or f"{LOC}/_tmp_pkg_attr_m5_cache").rstrip("/")
DIRS = ("out", "in")
SIDE = {"out": "in", "in": "out"}                      # the client pays (out) → the counterparty receives (its _in side)
GROUPS = list(PROC_RE) + ["bank"]
CP_SIDE = [c for c in CP_SIDE_COLS if f"{c}_in" in _cc and f"{c}_out" in _cc]
CP_FULL = [c for c in CP_FULL_COLS if c in _cc]
_mc = set(MET.columns)
NB_SIDE = [c for c in NB_SIDE_COLS if f"{c}_in" in _mc and f"{c}_out" in _mc]
NB_FIX = [c for c in NB_FIX_COLS if c in _mc]
NB_MET = sorted({f"{c}_{s}" for c in NB_SIDE for s in DIRS} | set(NB_FIX))

# ── customer dimension: one row per mdm_id (asserted), power id, family, 6-digit NAICS ──
_ns = F.col("naics_status") == "valid" if "naics_status" in _dc else F.lit(True)
DIMX = (spark.table(DIM_TABLE)
          .select(nz("mdm_id").alias("other"), pk_expr("customer_pwr_id", KEY).alias("o_pwr"),
                  (pk_expr("master_pwr_id", KEY) if "master_pwr_id" in _dc else F.lit(None).cast("string")).alias("o_fam"),
                  F.when(_ns, F.regexp_replace(F.coalesce(F.col("naics_cd").cast("string"), F.lit("")), "[^0-9]", "")).alias("n6"),
                  (nz("naics_desc") if "naics_desc" in _dc else F.lit(None).cast("string")).alias("naics_desc"))
          .filter("other is not null")
          .withColumn("o_fam", F.coalesce("o_fam", "o_pwr"))
          .withColumn("n6", F.when(F.length("n6") >= 2, F.substring("n6", 1, 6))))
n_dim_dup = DIMX.groupBy("other").count().filter("count > 1").count()
if n_dim_dup:      # one row per key is required before joining to edge rows; keep one deterministically
    DIMX = DIMX.groupBy("other").agg(F.max(F.struct("o_pwr", "o_fam", "n6", "naics_desc")).alias("s")).select("other", "s.*")
DIMP = DIMX.select("other", "o_pwr", "o_fam", "n6").repartition(SHUFFLE_PARTITIONS, "other").persist(DISK)
CLI = X.select(pk_expr("cust_pwr_id", KEY).alias("pwr")).filter("pwr is not null").distinct().persist(DISK)
CMAP = (DIMP.filter("o_pwr is not null").join(CLI.withColumnRenamed("pwr", "o_pwr"), "o_pwr", "left_semi")
            .select(F.col("other").alias("cust"), F.col("o_pwr").alias("pwr"), F.col("o_fam").alias("fam")).persist(DISK))
_cf = CLI.agg(F.count("*").alias("n"), F.sum(F.crc32("pwr")).alias("h")).first()
_dl = table_location(DIM_TABLE)
NET_SIG = md5(dict(v=NET_VERSION, key=KEY, proc_re=PROC_RE, proc_naics=PROC_NAICS, cp=[CP_SIDE, CP_FULL], nb=[NB_SIDE, NB_FIX],
                   clients=[_cf["n"], _cf["h"]], dim=path_fp(_dl) if _dl else "unknown"))
WANT = [ym_of(t) for t in range(END_T + 1)]

# ── one month: client side of the pair table → three cached outputs ─
def _grp_ext():
    g = F.lit(None).cast("string")
    for k in reversed(list(PROC_RE)): g = F.when(F.col("nm").rlike(PROC_RE[k]), F.lit(k)).otherwise(g)
    return F.when(g.isNull() & (F.col("o_class") == "bank"), F.lit("bank")).otherwise(g)
def _grp_int():
    g = F.lit(None).cast("string")
    for k in reversed(list(PROC_NAICS)):
        c_ = None
        for p_ in PROC_NAICS[k]:
            c_ = F.col("n6").startswith(p_) if c_ is None else (c_ | F.col("n6").startswith(p_))
        g = F.when(c_, F.lit(k)).otherwise(g)
    return g
def _side(c, prefix=""):          # metric of the side facing the client
    return F.when(F.col("dir") == "out", F.col(f"{prefix}{c}_in")).otherwise(F.col(f"{prefix}{c}_out"))
_s = lambda cond: F.sum(F.when(cond, F.col("amt")).otherwise(F.lit(0.0)))
ANY_PROC = "|".join(f"(?:{v})" for v in PROC_RE.values())
_sx = lambda x: F.sum(F.when(x.isNotNull(), F.col("amt") * x))

def net_month(m):
    t = t_of(m)
    P = (spark.table(PAIR_TABLE).filter(F.col("month") == m).filter(F.col("scope").isin("int", "ext"))
           .select(nz("cust").alias("cust"), F.col("dir").cast("string").alias("dir"), F.col("scope").cast("string").alias("scope"),
                   nz("other").alias("other"), F.col("amount").cast("double").alias("amt"),
                   (F.lower(F.coalesce(F.col("o_class").cast("string"), F.lit(""))) if "o_class" in _pc else F.lit("")).alias("o_class"),
                   (F.coalesce(F.col("o_self").cast("boolean"), F.lit(False)) if "o_self" in _pc else F.lit(False)).alias("o_self"))
           .filter((F.col("amt") > 0) & F.col("dir").isin(*DIRS) & F.col("other").isNotNull())
           .join(CMAP, "cust").drop("cust").persist(DISK))
    # ── outside PNC: counterparty health and processors by name (own accounts elsewhere excluded) ──
    sel = [nz("ent_id").alias("other"),
           F.trim(F.regexp_replace(F.upper(F.coalesce(F.col("ent_name").cast("string"), F.lit(""))), "[^A-Z0-9]+", " ")).alias("nm")]
    for c in CP_SIDE:
        for s_ in DIRS:
            v = F.col(f"{c}_{s_}").cast("double")
            sel.append((F.log1p(F.greatest(v, F.lit(0.0))) if c == "n_cust" else v).alias(f"cp_{c}_{s_}"))
    sel += [F.col(c).cast("double").alias(f"cp_{c}") for c in CP_FULL]
    sel.append((F.coalesce(F.col("eligible").cast("boolean"), F.lit(False)) if "eligible" in _cc else F.lit(True)).alias("_el"))
    # eligible counterparties carry the full metric set (≈ 99% of dollars); others are kept only when their name is a processor's
    CPm = spark.table(CPTY_TABLE).filter(F.col("month") == m).select(*sel).filter(F.col("_el") | F.col("nm").rlike(ANY_PROC))
    E = P.filter("scope = 'ext' and not o_self").join(CPm, "other", "left").withColumn("grp", _grp_ext())
    ag = [F.sum("amt").alias("tot_ext"), _s(F.coalesce(F.col("_el"), F.lit(False))).alias("cp_w")]
    for c in CP_SIDE + CP_FULL:
        x = _side(c, "cp_") if c in CP_SIDE else F.col(f"cp_{c}")
        ag += [_sx(x).alias(f"sx_cp_{c}"), _s(x.isNotNull()).alias(f"sw_cp_{c}")]
    ag += [_s(F.col("grp") == g).alias(f"ge_{g}") for g in GROUPS]
    EA = E.groupBy("pwr", "dir").agg(*ag)
    # ── inside PNC: other customers' metrics, processors and banks by NAICS; family flows excluded ──
    Mm = spark.table(MET_TABLE).filter(F.col("month") == m).select(nz("mdm_id").alias("other"),
                                                                    *[F.col(c).cast("double").alias(f"m_{c}") for c in NB_MET])
    I = (P.filter("scope = 'int'").join(DIMP, "other", "left")
          .filter(F.col("o_fam").isNull() | (F.col("o_fam") != F.col("fam")))
          .persist(DISK))
    IM = I.join(Mm, "other", "left").withColumn("grp", _grp_int())
    ag = [F.sum("amt").alias("tot_int"), _s(F.col("n6").isNotNull()).alias("n6_w")]
    for c in NB_SIDE + NB_FIX:
        x = _side(c, "m_") if c in NB_SIDE else F.col(f"m_{c}")
        ag += [_sx(x).alias(f"sx_nb_{c}"), _s(x.isNotNull()).alias(f"sw_nb_{c}")]
    ag += [_s(F.col("grp") == g).alias(f"gi_{g}") for g in GROUPS]
    IA = IM.groupBy("pwr", "dir").agg(*ag)
    # ── one row per client: weighted means, processor dollars and shares, by direction ──
    A = EA.join(IA, ["pwr", "dir"], "full")
    z = lambda c: F.coalesce(F.col(c), F.lit(0.0))
    rat = lambda a, b: F.when(b > 0, a / b)
    cols = {"tot_ext": z("tot_ext"), "tot_int": z("tot_int"), "cp_cov": rat(z("cp_w"), z("tot_ext"))}
    for c in CP_SIDE + CP_FULL: cols[f"cp_{c}"] = rat(F.col(f"sx_cp_{c}"), z(f"sw_cp_{c}"))
    for c in NB_SIDE + NB_FIX: cols[f"nb_{c}"] = rat(F.col(f"sx_nb_{c}"), z(f"sw_nb_{c}"))
    for g in GROUPS:
        cols[f"pr_{g}_amt"] = z(f"ge_{g}") + z(f"gi_{g}")
        cols[f"pr_{g}_share"] = rat(z(f"ge_{g}") + z(f"gi_{g}"), z("tot_ext") + z("tot_int"))
    A = A.select("pwr", "dir", *[v.alias(k) for k, v in cols.items()])
    W = A.groupBy("pwr").agg(*[F.max(F.when(F.col("dir") == d, F.col(k))).alias(f"{k}_{d}") for k in cols for d in DIRS])
    outs = {"base": W.withColumn("t", F.lit(t)),
            "nxa": I.filter("n6 is not null").groupBy("pwr", "dir", "n6").agg(F.sum("amt").alias("amt")),
            "intp": (I.filter("o_pwr is not null").join(CLI.withColumnRenamed("pwr", "o_pwr"), "o_pwr", "left_semi")
                       .groupBy("pwr", "dir", "o_pwr").agg(F.sum("amt").alias("amt")))}
    for k, df_ in outs.items():
        st_ = f"{CACHE}/{k}/_stage_month={m}"; rm(st_)
        df_.write.mode("overwrite").parquet(st_); mv(st_, f"{CACHE}/{k}/month={m}")
    P.unpersist(); I.unpersist()

# ── run the months that are missing or stale ─────────────────────────
NET_LOG, _h = [], display(HTML("<i>§3b · checking the network cache …</i>"), display_id=True)
todo = []
for m in WANT:
    fp = {k: part_fp(t_, m) for k, t_ in (("pair", PAIR_TABLE), ("cpty", CPTY_TABLE), ("met", MET_TABLE))}
    mk = read_json(f"{CACHE}/_markers/month={m}.json")
    ok = mk and mk.get("sig") == NET_SIG and mk.get("fp") == fp and all(exists(f"{CACHE}/{k}/month={m}/_SUCCESS") for k in ("base", "nxa", "intp"))
    if not ok: todo.append((m, fp))
for i, (m, fp) in enumerate(todo, 1):
    tm_ = time.time(); net_month(m)
    write_json(f"{CACHE}/_markers/month={m}.json", {"sig": NET_SIG, "fp": fp, "version": NET_VERSION,
                                                    "built_at": time.strftime("%Y-%m-%d %H:%M:%S")})
    NET_LOG.append({"month": m, "minutes": (time.time() - tm_) / 60})
    _h.update(HTML(f"<i>§3b · built {i} of {len(todo)} months (last {m}, {NET_LOG[-1]['minutes']:.1f} min)</i>"))
_h.update(HTML(f"<i>§3b · cache ready: {len(todo)} months built, {len(WANT) - len(todo)} reused</i>"))
rd = lambda k: spark.read.parquet(f"{CACHE}/{k}").filter(F.col("month").isin(WANT)).withColumn("t", t_col("month")).drop("month")

# ── NAICS mix: tracked groups chosen by out-dollars over the labelled months only ──
LAST_LABEL_T = min(END_T - CONFIRM_M + 1, END_T - BAL_EXIT_HOLD + 1) - H
NXA = rd("nxa").persist(DISK)
TOPC = []
for L, k in NAICS_TOPK.items():
    for d in (("out", "in") if L in NAICS_IN_LEVELS else ("out",)):
        q = (NXA.filter((F.col("t") <= LAST_LABEL_T) & (F.col("dir") == d) & (F.length("n6") >= L))
                .groupBy(F.substring("n6", 1, L).alias("code")).agg(F.sum("amt").alias("amt"))
                .orderBy(F.desc("amt"), "code").limit(k).collect())
        tot_ = NXA.filter((F.col("t") <= LAST_LABEL_T) & (F.col("dir") == d)).agg(F.sum("amt")).first()[0] or 1.0
        TOPC += [(L, d, r["code"], float(r["amt"] / tot_)) for r in q]
ag = [F.sum(F.when(F.col("dir") == d, F.col("amt"))).alias(f"kn_{d}") for d in DIRS]
ag += [F.sum(F.when((F.col("dir") == d) & F.col("n6").startswith(c), F.col("amt"))).alias(f"nx{L}_{c}_{d}") for L, d, c, _ in TOPC]
NX = NXA.groupBy("pwr", "t").agg(*ag)
NX = NX.select("pwr", "t", *[F.col(f"kn_{d}") for d in DIRS],
               *[F.when(F.col(f"kn_{d}") > 0, F.coalesce(F.col(f"nx{L}_{c}_{d}"), F.lit(0.0)) / F.col(f"kn_{d}")).alias(f"nx{L}_{c}_{d}")
                 for L, d, c, _ in TOPC])
n4 = (NXA.filter((F.col("dir") == "out") & (F.length("n6") >= 4)).groupBy("pwr", "t", F.substring("n6", 1, 4).alias("n4"))
         .agg(F.sum("amt").alias("amt")))
n4 = (n4.withColumn("prev", F.sum("amt").over(Window.partitionBy("pwr", "n4").orderBy("t").rangeBetween(-NEW_NAICS_M, -1)))
        .withColumn("p_", F.col("amt") / F.sum("amt").over(Window.partitionBy("pwr", "t"))))
N4 = n4.groupBy("pwr", "t").agg((-F.sum(F.col("p_") * F.log("p_"))).alias("nx_ent4_out"),
                                F.sum(F.when(F.coalesce("prev", F.lit(0.0)) <= 0, F.col("amt")).otherwise(0.0)).alias("_new"),
                                F.sum("amt").alias("_all"))
N4 = N4.select("pwr", "t", "nx_ent4_out",
               F.when(F.col("t") >= NEW_NAICS_M, F.col("_new") / F.col("_all")).alias("nx_new4_out"))

# ── PNC counterparties whose money has left, and their deposits (weights = last 6 months' dollars) ──
XN = (X.select(pk_expr("cust_pwr_id", KEY).alias("o_pwr"), F.col("t").alias("tt"), "dep_d6", "dep_bal_vs_norm",
               F.coalesce(((F.col("ab_t") + KNOWN_LAG) <= F.col("t")).cast("int"), F.lit(0)).alias("gone"))
        .groupBy("o_pwr", "tt").agg(F.max("gone").alias("gone"), F.avg("dep_d6").alias("o_d6"), F.avg("dep_bal_vs_norm").alias("o_bvn")))
IPx = (rd("intp").withColumn("k", F.explode(F.array(*[F.lit(k) for k in range(1, NB_BASE_M + 1)])))
         .withColumn("tt", F.col("t") + F.col("k")).filter(F.col("tt") <= END_T)
         .groupBy("pwr", "dir", "o_pwr", "tt").agg(F.sum("amt").alias("w")))
J = IPx.join(XN, ["o_pwr", "tt"], "left")
ag = []
for d in DIRS:
    dd = F.col("dir") == d
    wsum = lambda c: F.sum(F.when(dd & F.col(c).isNotNull(), F.col("w")))
    ag += [(F.sum(F.when(dd, F.col("w") * F.col("gone"))) / wsum("gone")).alias(f"nb_gone_share_{d}"),
           (F.sum(F.when(dd, F.col("w") * F.col("o_d6"))) / wsum("o_d6")).alias(f"nb_dep_d6_{d}"),
           (F.sum(F.when(dd, F.col("w") * F.col("o_bvn"))) / wsum("o_bvn")).alias(f"nb_bal_vs_norm_{d}")]
NB2 = J.groupBy("pwr", F.col("tt").alias("t")).agg(*ag)

# ── counterparty features v1.1 (optional) ────────────────────────────
CF_COLS, CF = [], None
if USE_CPF and table_exists(CPF_TABLE):
    cf_ = spark.table(CPF_TABLE)
    num = [c for c, ty in cf_.dtypes if (ty in NUM or ty.startswith("decimal")) and c not in CPF_SKIP | {"month", "t"}]
    pick = [c for c in num if re.match(CPF_RE, c)] or num
    if "customer_pwr_id" in cf_.columns and "month" in cf_.columns and pick:
        CF_COLS = [f"cf_{c}" for c in pick]
        CF = (cf_.filter(F.col("month").isin(WANT))
                 .select(pk_expr("customer_pwr_id", KEY).alias("pwr"), t_col("month").alias("t"),
                         *[F.col(c).cast("double").alias(f"cf_{c}") for c in pick])
                 .groupBy("pwr", "t").agg(*[F.max(c).alias(c) for c in CF_COLS]))

# ── all network columns, by client and month ─────────────────────────
NBR = rd("base")
for df_ in (NX, N4, NB2) + ((CF,) if CF is not None else ()):
    NBR = NBR.join(df_, ["pwr", "t"], "full")
for d in DIRS:
    NBR = NBR.withColumn(f"nx_known_share_{d}", F.when((F.col(f"tot_int_{d}") + F.col(f"tot_ext_{d}")) > 0,
                                                       F.coalesce(F.col(f"kn_{d}"), F.lit(0.0)) / (F.col(f"tot_int_{d}") + F.col(f"tot_ext_{d}"))))

# ── names, families, labels, kinds and forms ─────────────────────────
try:
    _cd = topd(spark.table(CPTY_DICT)); DEFC = dict(zip(_cd["column"], _cd["definition"]))
except Exception: DEFC = {}
WHO = {"out": ("payees", "paid to", "payments"), "in": ("payers", "received from", "receipts")}
CPT = {"retention": "share of their PNC customers retained", "lost_amt_share": "share of their PNC dollars from customers lost this month",
       "new_amt_share": "share of their PNC dollars from new customers", "rec_broken_amt_share": "share of their recurring PNC dollars broken",
       "top1_share": "share of their PNC dollars with their largest customer", "n_cust": "number of PNC customers (log)",
       "hub_tier": "hub tier (0–3)", "top_fi_share": "share of their dollars at their main bank", "fi_lost": "banks they stopped using",
       "two_way_ratio": "two-way ratio"}
NBT = {"retention": "share of counterparties retained", "lost_amt_share": "share of dollars from counterparties lost",
       "rec_broken_amt_share": "share of recurring dollars broken", "selfpay_share_out": "share of payments to own accounts at other banks",
       "fi_rec_broken_net_share_out": "share of recurring bank relationships broken", "pnc_fi_share_out": "share of payments to accounts at PNC"}
GRPT = {"mm": "money-movement processors (PayPal, Stripe, Wise, Bill.com …)", "payroll": "payroll processors (ADP, Paychex …)",
        "card": "card networks and acquirers", "invest": "brokerages and TreasuryDirect", "bank": "banks"}
N2 = {"11": "agriculture", "21": "mining, oil and gas", "22": "utilities", "23": "construction", "31": "manufacturing",
      "32": "manufacturing", "33": "manufacturing", "42": "wholesale trade", "44": "retail trade", "45": "retail trade",
      "48": "transportation", "49": "warehousing and couriers", "51": "information", "52": "finance and insurance",
      "53": "real estate and leasing", "54": "professional services", "55": "management of companies",
      "56": "administrative and waste services", "61": "education", "62": "health care", "71": "arts and recreation",
      "72": "accommodation and food", "81": "other services", "92": "public administration"}
_codes = sorted({c for _, _, c, _ in TOPC if len(c) > 2})
NDESC = {}
if _codes:
    _dd = topd(DIMX.filter(F.col("n6").isNotNull() & F.col("naics_desc").isNotNull()).groupBy("n6", "naics_desc").count())
    for c in _codes:
        h_ = _dd[_dd.n6.str.startswith(c)].sort_values(["count", "n6"], ascending=[False, True])
        NDESC[c] = h_.naics_desc.iloc[0] if len(h_) else ""
def ndesc(c): return N2.get(c, "") if len(c) == 2 else (("e.g. " + NDESC.get(c, "").lower()[:60]) if NDESC.get(c) else "")

NET = {}   # column → (family, label, kind, forms); kind "money" = log of dollars, 0 in a month without flows
for d in DIRS:
    who, verb, noun = WHO[d]
    for c in CP_SIDE + CP_FULL:
        txt = CPT.get(c) or DEFC.get(f"{c}_{SIDE[d]}") or DEFC.get(c) or c.replace("_", " ")
        NET[f"cp_{c}_{d}"] = ("counterparty health", f"{who.capitalize()} outside PNC: {txt} (dollar-weighted)", "mean", ("lvl", "d6"))
    NET[f"cp_cov_{d}"] = ("counterparty health", f"share of {noun} outside PNC with eligible counterparties (full metric set)", "mean", ("lvl", "d6"))
    for c in NB_SIDE + NB_FIX:
        NET[f"nb_{c}_{d}"] = ("PNC counterparties", f"PNC {who}: {NBT.get(c, c)} (dollar-weighted)", "mean", ("lvl", "d6"))
    NET[f"nb_gone_share_{d}"] = ("PNC counterparties", f"share of the last 6 months' {noun} with PNC {who} whose money has left", "mean", ("lvl", "d6"))
    NET[f"nb_dep_d6_{d}"] = ("PNC counterparties", f"PNC {who}: their balance change vs 4–6 months ago", "mean", ("lvl", "d6"))
    NET[f"nb_bal_vs_norm_{d}"] = ("PNC counterparties", f"PNC {who}: their balance vs normal", "mean", ("lvl", "d6"))
    for g in GROUPS:
        NET[f"pr_{g}_amt_{d}"] = ("processors", f"{verb.capitalize()} {GRPT[g]}: dollars (log)", "money", ("peer", "d6", "d3"))
        NET[f"pr_{g}_share_{d}"] = ("processors", f"{verb.capitalize()} {GRPT[g]}: share of all {noun}", "mean", ("lvl", "d6", "d3"))
    NET[f"nx_known_share_{d}"] = ("NAICS mix", f"share of {noun} with PNC customers of known NAICS", "mean", ("lvl", "d6"))
for L, d, c, _ in TOPC:
    NET[f"nx{L}_{c}_{d}"] = ("NAICS mix", f"Share of {WHO[d][2]} {'to' if d == 'out' else 'from'} PNC customers in NAICS {c}"
                             + (f" ({ndesc(c)})" if ndesc(c) else ""), "mean", ("lvl", "d6", "d3"))
NET["nx_ent4_out"] = ("NAICS mix", "spread of payments across NAICS-4 industries (entropy)", "mean", ("lvl", "d6", "d3"))
NET["nx_new4_out"] = ("NAICS mix", "share of payments to NAICS-4 industries not paid in the previous 6 months", "mean", ("lvl", "d6", "d3"))
_cfd = {}
try:
    _x = topd(spark.table(CPF_TABLE.replace("_monthly", "_dictionary"))); _cfd = dict(zip(_x["column"], _x["definition"]))
except Exception: pass
for c in CF_COLS:
    NET[c] = ("counterparty features v1.1", f"{_cfd.get(c[3:], c[3:].replace('_', ' '))} (cpty features v1.1)",
              "money" if re.search(r"_amt_(in|out)$", c) else "mean", ("lvl", "d6"))
NETCOLS = [c for c in NET if c in NBR.columns]
clash = [c for c in NETCOLS if c in FEAT.columns]
if clash: raise RuntimeError(f"network column names clash with existing columns: {clash[:5]}")

# ── join, transform, change forms ────────────────────────────────────
seen = F.col("pay_seen") == 1
RAWP = [f"pr_{g}_amt_out" for g in GROUPS]
F2 = FEAT.join(NBR.select("pwr", "t", *NETCOLS), ["pwr", "t"], "left")
lv = []
for c in NETCOLS:
    v = F.col(c)
    lv.append((F.when(seen, F.log1p(F.greatest(F.coalesce(v, F.lit(0.0)), F.lit(0.0)))) if NET[c][2] == "money" else v).alias(c))
F2 = F2.select(*[c for c in F2.columns if c not in NETCOLS],
               *[F.when(seen, F.coalesce(F.col(c), F.lit(0.0))).alias(f"raw_{c}") for c in RAWP], *lv)
ch = []
for c in NETCOLS:
    for nm, (a, b) in (("d6", (-6, -4)), ("d3", (-3, -1))):
        if nm in NET[c][3]:
            ch.append(F.when(F.count(c).over(wr(a, b)) >= 2, F.col(c) - F.avg(c).over(wr(a, b))).alias(f"{c}__{nm}"))
_old = FEAT
FEAT = F2.select("*", *ch).persist(DISK)
n_feat_rows = FEAT.count(); _old.unpersist(); NXA.unpersist()
FORMS = {c: ("peer", "d6", "d3") for c in MCOLS0}
FORMS.update({c: NET[c][3] for c in NETCOLS})
MCOLS = MCOLS0 + NETCOLS
NETSET = set(NETCOLS)
for c in NETCOLS: FAMILY[c] = NET[c][0]; DEFN[c] = NET[c][1]

# ── coverage ─────────────────────────────────────────────────────────
cv = FEAT.filter("live = 1").agg(F.count("*").alias("n"), *[F.avg(F.col(c).isNotNull().cast("int")).alias(c) for c in NETCOLS],
                                 *[F.avg((F.col(f"raw_{c}") > 0).cast("int")).alias(f"any_{c}") for c in RAWP]).first().asDict()
NFAM = (pd.DataFrame([{"family": NET[c][0], "column": c, "coverage": cv[c]} for c in NETCOLS])
          .groupby("family").agg(columns=("column", "size"), median_coverage=("coverage", "median"), max_coverage=("coverage", "max"))
          .reset_index())
PRT = pd.DataFrame([{"group": GRPT[g], "client-months paying it (live)": cv[f"any_pr_{g}_amt_out"]} for g in GROUPS])
NXT = pd.DataFrame([{"level": L, "direction": d, "code": c, "label": ndesc(c), "share of labelled dollars": s} for L, d, c, s in TOPC])
R.update(net_signature=NET_SIG, net_cache=CACHE, net_built=len(todo), net_cols=len(NETCOLS), net_fam=NFAM, net_proc=PRT, net_naics=NXT,
         net_groups=GRPT, cpf_used=bool(CF_COLS), n_dim_dup=n_dim_dup)
B.kv([("cache", f"{CACHE} · signature {NET_SIG}"),
      ("months built this run / reused", f"{len(todo)} / {len(WANT) - len(todo)}"
       + (f" ({sum(r['minutes'] for r in NET_LOG):,.1f} min)" if NET_LOG else "")),
      ("clients (deposit power ids) · customer-dim duplicates fixed", f"{_cf['n']:,} · {n_dim_dup:,}"),
      ("network columns → with change forms", f"{len(NETCOLS)} → {len(NETCOLS) + len(ch)}"),
      ("counterparty features v1.1 table", f"{len(CF_COLS)} columns joined" if CF_COLS else "not used (absent or USE_CPF = False)")],
     "3d · Network features")
B.table(NFAM, "3e · Families (coverage = share of live client-months with a value)", pct=("median_coverage", "max_coverage"),
        save="m5_3e_network_families")
B.table(PRT, "3f · Processors, brokerages and banks: live client-months with payments to them", pct=("client-months paying it (live)",),
        save="m5_3f_processors")
B.table(NXT, "3g · NAICS groups tracked (chosen on the labelled months)", pct=("share of labelled dollars",), save="m5_3g_naics")
B.check("network features", "READY" if NETCOLS else "FIX", f"{len(NETCOLS)} columns from {len(NFAM)} families")
B.show(t0)

## §4 Model frame: money-still-here rows, peer gaps, de-duplicated features

**Rows:** live, observed ≥ 3 months, normal balance ≥ USD 10k, current balance ≥ half of normal, money not yet
gone — exactly the clients the list can call. **Label:** the money leaves (full or balance exit) in the next 6
months. **Size peers:** the same month's clients in the same decile of normal balance; each payment level becomes
its gap to the peers' median. **Size anchors:** log balance, log normal balance, log payment flow.
**Network features** (§3b) enter as levels and changes; dollars through processors also get a gap to size peers.
Near-duplicates are resolved in favour of model 4's features, so the ablation in §5–6 compares like with like.

In [ ]:
# =====================================================================
# 4 · MODEL FRAME, TRAINING SAMPLE, FEATURE SETS                  [OUTPUT BLOCK 4]
# =====================================================================
t0 = time.time(); B = Block("§4 · Model frame and feature sets")
LAST_LABEL_T = min(END_T - CONFIRM_M + 1, END_T - BAL_EXIT_HOLD + 1) - H
MF = (FEAT.filter((F.col("live") == 1) & (F.col("hist_t") >= MIN_HIST) & F.col("bal_norm").isNotNull()
                  & (F.col("msh") == 1) & (F.col("ab_t").isNull() | (F.col("t") < F.col("ab_t"))))
          .withColumn("y", F.coalesce(((F.col("ab_t") > F.col("t")) & (F.col("ab_t") <= F.col("t") + H)).cast("int"), F.lit(0)))
          .withColumn("labelled", (F.col("t") <= LAST_LABEL_T).cast("int"))
          .withColumn("dep_log_norm", F.log1p("bal_norm"))
          .withColumn("size_dec", F.least(F.floor(F.percent_rank().over(Window.partitionBy("t").orderBy("bal_norm")) * 10),
                                          F.lit(9)).cast("int")))
wpeer = Window.partitionBy("t", "size_dec")
PEER_COLS = [c for c in MCOLS if "peer" in FORMS[c]]
MF = MF.select("*", *[(F.col(c) - F.expr(f"percentile_approx(`{c}`, 0.5)").over(wpeer)).alias(f"{c}__peer") for c in PEER_COLS]).persist(DISK)
DEP["dep_log_norm"] = ("size anchor", "normal balance, log"); DEP["dep_log_bal"] = ("size anchor", "balance, log")
ANCHORS = ["dep_log_bal", "dep_log_norm", "pay_log_flow"]
FEATS0 = (ANCHORS + [f for f in DEP_FEATS if f not in ANCHORS] + [f"{c}__peer" for c in PEER_COLS]
          + [c for c in MCOLS if "lvl" in FORMS[c]]
          + [f"{c}__{k}" for c in MCOLS for k in ("d6", "d3") if k in FORMS[c]] + STATUS_FEATS)
is_net = lambda f: f.split("__")[0] in NETSET
META = ["cust_pwr_id", "t", "y", "msh", "bal", "bal_norm", "capped", "ab_t", "exit_type", "size_dec"]
keep = F.expr(f"pmod(hash(cust_pwr_id, t, {SEED}), 10000)") < int(NEG_RATE * 10000)
TR = topd(MF.filter("labelled = 1").filter((F.col("y") == 1) | keep)
            .withColumn("w", F.when(F.col("y") == 1, F.lit(1.0)).otherwise(F.lit(1.0 / NEG_RATE)))
            .select(*META, "w", *[F.col(f).cast("float").alias(f) for f in FEATS0]))
TR[FEATS0] = TR[FEATS0].astype("float32")

# ── drop empty / constant, then near-duplicates (keep the better-covered, earlier-listed one) ──
miss0 = TR[FEATS0].isna().mean()
FEATS = [f for f in FEATS0 if miss0[f] < 0.995 and TR[f].nunique(dropna=True) > 1]
S_ = TR[FEATS].sample(min(len(TR), 100_000), random_state=SEED)
S_ = S_.fillna(S_.median()).to_numpy(np.float64)
CM_ = np.nan_to_num(np.corrcoef(S_, rowvar=False))
order = sorted(range(len(FEATS)), key=lambda j: (is_net(FEATS[j]), miss0[FEATS[j]], j))   # model-4 features win ties: a fair ablation
kept, dupes = [], []
for j in order:
    hit = [i for i in kept if abs(CM_[i, j]) > CORR_MAX]
    if hit: dupes.append({"dropped": FEATS[j], "kept": FEATS[hit[0]], "r": float(CM_[hit[0], j])})
    else: kept.append(j)
FEATS = [FEATS[j] for j in sorted(kept)]
DUP = pd.DataFrame(dupes); DUP.to_csv(OUT / "m5_4_dropped_duplicates.csv", index=False)

def fam_of(f):
    if f in ANCHORS: return "size anchor"
    if f in DEP: return "deposit · " + DEP[f][0]
    if f in STATUS_FEATS: return "payment · activity status"
    if f.startswith("miss·"): return "missing share"
    base = f.split("__")[0]
    if base in NETSET: return "network · " + FAMILY.get(base, "other")
    return "payment · " + FAMILY.get(base, "other")
def form_of(f):
    if f in ANCHORS: return "size anchor"
    if f.endswith("__peer"): return "gap to size peers"
    if f.endswith("__d6") or f in ("dep_d6", "dep_live_d6", "dep_sib_d6") or f.endswith("_d6"): return "change vs 4–6 months ago"
    if f.endswith("__d3") or f == "dep_d3": return "change vs last 3 months"
    if f in LATE_FEATS: return "drain in progress"
    if f in NETSET: return "network level"
    return "other"
def label(f):
    if f in DEP: return DEP[f][1]
    if f.startswith("miss·"): return f"share of {f[5:]} features not observed"
    st = {"pay_seen": "has payment history", "pay_silent": "no payments this month (silent)",
          "pay_months_since_act": "months since the last payment", "pay_log_flow": "payment flow in + out, log"}
    if f in st: return st[f]
    base, _, tr = f.partition("__")
    d = DEFN.get(base, base.replace("_", " "))
    return d + {"d6": " — change vs 4–6 months ago", "d3": " — change vs the last 3 months",
                "peer": " — gap to size peers", "": ""}[tr]
FAM = {f: fam_of(f) for f in FEATS}
FAM_SETS = sorted(set(v for v in FAM.values() if v.startswith(("payment", "network"))))
FAM_IDX = {s_: [j for j, f in enumerate(FEATS) if FAM[f] == s_] for s_ in FAM_SETS}
COLS = FEATS + [f"miss·{s_}" for s_ in FAM_SETS]
LABEL = {c: label(c) for c in COLS}; FAMOF = {c: (FAM.get(c) or "missing share") for c in COLS}
FS = pd.DataFrame([{"feature": f, "set": FAM[f], "form": form_of(f), "coverage": 1 - miss0[f]} for f in FEATS])
FST = FS.groupby(["set"]).agg(features=("feature", "size"), median_coverage=("coverage", "median")).reset_index()
NET_KEPT = int(FS.set.str.startswith("network").sum())
FFORM = FS.groupby("form").agg(features=("feature", "size")).reset_index()
FS.assign(label=FS.feature.map(label)).to_csv(OUT / "m5_4_feature_list.csv", index=False)
by_t = topd(MF.groupBy("t").agg(F.count("*").alias("money_still_here"), F.sum("y").alias("leaving_6m"),
                                F.sum(F.when(F.col("y") == 1, F.col("capped"))).alias("leaving_capped"),
                                F.sum("capped").alias("capped")).orderBy("t"))
by_t["month"] = by_t.t.map(ym_of); by_t["rate_6m"] = by_t.leaving_6m / by_t.money_still_here
by_t["labelled"] = by_t.t <= LAST_LABEL_T
R.update(last_label=ym_of(LAST_LABEL_T), net_kept=NET_KEPT, train_rows=len(TR), train_pos=int(TR.y.sum()), n_feats=len(FEATS),
         n_dupes=len(DUP), feature_sets=FST, forms=FFORM,
         base_rate=float(by_t.loc[by_t.labelled, "leaving_6m"].sum() / by_t.loc[by_t.labelled, "money_still_here"].sum()))
B.kv([("labelled months", f"… {ym_of(LAST_LABEL_T)} (outcome fully observed)"),
      ("money-still-here client-months, labelled", int(by_t.loc[by_t.labelled, "money_still_here"].sum())),
      ("leave within 6 months (base rate)", f"{R['base_rate']:.2%}"),
      ("training sample", f"{len(TR):,} rows · {R['train_pos']:,} leaving · stayers at {NEG_RATE:.0%}, weight {1 / NEG_RATE:.1f}"),
      ("features", f"{len(FEATS0)} built → {len(FEATS)} kept ({len(DUP)} near-duplicates dropped) + {len(FAM_SETS)} missing shares"),
      ("…of which network features (model-4 features: the rest)", f"{NET_KEPT} · {len(FEATS) - NET_KEPT}")],
     "4a · Model frame")
B.table(by_t[["month", "money_still_here", "leaving_6m", "rate_6m", "capped", "leaving_capped", "labelled"]],
        "4b · Money-still-here client-months by month", pct=("rate_6m",), money=("capped", "leaving_capped"), save="m5_4b_by_month")
B.table(FST, "4c · Feature sets", pct=("median_coverage",), save="m5_4c_feature_sets")
B.table(FFORM, "4c · Feature forms")
B.show(t0)

## §5 Rolling monthly test folds — simple model and tree model

For each test month T both models are fitted on months whose outcome was known at T (origins ≤ T − 6). **20% of
clients are held out** (all their months, chosen by a fixed hash): on the first fold they choose the ridge penalty
and the tree depth; on every fold they stop the trees early; those from the **3 most recent labelled months** calibrate the tree's probabilities
(isotonic, capped at 95%).
Held-out clients never share a label window with the fit, so validation is as strict as test. Both models then
score every money-still-here client in T.
**Ablation:** a second tree on model 4's features only is fitted in every fold on the same rows, tuned and calibrated the
same way. **2026 months:** the folds continue past the last fully labelled month; each later month's models are fitted on
the origins whose outcome was known at that month (the same rule), so their scores are honest forecasts. Their outcomes are
only partly observed and are read in §8b.

In [ ]:
# =====================================================================
# 5 · ROLLING TEST FOLDS — simple (ridge logistic) and tree model   [OUTPUT BLOCK 5]
# =====================================================================
t0 = time.time(); B = Block("§5 · Rolling monthly test folds, ablation and the 2026 months")
TEST_T = list(range(t_of(TEST_FIRST), LAST_LABEL_T + 1))
if len(TEST_T) < 3: raise RuntimeError(f"only {len(TEST_T)} test months")
FWD_T = list(range(LAST_LABEL_T + 1, END_T + 1)) if FORWARD else []      # later months: partial outcomes only
NETF = [f for f in FEATS if FAM[f].startswith("network")]
BASE_FEATS = [f for f in FEATS if not FAM[f].startswith("network")]   # model 4's features
ABL = bool(GBM and NETF and ABLATION)
BI = np.array([FEATS.index(f) for f in BASE_FEATS])

def prep_fit(Xtr):
    s = Xtr[:min(len(Xtr), 200_000)]
    lo, hi = np.nanquantile(s, CLIP_Q[0], axis=0), np.nanquantile(s, CLIP_Q[1], axis=0)
    xc = np.clip(Xtr, lo, hi)
    mu = np.nanmean(xc, axis=0); sd = np.nanstd(xc, axis=0); sd[~(sd > 1e-9)] = 1.0
    mf = np.stack([np.isnan(Xtr[:, FAM_IDX[s_]]).mean(axis=1) for s_ in FAM_SETS], axis=1) if FAM_SETS else np.zeros((len(Xtr), 0))
    return dict(lo=np.nan_to_num(lo), hi=np.nan_to_num(hi), mu=np.nan_to_num(mu), sd=sd, mm=mf.mean(axis=0), ms=mf.std(axis=0) + 1e-9)

def transform(Xr, P):
    z = np.nan_to_num((np.clip(Xr, P["lo"], P["hi"]) - P["mu"]) / P["sd"], nan=0.0)
    mf = np.stack([np.isnan(Xr[:, FAM_IDX[s_]]).mean(axis=1) for s_ in FAM_SETS], axis=1) if FAM_SETS else np.zeros((len(Xr), 0))
    return np.hstack([z, (mf - P["mm"]) / P["ms"]]).astype(np.float32)

def logit_newton(Z, y, w, l2, max_it=60, tol=1e-6):
    """Weighted ridge logistic by DAMPED Newton; intercept unpenalised; chunked."""
    n, k = Z.shape; w = w / w.mean(); CH = 50_000
    pbar = np.average(y, weights=w); b = np.zeros(k + 1); b[0] = np.log(pbar / (1 - pbar))
    def eta(b): return b[0] + np.concatenate([Z[i:i + CH].astype(np.float64) @ b[1:] for i in range(0, n, CH)])
    def obj(e, b): return float(np.sum(w * (np.logaddexp(0, e) - y * e)) + 0.5 * l2 * b[1:] @ b[1:])
    e = eta(b); f = obj(e, b); conv = False
    for it in range(1, max_it + 1):
        p = 1 / (1 + np.exp(-np.clip(e, -35, 35))); r = w * (p - y); v = w * p * (1 - p)
        g = np.empty(k + 1); g[0] = r.sum(); g[1:] = l2 * b[1:]
        Hs = np.zeros((k + 1, k + 1)); Hs[0, 0] = v.sum()
        for i in range(0, n, CH):
            Zc = Z[i:i + CH].astype(np.float64); rc, vc = r[i:i + CH], v[i:i + CH]
            g[1:] += Zc.T @ rc; Hs[1:, 0] += Zc.T @ vc; Hs[1:, 1:] += (Zc * vc[:, None]).T @ Zc
        Hs[0, 1:] = Hs[1:, 0]; Hs[1:, 1:] += l2 * np.eye(k)
        step = np.linalg.solve(Hs, g); s_ = 1.0
        while True:
            bn = b - s_ * step; en = eta(bn); fn = obj(en, bn)
            if fn <= f or s_ < 1e-8: break
            s_ *= 0.5
        b, e, f_old, f = bn, en, f, fn
        if np.max(np.abs(g)) / n < tol or abs(f_old - f) < 1e-10 * max(1.0, abs(f)): conv = True; break
    return b, dict(converged=conv, iters=it)

def lin_score(b, Z): return 1 / (1 + np.exp(-np.clip(b[0] + Z @ b[1:], -35, 35)))

# ── tree engine wrappers: fit(X, y, w, Xv, yv, wv, depth) → model; predict; contributions ──
def gbm_fit(X, y, w, Xv, yv, wv, depth):
    if GBM == "xgboost":
        import xgboost as xgb
        pr = dict(objective="binary:logistic", eval_metric="logloss", eta=GBM_ETA, max_depth=depth, subsample=0.8,
                  colsample_bytree=0.6, min_child_weight=20, reg_lambda=1.0, tree_method="hist", nthread=os.cpu_count(), seed=SEED)
        dt = xgb.DMatrix(X, label=y, weight=w, missing=np.nan); dv = xgb.DMatrix(Xv, label=yv, weight=wv, missing=np.nan)
        m = xgb.train(pr, dt, num_boost_round=GBM_MAX_TREES, evals=[(dv, "val")], early_stopping_rounds=GBM_EARLY, verbose_eval=False)
        return dict(kind="xgboost", m=m, n=int(m.best_iteration) + 1)
    if GBM == "lightgbm":
        import lightgbm as lgb
        pr = dict(objective="binary", learning_rate=GBM_ETA, num_leaves=min(2 ** depth - 1, 255), max_depth=depth,
                  min_data_in_leaf=200, feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  verbose=-1, seed=SEED, num_threads=os.cpu_count())
        dt = lgb.Dataset(X, y, weight=w); dv = lgb.Dataset(Xv, yv, weight=wv, reference=dt)
        try:    m = lgb.train(pr, dt, GBM_MAX_TREES, valid_sets=[dv], callbacks=[lgb.early_stopping(GBM_EARLY, verbose=False)])
        except (AttributeError, TypeError): m = lgb.train(pr, dt, GBM_MAX_TREES, valid_sets=[dv], early_stopping_rounds=GBM_EARLY, verbose_eval=False)
        return dict(kind="lightgbm", m=m, n=int(m.best_iteration or GBM_MAX_TREES))
    from sklearn.ensemble import HistGradientBoostingClassifier
    m = HistGradientBoostingClassifier(learning_rate=GBM_ETA, max_depth=depth, max_iter=GBM_MAX_TREES, early_stopping=True,
                                       validation_fraction=0.1, n_iter_no_change=GBM_EARLY // 2, l2_regularization=1.0, random_state=SEED)
    m.fit(X, y, sample_weight=w)
    return dict(kind="sklearn", m=m, n=int(m.n_iter_))

def gbm_predict(M, X, contrib=False):
    if M["kind"] == "xgboost":
        import xgboost as xgb
        d = xgb.DMatrix(X, missing=np.nan)
        try:    p = M["m"].predict(d, iteration_range=(0, M["n"]), pred_contribs=contrib)
        except TypeError: p = M["m"].predict(d, ntree_limit=M["n"], pred_contribs=contrib)
        return p[:, :-1] if contrib else p
    if M["kind"] == "lightgbm":
        p = M["m"].predict(X, num_iteration=M["n"], pred_contrib=contrib)
        return p[:, :-1] if contrib else p
    return None if contrib else M["m"].predict_proba(X)[:, 1]

Xall = TR[FEATS].to_numpy(np.float32)
Xb = np.ascontiguousarray(Xall[:, BI]) if ABL else None
yall, wall, tall = TR.y.to_numpy(float), TR.w.to_numpy(float), TR.t.to_numpy()
HOLD = (pd.util.hash_pandas_object(TR.cust_pwr_id.astype(str), index=False).to_numpy() % 1000) < int(HOLD_FRAC * 1000)
TR = TR[META + ["w"]]
L2, DEPTH, DEPTH_B, TUNE = None, None, None, []
SCORES, FOLDS, SAMPLE, FWDS, FWD_FOLDS = [], [], [], [], []
ABS_L, ABS_G, N_SC = np.zeros(len(COLS)), np.zeros(len(FEATS)), 0
rng = np.random.default_rng(SEED)
if GBM: from sklearn.isotonic import IsotonicRegression

def tune_depth(Xm, mf, mv, name):
    for d_ in GBM_GRID:
        Gd = gbm_fit(Xm[mf], yall[mf], wall[mf], Xm[mv], yall[mv], wall[mv], d_)
        TUNE.append({"model": name, "setting": f"depth {d_}, {Gd['n']} trees", "val_auc": auc(yall[mv], gbm_predict(Gd, Xm[mv]))})
    best = max((r for r in TUNE if r["model"] == name), key=lambda r: r["val_auc"])
    return int(best["setting"].split(",")[0].split()[1])

def fit_tree(Xm, mf, mv, mc, depth):
    G = gbm_fit(Xm[mf], yall[mf], wall[mf], Xm[mv], yall[mv], wall[mv], depth)
    G["iso"] = IsotonicRegression(out_of_bounds="clip", y_min=0.001, y_max=P_CAP).fit(
        gbm_predict(G, Xm[mc]), yall[mc], sample_weight=wall[mc])
    return G

FIT = {}
for T in TEST_T + FWD_T:
    tf = time.time(); fwd = T > LAST_LABEL_T
    cut = min(T - H, LAST_LABEL_T)          # origins whose outcome was known at T; later months share the last cut
    if FIT.get("cut") != cut:
        m = tall <= cut; mv = m & HOLD; mf = m & ~HOLD
        if mv.sum() == 0 or yall[mv].sum() == 0: mf, mv = m, m
        mc = mv & (tall >= cut - CAL_RECENT + 1)                  # held-out clients, most recent labelled months
        if yall[mc].sum() < 50: mc = mv
        # ── simple model ─────────────────────────────────────────────
        if L2 is None:
            Pv = prep_fit(Xall[mf]); Zf = transform(Xall[mf], Pv); Zv = transform(Xall[mv], Pv)
            for l2 in L2_GRID:
                bb, _ = logit_newton(Zf, yall[mf], wall[mf], l2)
                TUNE.append({"model": "simple", "setting": f"L2 = {l2:g}", "val_auc": auc(yall[mv], lin_score(bb, Zv))})
            L2 = max((r for r in TUNE if r["model"] == "simple"), key=lambda r: r["val_auc"])["setting"]
            L2 = float(L2.split("= ")[1]); del Zf, Zv
        P = prep_fit(Xall[m]); Z = transform(Xall[m], P)
        b, info = logit_newton(Z, yall[m], wall[m], L2); del Z
        # ── tree on all features, and on model 4's features only (same rows, same split) ──
        G = Gb = None
        if GBM:
            if DEPTH is None: DEPTH = tune_depth(Xall, mf, mv, f"tree ({GBM})")
            G = fit_tree(Xall, mf, mv, mc, DEPTH)
            if ABL:
                if DEPTH_B is None: DEPTH_B = tune_depth(Xb, mf, mv, f"tree, model-4 features ({GBM})")
                Gb = fit_tree(Xb, mf, mv, mc, DEPTH_B)
        FIT = dict(cut=cut, b=b, P=P, info=info, G=G, Gb=Gb, n=int(m.sum()), pos=int(yall[m].sum()))
    b, P, info, G, Gb = FIT["b"], FIT["P"], FIT["info"], FIT["G"], FIT["Gb"]
    # ── score the month ──────────────────────────────────────────────
    te = topd(MF.filter(F.col("t") == T).select(*META, *[F.col(f).cast("float").alias(f) for f in FEATS]))
    Xt = te[FEATS].to_numpy(np.float32); Zt = transform(Xt, P)
    Cl = Zt * b[1:].astype(np.float32); te["p_lin"] = lin_score(b, Zt)
    Cg = None
    if G:
        te["p_gbm_raw"] = gbm_predict(G, Xt); te["p_gbm"] = G["iso"].predict(te.p_gbm_raw.to_numpy()); Cg = gbm_predict(G, Xt, contrib=True)
    else:
        te["p_gbm"] = np.nan
    te["p_gbm_base"] = Gb["iso"].predict(gbm_predict(Gb, np.ascontiguousarray(Xt[:, BI]))) if Gb else np.nan
    RC, RN = (Cg, FEATS) if Cg is not None else (Cl, COLS)          # reason codes: tree SHAP when available
    top = np.argsort(-RC, axis=1)[:, :3]
    for k_ in range(3):
        te[f"r{k_ + 1}"] = [RN[j] for j in top[:, k_]]
    te["tm"] = T
    if fwd:
        FWDS.append(te.drop(columns=FEATS))
        FWD_FOLDS.append({"month": ym_of(T), "trained_on_origins_to": ym_of(cut), "train_rows": FIT["n"], "train_leaving": FIT["pos"],
                          "scored": len(te), "mean_p_tree": float(te.p_gbm.mean()) if G else np.nan, "minutes": (time.time() - tf) / 60})
        print(f"  {ym_of(T)} (2026 check): {len(te):,} scored · models trained on origins ≤ {ym_of(cut)} · {FWD_FOLDS[-1]['minutes']:.1f} min")
        continue
    ABS_L += np.abs(Cl).sum(axis=0); N_SC += len(te)
    if Cg is not None: ABS_G += np.abs(Cg).sum(axis=0)
    si = rng.random(len(te)) < min(1.0, 15_000 / max(len(te), 1))
    SAMPLE.append(pd.DataFrame(Xt[si], columns=FEATS).assign(y=te.y.values[si], size_dec=te.size_dec.values[si]))
    SCORES.append(te.drop(columns=FEATS))
    FOLDS.append({"test_month": ym_of(T), "train_rows": FIT["n"], "train_leaving": FIT["pos"],
                  "simple_converged": info["converged"], "trees": (G or {}).get("n"), "trees_base": (Gb or {}).get("n"),
                  "test_rows": len(te), "test_leaving": int(te.y.sum()), "auc_simple": auc(te.y, te.p_lin),
                  "auc_tree": auc(te.y, te.p_gbm) if G else np.nan, "auc_tree_base": auc(te.y, te.p_gbm_base) if Gb else np.nan,
                  "mean_p_simple": float(te.p_lin.mean()), "mean_p_tree": float(te.p_gbm.mean()) if G else np.nan,
                  "actual_rate": float(te.y.mean()), "minutes": (time.time() - tf) / 60})
    MODEL = dict(b=b, P=P, G=G)
    print(f"  {ym_of(T)}: {len(te):,} scored · AUC simple {FOLDS[-1]['auc_simple']:.3f} · tree {FOLDS[-1]['auc_tree']:.3f} · "
          f"model-4 features {FOLDS[-1]['auc_tree_base']:.3f} · {FOLDS[-1]['minutes']:.1f} min")
SC = pd.concat(SCORES, ignore_index=True); SMP = pd.concat(SAMPLE, ignore_index=True); FO = pd.DataFrame(FOLDS)
FW = pd.concat(FWDS, ignore_index=True) if FWDS else SC.iloc[:0].copy()
PRIMARY = "p_gbm" if GBM else "p_lin"
for d_ in (SC, FW):
    d_["capped"] = d_.capped.fillna(0.0).clip(lower=0); d_["p"] = d_[PRIMARY]
MODEL_FWD = dict(b=FIT["b"], P=FIT["P"], G=FIT["G"], Gb=FIT["Gb"])
R.update(test_months=[ym_of(t) for t in TEST_T], n_test=len(TEST_T), folds=FO, tune=pd.DataFrame(TUNE), L2=L2, depth=DEPTH,
         depth_base=DEPTH_B, engine=GBM, primary="tree model" if GBM else "simple model", ablation=ABL, n_net_feats=len(NETF),
         auc_lin=auc(SC.y, SC.p_lin), auc_gbm=auc(SC.y, SC.p_gbm) if GBM else np.nan,
         auc_gbm_base=auc(SC.y, SC.p_gbm_base) if ABL else np.nan, fwd_folds=pd.DataFrame(FWD_FOLDS),
         fwd_months=[ym_of(t) for t in FWD_T])
B.table(pd.DataFrame(TUNE), "5a · Settings chosen on the held-out clients (first fold)", save="m5_5a_tuning")
B.table(FO, "5b · One fold per test month", pct=("mean_p_simple", "mean_p_tree", "actual_rate"), save="m5_5b_folds")
B.kv([("pooled AUC, simple model", R["auc_lin"]), ("pooled AUC, tree model (all features)", R["auc_gbm"]),
      ("pooled AUC, tree on model 4's features", R["auc_gbm_base"]),
      ("tree − simple", R["auc_gbm"] - R["auc_lin"] if GBM else np.nan),
      ("all features − model 4's features (tree)", R["auc_gbm"] - R["auc_gbm_base"] if ABL else np.nan),
      ("months where all features beat model 4's features", f"{int((FO.auc_tree > FO.auc_tree_base).sum())} of {len(FO)}" if ABL else "—"),
      ("AUC range across months (tree)", f"{FO.auc_tree.min():.3f} – {FO.auc_tree.max():.3f}" if GBM else "—"),
      ("network features in the model / model-4 features", f"{len(NETF)} / {len(BASE_FEATS)}"),
      ("primary model for the list, reasons and examples", R["primary"])], "5c · Discrimination and ablation")
if FWD_FOLDS: B.table(pd.DataFrame(FWD_FOLDS), "5d · Later months (2026): scored with models that saw only outcomes known then",
                      pct=("mean_p_tree",), save="m5_5d_forward_folds")
if not GBM: B.check("tree model", "FIX", "no tree library in new_default: pip install xgboost (or lightgbm) and rerun")
if GBM and not ABL: B.check("ablation", "INFO", "no network features survived, or ABLATION = False: no model-4-features tree")
B.show(t0)

## §6 Predictive power

**In the model:** each feature's share of the average absolute contribution on the test months — exact TreeSHAP
for the tree model, coefficient × standardised value for the simple model. **Alone:** how well the feature ranks
leavers by itself (AUC folded so 0.5 = nothing). Features are also grouped by *form* (size anchor, gap to size
peers, change, drain in progress), which answers whether the model now runs on change rather than size.

In [ ]:
# =====================================================================
# 6 · PREDICTIVE POWER                                              [OUTPUT BLOCK 6]
# =====================================================================
t0 = time.time(); B = Block("§6 · Predictive power")
alone = {}
for f in FEATS:
    a = auc(SMP.y, SMP[f].to_numpy(float))
    alone[f] = (max(a, 1 - a) if pd.notna(a) else np.nan, "higher" if (pd.notna(a) and a >= 0.5) else "lower")
use_g = GBM is not None and ABS_G.sum() > 0
shr = (ABS_G / ABS_G.sum()) if use_g else (ABS_L[:len(FEATS)] / ABS_L.sum())
lin_shr = ABS_L / ABS_L.sum()
IMP = pd.DataFrame([{"feature": f, "label": LABEL[f], "set": FAM[f], "form": form_of(f), "share": shr[j],
                     "share_simple": lin_shr[j], "simple_sign": "risk up when high" if MODEL["b"][1 + j] > 0 else "risk up when low",
                     "alone_auc": alone[f][0], "leavers_sit": alone[f][1]} for j, f in enumerate(FEATS)])
IMP = IMP.sort_values("share", ascending=False).reset_index(drop=True)
IMP.to_csv(OUT / "m5_6_importance_all.csv", index=False)
FAMI = IMP.groupby("set").agg(share=("share", "sum"), share_simple=("share_simple", "sum"), features=("feature", "size"),
                              best_alone=("alone_auc", "max")).sort_values("share", ascending=False).reset_index()
if not use_g: FAMI.loc[len(FAMI)] = ["missing share", float(lin_shr[len(FEATS):].sum()), float(lin_shr[len(FEATS):].sum()), len(FAM_SETS), np.nan]
FORM = IMP.groupby("form").agg(share=("share", "sum"), share_simple=("share_simple", "sum"), features=("feature", "size")) \
          .sort_values("share", ascending=False).reset_index()
R.update(imp=IMP, fam_imp=FAMI, form_imp=FORM, imp_basis="tree SHAP" if use_g else "simple model contributions",
         miss_share_simple=float(lin_shr[len(FEATS):].sum()))
B.table(IMP.head(25), f"6a · Top 25 features ({R['imp_basis']}; share_simple = the simple model's share)",
        pct=("share", "share_simple"), save="m5_6a_top")
B.table(FAMI, "6b · Feature sets", pct=("share", "share_simple"), save="m5_6b_sets")
B.table(FORM, "6c · Feature forms: size, peer gap, change, drain in progress", pct=("share", "share_simple"), save="m5_6c_forms")
B.kv([("missing-share columns in the simple model", f"{R['miss_share_simple']:.1%} of contribution (model 1: 24.5% from 89 flags)")],
     "6d · Missingness")
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
t20 = IMP.head(20)[::-1]
ax[0].barh([l[:60] for l in t20.label], t20.share, color="#1f4e79"); ax[0].set_title(f"Top 20 features ({R['imp_basis']})")
f12 = FAMI.head(12)[::-1]
ax[1].barh(f12.set, f12.share, color="#c55a11"); ax[1].set_title("Feature sets")
for a in ax: a.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
savefig(fig, "m5_6_importance")
B.show(t0)

## §6b What the network features add

AUC by test month for the simple model, the tree on model 4's features and the tree on all features; the network families'
share of the tree's attribution; the top network features; and each processor, NAICS-mix and counterparty signal on its
own, **size-matched**: how often leavers' months sit beyond the stayers' 10th or 90th percentile of the same size decile.

In [ ]:
# =====================================================================
# 6b · WHAT THE NETWORK FEATURES ADD                                 [OUTPUT BLOCK 6b]
# =====================================================================
t0 = time.time(); B = Block("§6b · What the network features add")
ABLT = FO[["test_month", "test_leaving", "auc_simple", "auc_tree_base", "auc_tree"]].copy()
ABLT["all_minus_model4"] = ABLT.auc_tree - ABLT.auc_tree_base
NETI = IMP[IMP.set.str.startswith("network")].copy()
NETI["rank"] = NETI.index + 1                        # rank among all features (IMP is sorted by share)
NFAMI = FAMI[FAMI.set.str.startswith("network")].copy()

# ── each network signal on its own: leavers vs stayers of the same size (test months) ──
SIG_FAMS = ("network · processors", "network · NAICS mix", "network · PNC counterparties", "network · counterparty health")
_w = SMP.size_dec.where(SMP.y == 1).value_counts(normalize=True)          # leavers' size mix
rows = []
for f in [f for f in NETF if FAM[f] in SIG_FAMS]:
    x, y_, dec = SMP[f].to_numpy(float), SMP.y.to_numpy(), SMP.size_dec.to_numpy()
    ok = ~np.isnan(x)
    if ok.sum() < 200 or y_[ok].sum() < 20: continue
    r_ = {"feature": f, "label": LABEL[f], "set": FAM[f].split(" · ")[1], "form": form_of(f)}
    a = auc(y_[ok], x[ok]); r_["alone_auc"] = max(a, 1 - a); r_["leavers"] = "higher" if a >= 0.5 else "lower"
    # rate beyond the stayers' 90th (or below the 10th) percentile of the same size decile, stayers reweighted to the leavers' mix
    hit_l, hit_s = {"rises": [], "falls": []}, {"rises": [], "falls": []}
    for dd, wt in _w.items():
        s_ = x[ok & (dec == dd) & (y_ == 0)]; l_ = x[ok & (dec == dd) & (y_ == 1)]
        if len(s_) < 30 or not len(l_): continue
        q10, q90 = np.quantile(s_, 0.10), np.quantile(s_, 0.90)
        for side, fl, fs in (("rises", l_ > q90, s_ > q90), ("falls", l_ < q10, s_ < q10)):
            hit_l[side].append(wt * fl.mean()); hit_s[side].append(wt * fs.mean())
    lift = {k: (sum(hit_l[k]) / sum(hit_s[k]) if hit_s[k] and sum(hit_s[k]) > 0 else np.nan) for k in hit_l}
    side = "rises" if np.nan_to_num(lift["rises"]) >= np.nan_to_num(lift["falls"]) else "falls"
    r_.update(signal=side, leavers_beyond=sum(hit_l[side]) if hit_l[side] else np.nan,
              stayers_beyond=sum(hit_s[side]) if hit_s[side] else np.nan, lift=lift[side],
              coverage=float(ok.mean()), shap_share=float(IMP.loc[IMP.feature == f, "share"].iloc[0]),
              shap_rank=int(IMP.index[IMP.feature == f][0]) + 1)
    rows.append(r_)
NSIG = pd.DataFrame(rows)
if len(NSIG): NSIG = NSIG.sort_values(["alone_auc"], ascending=False).reset_index(drop=True)
best_pn = NSIG[NSIG.set.isin(["processors", "NAICS mix"])].alone_auc.max() if len(NSIG) else np.nan
R.update(abl_t=ABLT, net_imp=NETI.head(25), net_fam_imp=NFAMI, net_sig=NSIG.head(30), best_pn_auc=float(best_pn) if pd.notna(best_pn) else np.nan,
         n_net_top25=int(IMP.head(25).set.str.startswith("network").sum()),
         net_share=float(NFAMI.share.sum()) if len(NFAMI) else 0.0)
B.table(ABLT, "6e · AUC by test month: simple, tree on model 4's features, tree on all features", save="m5_6e_ablation")
B.kv([("pooled AUC: all features / model 4's features", f"{R['auc_gbm']:.4f} / {R['auc_gbm_base']:.4f}" if ABL else "—"),
      ("network features' share of the tree's attribution", f"{R['net_share']:.1%}"),
      ("network features in the top 25", R["n_net_top25"]),
      ("best processor or NAICS-mix feature on its own (AUC)", f"{R['best_pn_auc']:.3f}" if pd.notna(R["best_pn_auc"]) else "—")],
     "6f · Summary")
B.table(NFAMI, "6g · Network families (tree SHAP share)", pct=("share", "share_simple"), save="m5_6g_network_families")
B.table(NETI.head(25)[["rank", "feature", "label", "set", "form", "share", "alone_auc", "leavers_sit"]],
        "6h · Top network features (rank among all features)", pct=("share",), save="m5_6h_network_top")
B.table(NSIG.head(30), "6i · Network signals on their own, size-matched (beyond = beyond the stayers' 10th/90th percentile)",
        pct=("leavers_beyond", "stayers_beyond", "coverage", "shap_share"), save="m5_6i_network_signals")
if ABL:
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
    for c_, lab_, col_ in (("auc_simple", "simple", "grey"), ("auc_tree_base", "tree, model 4's features", "#c55a11"),
                           ("auc_tree", "tree, all features", "black")):
        ax[0].plot(ABLT.test_month, ABLT[c_], marker="o", label=lab_, color=col_)
    ax[0].set_title("AUC by test month"); ax[0].legend(fontsize=8); ax[0].tick_params(axis="x", rotation=90)
    f12 = NFAMI.head(8)[::-1]
    ax[1].barh(f12.set.str.replace("network · ", ""), f12.share, color="#1f4e79"); ax[1].set_title("Network families: share of the tree's attribution")
    ax[1].xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
    fig.tight_layout(); savefig(fig, "m5_6b_ablation")
    figdata("m5_6b_ablation", ABLT)
B.show(t0)

## §7 Checking the largest departures

A handful of very large clients carry most of the dollars, so each of the largest departures is checked in the
data before its dollars are counted. For every client: did the balance **come back** (≥ 50% of normal after the
exit month)? Did any of its accounts get **re-linked** to another customer id? Did **other customers in the same
relationship** gain what this one lost (money moving inside PNC)? Does the client **keep transacting** here
afterwards? Were payments to **its own name at other banks** visible around the exit? Verdicts:

| Verdict (checked in this order) | Rule | Counted as a loss? |
|---|---|---|
| came back | balance returned to ≥ 50% of normal | no |
| moved to another bank | payments to its own name at other banks (exit month and 3 before) exceed its own level 6–9 months earlier by ≥ 50% of the drop | yes |
| moved within PNC | a re-linked account carries ≥ 50% of the drop afterwards, or siblings gained 50–150% of it | no |
| still transacting | payment activity after the exit ≥ 50% of before | yes, flagged |
| departed | none of the above | yes |

In [ ]:
# =====================================================================
# 7 · CHECKING THE LARGEST DEPARTURES                              [OUTPUT BLOCK 7]
# =====================================================================
t0 = time.time(); B = Block("§7 · Checking the largest departures")
ex = topd(FEAT.filter(F.col("ab_t").isNotNull() & F.col("ab_t").between(MIN_HIST + 1, LAST_LABEL_T + H))
              .groupBy("cust_pwr_id").agg(F.first("ab_t").alias("ab_t"), F.first("exit_type").alias("exit_type"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 1, F.col("bal_norm"))).alias("norm"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 3, F.col("msh"))).alias("msh_m3"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 3, F.col("capped"))).alias("capped_m3")))
ex = ex.dropna(subset=["norm"])
AUD_CACHE = {}
def audit(ids):
    """Evidence and verdict for each departing client id (cached)."""
    new = [c for c in dict.fromkeys(ids) if c not in AUD_CACHE]
    if new:
        P_ = topd(FEAT.filter(F.col("cust_pwr_id").isin(new)).select("cust_pwr_id", "t", "bal", "bal_norm", "live", "ab_t",
                                                                     "exit_type", "first_t", "rltn_pwr_id", "raw_ntx", "raw_selfpay_out"))
        accs = [r[0] for r in AMP.filter(F.col("cust_pwr_id").isin(new)).select("acct_full_acct_id").distinct().collect()]
        RL = topd(AMP.filter(F.col("acct_full_acct_id").isin(accs or ["__none__"]))
                     .select("acct_full_acct_id", "t", "cust_pwr_id", "live", "acct_bal"))
        rls = P_.dropna(subset=["rltn_pwr_id"]).groupby("cust_pwr_id").rltn_pwr_id.last()
        SIB = topd(FEAT.filter(F.col("rltn_pwr_id").isin(list(set(rls.values)) or ["__none__"]))
                       .groupBy("rltn_pwr_id", "t", "cust_pwr_id").agg(F.sum("bal").alias("bal"))) if len(rls) else pd.DataFrame(
                       columns=["rltn_pwr_id", "t", "cust_pwr_id", "bal"])
        for c in new:
            p = P_[P_.cust_pwr_id == c].sort_values("t").set_index("t")
            if not len(p) or p.ab_t.isna().all(): AUD_CACHE[c] = {"verdict": "no exit"}; continue
            e = int(p.ab_t.dropna().iloc[0]); norm = p.bal_norm.get(e - 1, np.nan)
            pre = p.bal.loc[e - 3:e - 1].mean(); post = p.bal.loc[e:e + 2].mean()
            back = p.bal.loc[e + 1:].max() / norm if norm and len(p.bal.loc[e + 1:]) else np.nan
            r_ = RL[RL.acct_full_acct_id.isin(RL.loc[RL.cust_pwr_id == c, "acct_full_acct_id"])]
            relinked = int(r_[(r_.t >= e - 1) & (r_.cust_pwr_id != c)].acct_full_acct_id.nunique())
            rk_ = r_[r_.t.between(e, e + 2) & (r_.cust_pwr_id != c)]
            relink_share = rk_.groupby("t").acct_bal.sum().mean() / (pre - post) if len(rk_) and pre - post > 0 else 0.0
            absorb = np.nan
            if c in rls.index:
                sb = SIB[(SIB.rltn_pwr_id == rls[c]) & (SIB.cust_pwr_id != c)].groupby("t").bal.sum()
                if len(sb) and pre - post > 0:
                    absorb = (sb.loc[e:e + 2].mean() - sb.loc[e - 3:e - 1].mean()) / (pre - post)
            act_b = p.raw_ntx.loc[e - 6:e - 4].mean(); act_a = p.raw_ntx.loc[e:e + 2].mean()
            act = act_a / act_b if act_b and act_b > 0 else np.nan
            selfp = p.raw_selfpay_out.loc[e - 3:e].sum() / (pre - post) if pre - post > 0 else np.nan
            _b = p.raw_selfpay_out.loc[e - 9:e - 4]
            sp_base = float(_b.mean()) * 4 if _b.notna().any() else 0.0          # same 4-month length, 6–9 months earlier
            selfx = (p.raw_selfpay_out.loc[e - 3:e].sum() - sp_base) / (pre - post) if pre - post > 0 else np.nan
            if pd.notna(back) and back >= 0.5: v = "came back"
            elif pd.notna(selfx) and selfx >= 0.5: v = "moved to another bank"
            elif relink_share >= 0.5 or (pd.notna(absorb) and 0.5 <= absorb <= 1.5): v = "moved within PNC"
            elif pd.notna(act) and act >= 0.5: v = "still transacting"
            else: v = "departed"
            AUD_CACHE[c] = {"client": cid(c), "exit_type": p.exit_type.dropna().iloc[0] if p.exit_type.notna().any() else None,
                            "exit_month": ym_of(e), "normal": norm, "months_observed_before": int(e - p.first_t.iloc[0]),
                            "balance_after_vs_normal": back, "accounts_relinked": relinked, "relinked_balance_vs_drop": relink_share, "siblings_absorbed": absorb,
                            "activity_after_vs_before": act, "own_name_out_vs_drop": selfp, "own_name_excess_vs_drop": selfx,
                            "live_accounts_at_end": int(r_[(r_.t == END_T) & (r_.cust_pwr_id == c)].live.sum()),
                            "verdict": v}
    return pd.DataFrame([{"cust_pwr_id": c, **AUD_CACHE[c]} for c in dict.fromkeys(ids)])

pool_all = (SC[SC.y == 1].sort_values("tm").groupby("cust_pwr_id")
              .agg(ab_t=("ab_t", "first"), exit_type=("exit_type", "first"), capped=("capped", "max"),
                   bal_norm=("bal_norm", "first"), tm=("tm", "first")).reset_index())
top_ids = list(ex.sort_values("norm", ascending=False).cust_pwr_id.head(N_AUDIT)) + \
          list(pool_all.sort_values("capped", ascending=False).cust_pwr_id.head(N_AUDIT))
AUD = audit(top_ids)
AUD = AUD.merge(pool_all[["cust_pwr_id", "capped"]].rename(columns={"capped": "test_pool_capped"}), on="cust_pwr_id", how="left")
AUD.to_csv(OUT / "m5_7_audit_FULL_IDS.csv", index=False)
EXCL = set(AUD.loc[AUD.verdict.isin(["came back", "moved within PNC"]), "cust_pwr_id"])
VC = AUD.groupby("verdict").agg(clients=("client", "size"), normal=("normal", "sum")).reset_index()
R.update(largest_excluded=bool(len(AUD) and pool_all.sort_values("capped", ascending=False).cust_pwr_id.iloc[0] in EXCL),
         audit=AUD.rename(columns={"cust_pwr_id": "power_id"}) if SHOW_PWR_IDS else AUD.drop(columns="cust_pwr_id"), audit_counts=VC, n_not_clean=int((AUD.verdict != "departed").sum()),
         n_audited=len(AUD), excl_ids=EXCL)
B.table(R["audit"], f"7a · The largest departures ({len(AUD)} clients: top {N_AUDIT} of the window, top {N_AUDIT} of the test months)",
        pct=("balance_after_vs_normal", "relinked_balance_vs_drop", "siblings_absorbed", "activity_after_vs_before", "own_name_out_vs_drop", "own_name_excess_vs_drop"),
        money=("normal", "test_pool_capped"))
B.table(VC, "7b · Verdicts", money=("normal",))
B.check("largest departures", "INFO", f"{len(EXCL)} of {len(AUD)} are not counted as losses (came back or moved within PNC); "
        "full ids in m5_7_audit_FULL_IDS.csv for an RM check")
B.show(t0)

## §8 How much is at risk; accuracy and savings against calls

The list each month: money-still-here clients ranked by `p × normal balance^α`, top K called. A departing client
is credited once, at its first alert, with its capped balance (the lesser of current and normal). Dollars are
annualised over the test months. Savings = dollars reached × an assumed save rate; there is no call cost.
Headline dollars are also shown **without the largest 1 and 5 departing clients** and **without the clients §7
found were not real losses**. Simple and tree model are compared on the same rows with a paired bootstrap over
departing clients.
The tree on model 4's features (`tree_base`) is ranked and valued the same way, with a paired bootstrap for the
difference the network features make.

In [ ]:
# =====================================================================
# 8 · AT RISK · ACCURACY VS CALLS · SAVINGS · MODEL COMPARISON      [OUTPUT BLOCK 8]
# =====================================================================
t0 = time.time(); B = Block("§8 · At risk, accuracy, savings")
span = (LAST_LABEL_T + H) - (MIN_HIST + 1) + 1
ex["band"] = pd.cut(ex.norm, [b[0] for b in BANDS] + [BANDS[-1][1]], labels=[b[2] for b in BANDS], right=False)
exv = ex[~ex.cust_pwr_id.isin(EXCL)]
AR = exv.groupby(["band", "exit_type"], observed=False).agg(clients=("norm", "size"), normal_balance=("norm", "sum")).reset_index()
AR["per_year"] = AR.normal_balance * 12 / span
R.update(at_risk_year=exv.norm.sum() * 12 / span, exits_per_year=len(exv) * 12 / span, at_risk_tab=AR,
         msh_m3=float(exv.msh_m3.mean()), capped_m4_year=float(exv.capped_m3.fillna(0).sum() * 12 / span),
         bal_exit_share=float(exv.loc[exv.exit_type == "balance exit", "norm"].sum() / exv.norm.sum()),
         top_client_share=float(exv.norm.max() / exv.norm.sum()), excl_normal=float(ex.loc[ex.cust_pwr_id.isin(EXCL), "norm"].sum() * 12 / span))
B.kv([("clients whose money left (per year, checked exclusions removed)", f"{R['exits_per_year']:,.0f}"),
      ("their normal balance (per year)", f"${R['at_risk_year']:,.0f}"),
      ("…of which balance exits (accounts left open)", f"{R['bal_exit_share']:.1%}"),
      ("…still here 3 months before (clients) / capped dollars per year", f"{R['msh_m3']:.1%} / ${R['capped_m4_year']:,.0f}"),
      ("largest single client's share", f"{R['top_client_share']:.1%}"),
      ("removed by §7 (came back / moved within PNC), per year", f"${R['excl_normal']:,.0f}")], "8a · How much is at risk")
B.table(AR, "8a · By size band and exit type (annualised)", money=("normal_balance", "per_year"), save="m5_8a_at_risk")

Q = SC.copy(); Q["norm"] = Q.bal_norm.clip(lower=1.0)
yr = 12 / R["n_test"]
MODELS = [("simple", "p_lin")] + ([("tree", "p_gbm")] if GBM else []) + ([("tree_base", "p_gbm_base")] if ABL else [])
first_alert, per_month = {}, []
for mn, pc in MODELS:
    for a in ALPHAS:
        Q[f"rank_{mn}_{a}"] = Q.assign(_s=Q[pc] * Q.norm ** a).groupby("tm")["_s"].rank(ascending=False, method="first")
        for K in CALLS:
            on = Q[Q[f"rank_{mn}_{a}"] <= K]
            for T, g in Q.groupby("tm"):
                o = on[on["tm"] == T]; lv = g[g.y == 1]
                per_month.append({"model": mn, "alpha": a, "calls": K, "month": ym_of(T), "precision": o.y.mean(),
                                  "recall_clients": o.y.sum() / max(len(lv), 1),
                                  "recall_dollars": o.loc[o.y == 1, "capped"].sum() / max(lv.capped.sum(), 1)})
            first_alert[(mn, a, K)] = on[on.y == 1].sort_values("tm").drop_duplicates("cust_pwr_id")
PM_ = pd.DataFrame(per_month)
pool = pool_all.copy(); POOL = pool.capped.sum()
top_pool = list(pool.sort_values("capped", ascending=False).cust_pwr_id)
def reached(fa, drop=()):
    f = fa[~fa.cust_pwr_id.isin(drop)]; return f.capped.sum() * yr, len(f)
rows = []
for (mn, a, K), fa in first_alert.items():
    d_, n_ = reached(fa)
    rows.append({"model": mn, "alpha": a, "calls": K, "clients_reached": n_, "share_of_departing": n_ / max(len(pool), 1),
                 "dollars_reached_year": d_, "share_of_dollars": d_ / yr / max(POOL, 1),
                 **{f"save_{int(s * 100)}pct_year": d_ * s for s in SAVE_RATES}})
VAL = pd.DataFrame(rows)
ACC = (PM_.groupby(["model", "alpha", "calls"]).agg(precision_mean=("precision", "mean"), precision_min=("precision", "min"),
                                                    precision_max=("precision", "max"), recall_clients=("recall_clients", "mean"),
                                                    recall_dollars=("recall_dollars", "mean")).reset_index())
MAIN = "tree" if GBM else "simple"
fa0 = first_alert[(MAIN, A_MAIN, K_MAIN)]
variants = [("all departing clients", ()), *[(f"without the largest {k}", tuple(top_pool[:k])) for k in TOP_EXCL],
            ("without clients §7 found were not losses", tuple(EXCL))]
HV = []
for nm, drop in variants:
    d_, n_ = reached(fa0, drop); pl = pool[~pool.cust_pwr_id.isin(drop)].capped.sum() * yr
    HV.append({"dollars": nm, "clients_reached": n_, "reached_per_year": d_, "departing_per_year": pl, "share": d_ / max(pl, 1)})
d_ = float(fa0.capped.clip(upper=CAP_CREDIT).sum() * yr); pl = float(pool.capped.clip(upper=CAP_CREDIT).sum() * yr)
HV.append({"dollars": f"each client's credit capped at ${CAP_CREDIT / 1e6:,.0f}m", "clients_reached": len(fa0),
           "reached_per_year": d_, "departing_per_year": pl, "share": d_ / max(pl, 1)})
HV = pd.DataFrame(HV)
# paired bootstrap over departing clients: tree − simple at the main operating point
val_of = lambda mn: pool.cust_pwr_id.map(first_alert[(mn, A_MAIN, K_MAIN)].set_index("cust_pwr_id").capped).fillna(0.0).to_numpy()
cs = val_of("simple"); ct = val_of("tree") if GBM else cs; cb = val_of("tree_base") if ABL else ct
bs_t, bs_d, bs_b = [], [], []
for _ in range(2000):
    i = rng.integers(0, len(pool), len(pool)); bs_t.append(ct[i].sum() * yr); bs_d.append((ct[i] - cs[i]).sum() * yr)
    bs_b.append((ct[i] - cb[i]).sum() * yr)
_pm = PM_[(PM_.model == MAIN) & (PM_.alpha == A_MAIN) & (PM_.calls == K_MAIN)]
R.update(pool_year=POOL * yr, n_departing=len(pool), val=VAL, acc=ACC, per_month=PM_, headline=HV,
         main_reached=float(HV.reached_per_year.iloc[0]), main_ci=(float(np.percentile(bs_t, 2.5)), float(np.percentile(bs_t, 97.5))),
         main_share=float(HV.share.iloc[0]), diff=float((ct - cs).sum() * yr),
         diff_ci=(float(np.percentile(bs_d, 2.5)), float(np.percentile(bs_d, 97.5))),
         diff_base=float((ct - cb).sum() * yr), diff_base_ci=(float(np.percentile(bs_b, 2.5)), float(np.percentile(bs_b, 97.5))),
         base_reached=float(cb.sum() * yr),
         prec_ratio=float(_pm.precision.max() / _pm.precision.min()) if _pm.precision.min() > 0 else np.inf,
         drop_top5=1 - float(HV.reached_per_year.iloc[2] / HV.reached_per_year.iloc[0]) if len(HV) > 2 and HV.reached_per_year.iloc[0] else np.nan,
         drop_cap=1 - float(HV.reached_per_year.iloc[-1] / HV.reached_per_year.iloc[0]) if HV.reached_per_year.iloc[0] else np.nan)
for nm_, df_ in (("m5_8_accuracy_all", ACC), ("m5_8_value_all", VAL), ("m5_8_per_month_all", PM_), ("m5_8_headline", HV),
                 ("m5_8_at_risk", AR)): figdata(nm_, df_)
CMP = VAL[(VAL.alpha == A_MAIN)].pivot(index="calls", columns="model", values="dollars_reached_year").reset_index()
CMP = CMP.merge(ACC[ACC.alpha == A_MAIN].pivot(index="calls", columns="model", values="precision_mean").add_prefix("precision_").reset_index(), on="calls")
B.kv([(f"departing clients with money still here, test months", f"{len(pool):,} · ${POOL * yr:,.0f} capped per year"),
      (f"reached at {K_MAIN:,} calls, α = {A_MAIN} ({MAIN} model)",
       f"{len(fa0):,} clients · ${R['main_reached']:,.0f}/yr (95% {R['main_ci'][0]:,.0f} – {R['main_ci'][1]:,.0f})"),
      ("tree − simple, dollars reached per year", f"${R['diff']:,.0f} (95% {R['diff_ci'][0]:,.0f} – {R['diff_ci'][1]:,.0f})" if GBM else "—"),
      ("all features − model 4's features (trees), dollars reached per year",
       f"${R['diff_base']:,.0f} (95% {R['diff_base_ci'][0]:,.0f} – {R['diff_base_ci'][1]:,.0f})" if ABL else "—"),
      ("best ÷ worst month precision", f"{R['prec_ratio']:.2f}×")], "8b · Headline")
B.table(HV, f"8b · Headline dollars at {K_MAIN:,} calls, α = {A_MAIN}, with and without the largest clients",
        pct=("share",), money=("reached_per_year", "departing_per_year"), save="m5_8b_headline")
B.table(CMP, f"8c · Simple, tree, and tree on model 4's features (tree_base) at α = {A_MAIN}: dollars reached per year and precision",
        pct=tuple(c for c in CMP.columns if c.startswith("precision_")),
        money=tuple(c for c in CMP.columns if c in ("simple", "tree", "tree_base")), save="m5_8c_compare")
B.table(ACC[ACC.model == MAIN], f"8d · Accuracy against calls ({MAIN} model; mean over test months, min/max = spread)",
        pct=("precision_mean", "precision_min", "precision_max", "recall_clients", "recall_dollars"), save="m5_8d_accuracy")
B.table(VAL[VAL.model == MAIN].drop(columns="model"), f"8e · Dollars reached and potential savings per year ({MAIN} model, no call cost)",
        pct=("share_of_departing", "share_of_dollars"),
        money=tuple(["dollars_reached_year"] + [f"save_{int(s * 100)}pct_year" for s in SAVE_RATES]), save="m5_8e_value")
BYM = (PM_[(PM_.calls == K_MAIN) & (PM_.alpha == A_MAIN)].pivot(index="month", columns="model", values="precision")
          .add_prefix("precision_").reset_index().merge(FO[["test_month"] + [f"auc_{mn}" for mn, _ in MODELS]], left_on="month",
                                                        right_on="test_month").drop(columns="test_month"))
B.table(BYM, f"8f · By test month at {K_MAIN:,} calls, α = {A_MAIN}", pct=tuple(c for c in BYM.columns if c.startswith("precision_")),
        save="m5_8f_by_month")
SC["bin"] = pd.qcut(SC.p.rank(method="first"), 10, labels=False)
CAL = SC.groupby("bin").agg(mean_predicted=("p", "mean"), actual=("y", "mean"), rows=("y", "size")).reset_index()
CAL["ratio"] = CAL.actual / CAL.mean_predicted
CAL_S = SC.assign(b_=pd.qcut(SC.p_lin.rank(method="first"), 10, labels=False)).groupby("b_").agg(
    mean_predicted=("p_lin", "mean"), actual=("y", "mean")).reset_index().rename(columns={"b_": "bin"})
R.update(cal=CAL, cal_simple=CAL_S, bym=BYM, cmp=CMP,
         top_dec_ratio=float(CAL.mean_predicted.iloc[-1] / CAL.actual.iloc[-1]) if CAL.actual.iloc[-1] else np.nan)
for nm_, df_ in (("m5_8_calibration_primary", CAL), ("m5_8_calibration_simple", CAL_S), ("m5_8_by_month", BYM)): figdata(nm_, df_)
B.table(CAL, f"8g · Calibration ({MAIN} model), predicted vs actual by decile", pct=("mean_predicted", "actual"))

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for a in ALPHAS:
    d = ACC[(ACC.model == MAIN) & (ACC.alpha == a)]; v = VAL[(VAL.model == MAIN) & (VAL.alpha == a)]
    ax[0].plot(d.calls, d.precision_mean, marker="o", label=f"α={a}"); ax[1].plot(d.calls, d.recall_dollars, marker="o", label=f"α={a}")
    ax[2].plot(v.calls, v.dollars_reached_year / 1e6, marker="o", label=f"α={a}")
for mn, _ in MODELS:
    v = VAL[(VAL.model == mn) & (VAL.alpha == A_MAIN)]
    ax[2].plot(v.calls, v.dollars_reached_year / 1e6, ls=":", color={"simple": "grey", "tree": "black", "tree_base": "#c55a11"}[mn], label=f"{mn}, α={A_MAIN}")
for a_, t_ in zip(ax, ["Precision", "Share of departing dollars reached (per month)", "Dollars reached per year ($m)"]):
    a_.set_title(t_); a_.set_xlabel("calls per month"); a_.legend(fontsize=8)
ax[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax[1].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
savefig(fig, "m5_8_accuracy_vs_calls")
fig, ax = plt.subplots(1, len(SAVE_RATES), figsize=(4 * len(SAVE_RATES), 3.6))
for i, s in enumerate(SAVE_RATES):
    for a in ALPHAS:
        v = VAL[(VAL.model == MAIN) & (VAL.alpha == a)]; ax[i].plot(v.calls, v[f"save_{int(s * 100)}pct_year"] / 1e6, marker="o", label=f"α={a}")
    ax[i].set_title(f"Savings/yr at {s:.0%} save rate ($m)"); ax[i].set_xlabel("calls per month"); ax[i].legend(fontsize=8)
savefig(fig, "m5_8_savings_vs_calls")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for mn, _ in MODELS:
    ax[0].plot(BYM.month, BYM[f"precision_{mn}"], marker="o", label=mn)
    ax[1].plot(FO.test_month, FO[f"auc_{mn}"], marker="o", label=mn)
ax[0].set_title(f"Precision by test month ({K_MAIN:,} calls, α={A_MAIN})"); ax[1].set_title("AUC by test month")
for a_ in ax: a_.legend(); a_.tick_params(axis="x", rotation=90)
ax[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
savefig(fig, "m5_8_by_month")
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.plot(CAL.mean_predicted, CAL.actual, marker="o"); mx = max(CAL.mean_predicted.max(), CAL.actual.max())
ax.plot([0, mx], [0, mx], color="grey", lw=0.8); ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title("Calibration by decile")
savefig(fig, "m5_8_calibration")
B.show(t0)

## §8b The 2026 months: forward check with partial follow-up

Exits are fully observed only through `END_T − 2` (a balance exit needs three months, a full exit two). Each 2026 month is
evaluated on the exits seen so far — 5 months of follow-up for 2026-01, down to 1 month — and compared with the test
months **cut to the same follow-up**, so a shorter window is not mistaken for a weaker model. Months with no follow-up yet
are scored only; the last of them is the current list (§10b).

In [ ]:
# =====================================================================
# 8b · THE 2026 MONTHS: FORWARD CHECK WITH PARTIAL FOLLOW-UP          [OUTPUT BLOCK 8b]
# =====================================================================
t0 = time.time(); B = Block("§8b · The 2026 months: forward check with partial follow-up")
OBS_END = END_T - max(CONFIRM_M, BAL_EXIT_HOLD) + 1       # an exit starting by this month is fully observed (both kinds)
def y_within(df, Fm):
    return (df.ab_t.notna() & (df.ab_t > df["tm"]) & (df.ab_t <= df["tm"] + Fm)).astype(int).to_numpy()
def month_stats(g, pc, yv):
    s_ = g[pc].to_numpy(float) * g.bal_norm.clip(lower=1.0).to_numpy(float) ** A_MAIN
    top_ = np.argsort(-s_, kind="stable")[:K_MAIN]; cap = g.capped.to_numpy(float)
    return dict(auc=auc(yv, g[pc]), precision=float(yv[top_].mean()) if len(top_) else np.nan,
                recall_dollars=float((cap[top_] * yv[top_]).sum() / max((cap * yv).sum(), 1.0)), leaving=int(yv.sum()), rate=float(yv.mean()))
_tmemo = {}
def test_at(mn, pc, Fm):
    """Test months scored the same way, with outcomes cut to the same follow-up."""
    if (mn, Fm) not in _tmemo:
        _tmemo[(mn, Fm)] = pd.DataFrame([month_stats(h, pc, y_within(h, Fm)) for _, h in SC.groupby("tm")]).mean()
    return _tmemo[(mn, Fm)]
rows, pend = [], []
for T, g in FW.groupby("tm"):
    Fm = min(H, OBS_END - int(T))
    if Fm < 1:
        pend.append({"month": ym_of(T), "scored": len(g), "why": "no observed follow-up yet"}); continue
    yv = y_within(g, Fm)
    for mn, pc in MODELS:
        fs, ts = month_stats(g, pc, yv), test_at(mn, pc, Fm)
        rows.append({"month": ym_of(T), "follow_up_months": Fm, "model": mn, "scored": len(g), "leaving_so_far": fs["leaving"],
                     "rate": fs["rate"], "test_rate": ts["rate"], "auc": fs["auc"], "test_auc": ts["auc"],
                     "precision": fs["precision"], "test_precision": ts["precision"],
                     "recall_dollars": fs["recall_dollars"], "test_recall_dollars": ts["recall_dollars"]})
FWT = pd.DataFrame(rows)
if len(FWT):
    FWT["precision_ratio"] = FWT.precision / FWT.test_precision
    fm = FWT[FWT.model == MAIN]
    R.update(fwd_tab=FWT, fwd_prec=float(fm.precision.mean()), fwd_test_prec=float(fm.test_precision.mean()),
             fwd_prec_ratio=float(fm.precision.mean() / fm.test_precision.mean()) if fm.test_precision.mean() > 0 else np.nan,
             fwd_auc=float(fm.auc.mean()), fwd_test_auc=float(fm.test_auc.mean()), fwd_eval_months=fm.month.tolist(),
             fwd_pending=pd.DataFrame(pend), obs_end=ym_of(OBS_END))
    B.table(FWT, f"8h · 2026 months against the test months at the same follow-up ({K_MAIN:,} calls, α = {A_MAIN}; "
                 "test_* = mean over the test months)",
            pct=("rate", "test_rate", "precision", "test_precision", "recall_dollars", "test_recall_dollars"), save="m5_8h_forward")
    B.kv([("exits observed through", f"{ym_of(OBS_END)} (later exits not yet confirmed)"),
          (f"precision, 2026 months / test months at the same follow-up ({MAIN})",
           f"{R['fwd_prec']:.1%} / {R['fwd_test_prec']:.1%} ({R['fwd_prec_ratio']:.2f}×)"),
          (f"AUC, 2026 months / test months at the same follow-up ({MAIN})", f"{R['fwd_auc']:.3f} / {R['fwd_test_auc']:.3f}"),
          ("months scored but not yet evaluable", ", ".join(p_["month"] for p_ in pend) or "—")], "8i · Forward check")
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
    for mn, _ in MODELS:
        d_ = FWT[FWT.model == mn]; col_ = {"simple": "grey", "tree": "black", "tree_base": "#c55a11"}[mn]
        ax[0].plot(d_.month, d_.precision, marker="o", color=col_, label=f"{mn}, 2026 month")
        ax[0].plot(d_.month, d_.test_precision, ls=":", color=col_, label=f"{mn}, test months same follow-up")
        ax[1].plot(d_.month, d_.auc, marker="o", color=col_, label=f"{mn}, 2026 month")
        ax[1].plot(d_.month, d_.test_auc, ls=":", color=col_, label=f"{mn}, test months same follow-up")
    ax[0].set_title(f"Precision at {K_MAIN:,} calls (follow-up shortens to the right)"); ax[1].set_title("AUC")
    ax[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
    for a_ in ax: a_.legend(fontsize=7); a_.tick_params(axis="x", rotation=90)
    savefig(fig, "m5_8b_forward"); figdata("m5_8b_forward", FWT)
else:
    R.update(fwd_tab=None, fwd_prec_ratio=np.nan, fwd_pending=pd.DataFrame(pend), obs_end=ym_of(OBS_END))
    B.check("forward check", "INFO", "no 2026 month has an observed follow-up yet (or FORWARD = False)")
if pend: B.table(pd.DataFrame(pend), "8j · Scored, not yet evaluable (the current list comes from the last of these)")
B.show(t0)

## §9 When signals fire

**Event time** (describes behaviour; uses the exit date): money-still-here months of leavers aligned on the month
the money left, against stayers aligned on a pseudo-exit month drawn by a fixed hash. Only **change** features are
traced, so a client's size cannot pass for a signal, and every comparison is made **within the same size band**:
thresholds (stayers' 10th/90th percentile) are set per band and the band lifts are averaged with the leavers' band
mix. Onset = the earliest month from which the lift stays ≥ 1.5 through month −1. Drains already in progress are
listed separately as **late warnings**. **Model view:** each feature's contribution to leavers' scores minus
stayers', month by month; the total gap shows whether the score rises toward the exit. **Queue time:** how many
months before the money left a client first reached the list.

In [ ]:
# =====================================================================
# 9 · WHEN SIGNALS FIRE                                              [OUTPUT BLOCK 9]
# =====================================================================
t0 = time.time(); B = Block("§9 · When signals fire")
CHG_FORMS = ("change vs 4–6 months ago", "change vs last 3 months")
CHG = [f for f in IMP.feature if form_of(f) in CHG_FORMS or f in ("dep_bal_vs_norm", "dep_drawdown")]
TIMING = CHG[:N_TIMING] + [f for f in LATE_FEATS if f in FEATS]
span_pe = (LAST_LABEL_T + H) - 12 + 1
ev_base = (MF.withColumn("e", F.when(F.col("ab_t").isNotNull(), F.col("ab_t"))
                              .when(F.col("last_live_t") == END_T, F.lit(12) + F.expr(f"pmod(hash(cust_pwr_id), {span_pe})")))
             .withColumn("grp", F.when(F.col("ab_t").isNotNull(), F.lit("leaver")).otherwise(F.lit("stayer")))
             .filter(F.col("e").isNotNull() & ((F.col("grp") == "stayer") | (F.col("e") <= LAST_LABEL_T + H)))
             .withColumn("rel", F.col("t") - F.col("e")).filter(F.col("rel").between(-EVENT_PRE, -1)))
_cnt = {r["grp"]: r["n"] for r in ev_base.groupBy("grp").agg(F.countDistinct("cust_pwr_id").alias("n")).collect()}
def sub(n_per_group, salt):
    fr = {g: int(1e6 * min(1.0, n_per_group / max(_cnt.get(g, 1), 1))) for g in ("leaver", "stayer")}
    return ev_base.filter(F.expr(f"pmod(hash(cust_pwr_id, {salt}), 1000000)") <
                          F.when(F.col("grp") == "leaver", F.lit(fr["leaver"])).otherwise(F.lit(fr["stayer"])))
bandx = F.when(F.col("bal_norm") < 1e5, 0).when(F.col("bal_norm") < 1e6, 1).when(F.col("bal_norm") < 1e7, 2).otherwise(3)
EV = topd(sub(MAX_EVENT_CUST, SEED + 1).select("cust_pwr_id", "grp", "rel", bandx.alias("band"),
                                               *[F.col(f).cast("double").alias(f) for f in dict.fromkeys(TIMING)]))
lv_rows, st_rows = EV[EV.grp == "leaver"], EV[EV.grp == "stayer"]
rels = list(range(-EVENT_PRE, 0))
wts = lv_rows.groupby(["rel", "band"]).size().unstack(fill_value=0).reindex(rels, fill_value=0)
tim, LIFT = [], {}
for f in dict.fromkeys(TIMING):
    Lb = {"falls": [], "rises": []}
    for bnd in sorted(EV.band.unique()):
        s_ = st_rows[st_rows.band == bnd][f].dropna()
        if len(s_) < 50: continue
        q10, q90 = s_.quantile(0.10), s_.quantile(0.90)
        e_ = EV[EV.band == bnd]; x = e_[f]
        for side, fire in (("falls", (x < q10) & x.notna()), ("rises", (x > q90) & x.notna())):
            r_ = fire.groupby([e_.grp, e_.rel]).mean()
            if "leaver" in r_.index.get_level_values(0) and "stayer" in r_.index.get_level_values(0):
                lift = (r_["leaver"] / r_["stayer"].replace(0, np.nan)).reindex(rels)
                w_ = (wts[bnd] if bnd in wts.columns else pd.Series(0, index=rels)).astype(float)
                Lb[side].append((lift.fillna(0) * w_ * lift.notna(), w_ * lift.notna()))
    comb = {k: (sum(n for n, _ in v) / sum(d for _, d in v).replace(0, np.nan)) if v else pd.Series(np.nan, index=rels)
            for k, v in Lb.items()}
    side = "falls" if np.nanmean(comb["falls"].loc[-3:-1]) >= np.nanmean(comb["rises"].loc[-3:-1]) else "rises"
    Lf = comb[side]; on = None
    for r_ in range(-1, -EVENT_PRE - 1, -1):
        if pd.notna(Lf.get(r_)) and Lf[r_] >= LIFT_BAR: on = r_
        else: break
    LIFT[f] = Lf
    tim.append({"feature": f, "label": LABEL[f], "set": FAM[f], "kind": "late warning" if f in LATE_FEATS else "early signal",
                "leavers": side, "onset_month": on, **{f"lift_{-k}m": Lf.get(k) for k in (-9, -6, -3, -1)},
                "coverage": float(lv_rows[f].notna().mean())})
TIM = pd.DataFrame(tim).sort_values(["kind", "onset_month", "lift_1m"], ascending=[True, True, False], na_position="last").reset_index(drop=True)
R.update(tim=TIM, n_pay_onset6=int(((TIM.onset_month <= -6) & TIM.set.str.startswith("payment")).sum()),
         n_ev_leavers=int(lv_rows.cust_pwr_id.nunique()), n_ev_stayers=int(st_rows.cust_pwr_id.nunique()))
B.table(TIM, "9a · Event time, size-matched: when each change signal fires (lift = leavers ÷ stayers of the same size band)",
        pct=("coverage",), save="m5_9a_onset")
fig, ax = plt.subplots(figsize=(12, 0.32 * len(TIM) + 1.5))
M_ = np.array([LIFT[f].values for f in TIM.feature], dtype=float)
im = ax.imshow(np.clip(M_, 0, 5), aspect="auto", cmap="Reds", vmin=1, vmax=5)
ax.set_yticks(range(len(TIM))); ax.set_yticklabels([f"{l[:62]} ({d})" for l, d in zip(TIM.label, TIM.leavers)], fontsize=8)
ax.set_xticks(range(len(rels))); ax.set_xticklabels(rels); ax.set_xlabel("months before the money left")
ax.set_title("Lift over size-matched stayers (1 = no difference, capped at 5)"); fig.colorbar(im, ax=ax, shrink=0.6)
savefig(fig, "m5_9a_onset_heatmap")
figdata("m5_9a_lift_by_month", pd.DataFrame([{"feature": f, "label": LABEL[f], "rel": r_, "lift": LIFT[f].get(r_)}
                                              for f in TIM.feature for r_ in rels]))

# ── model view: contributions, leavers minus stayers, by month ───────
EVS = topd(sub(N_SHAP_CUST, SEED + 2).select("grp", "rel", *[F.col(f).cast("float").alias(f) for f in FEATS]))
Xe = EVS[FEATS].to_numpy(np.float32)
Ce = gbm_predict(MODEL["G"], Xe, contrib=True) if MODEL["G"] else None
if Ce is None: Ce = (transform(Xe, MODEL["P"]) * MODEL["b"][1:].astype(np.float32))[:, :len(FEATS)]
CEdf = pd.DataFrame(Ce, columns=FEATS).assign(grp=EVS.grp.values, rel=EVS.rel.values)
mg = CEdf.groupby(["grp", "rel"]).mean()
gap = (mg.loc["leaver"] - mg.loc["stayer"]).reindex(rels)
TOTAL = gap.sum(axis=1)
top15 = list(IMP.feature.head(15))
MVT = gap[top15].T; MVT.columns = [str(c) for c in rels]
MVT.insert(0, "feature", [LABEL[f][:70] for f in top15]); MVT = MVT.reset_index(drop=True)
MVT.loc[len(MVT)] = ["ALL FEATURES (total log-odds gap)"] + list(TOTAL.values)
g6 = TOTAL.get(-6, np.nan); g1 = TOTAL.get(-1, np.nan)
R.update(model_view=MVT, gap6=float(g6), gap1=float(g1), gap_ratio=float(g6 / g1) if g1 else np.nan,
         lead6=[LABEL[f][:60] for f in gap.loc[-6].sort_values(ascending=False).head(5).index],
         lead1=[LABEL[f][:60] for f in gap.loc[-1].sort_values(ascending=False).head(5).index])
B.table(MVT, "9b · Model view: contribution to leavers' score minus stayers' (log-odds) by month before the money left",
        save="m5_9b_model_view")
B.kv([("total gap 6 months out ÷ 1 month out", f"{R['gap_ratio']:.2f} (model 1 was ≈ 0.8: flat)"),
      ("largest drivers 6 months out", " · ".join(R["lead6"])), ("largest drivers 1 month out", " · ".join(R["lead1"]))],
     "9b · Early vs late drivers")
fig, ax = plt.subplots(figsize=(8, 3.6)); ax.plot(rels, TOTAL.values, marker="o", color="#c00000")
ax.set_title("Leavers' score minus stayers' (log-odds), all features"); ax.set_xlabel("months before the money left")
savefig(fig, "m5_9b_total_gap")
figdata("m5_9b_total_gap", pd.DataFrame({"rel": rels, "gap": TOTAL.values}))
figdata("m5_9b_model_view", MVT)

# ── queue time ───────────────────────────────────────────────────────
fq = fa0.copy(); fq["lead"] = (fq.ab_t - fq["tm"]).astype(int)
fq["reason_set"] = fq.r1.map(lambda r: FAMOF.get(r, "?")); fq["reason_kind"] = fq.r1.map(lambda r: "late warning" if r in LATE_FEATS else "early signal")
LEAD = fq.groupby("lead").agg(clients=("cust_pwr_id", "size"), capped=("capped", "sum")).reset_index()
LEAD["share_clients"] = LEAD.clients / LEAD.clients.sum(); LEAD["share_dollars"] = LEAD.capped / LEAD.capped.sum()
RS = fq.groupby(["reason_kind", "reason_set"]).agg(clients=("cust_pwr_id", "size"), capped=("capped", "sum")).reset_index() \
       .sort_values("clients", ascending=False)
RS["share"] = RS.clients / RS.clients.sum()
# sustained alert: first of ≥ 2 consecutive months on the list, before the money left
rkm = f"rank_{MAIN}_{A_MAIN}"
onl = Q[(Q[rkm] <= K_MAIN) & Q.ab_t.notna() & (Q["tm"] < Q.ab_t) & Q.cust_pwr_id.isin(set(fq.cust_pwr_id))][["cust_pwr_id", "tm", "ab_t", "capped"]]
_on = set(zip(onl.cust_pwr_id, onl["tm"]))
onl = onl[[(c_, t_ + 1) in _on for c_, t_ in zip(onl.cust_pwr_id, onl["tm"])]]
SUS = onl.sort_values("tm").drop_duplicates("cust_pwr_id")
SUS = SUS.assign(lead=(SUS.ab_t - SUS["tm"]).astype(int))
SUS["lead_band"] = SUS.lead.map(lambda l: "7+" if l >= 7 else str(l))
LEAD_S = SUS.groupby("lead_band").agg(clients=("cust_pwr_id", "size"), capped=("capped", "sum")).reset_index()
_single = fq[~fq.cust_pwr_id.isin(set(SUS.cust_pwr_id))]
LEAD_S.loc[len(LEAD_S)] = ["one-month blips only", len(_single), float(_single.capped.sum())]
LEAD_S["share_clients"] = LEAD_S.clients / max(len(fq), 1); LEAD_S["share_dollars"] = LEAD_S.capped / max(fq.capped.sum(), 1)
R.update(lead_sus=LEAD_S, lead3_sus=float((SUS.lead >= 3).sum() / max(len(fq), 1)),
         lead3_sus_usd=float(SUS.loc[SUS.lead >= 3, "capped"].sum() / max(fq.capped.sum(), 1)),
         single_only=float(len(_single) / max(len(fq), 1)), sus_ids=SUS)
B.table(LEAD_S, "9c · Sustained alerts (first of ≥ 2 consecutive months on the list): months before the money left",
        pct=("share_clients", "share_dollars"), money=("capped",), save="m5_9c_lead_sustained")
figdata("m5_9c_lead_sustained", LEAD_S)
R.update(lead=LEAD, reason_sets=RS, lead3=float((fq.lead >= 3).mean()) if len(fq) else np.nan,
         lead3_usd=float(fq.loc[fq.lead >= 3, "capped"].sum() / max(fq.capped.sum(), 1)),
         late_share=float((fq.reason_kind == "late warning").mean()) if len(fq) else np.nan)
B.table(LEAD, f"9c · Queue time: months from the first alert to the money leaving ({K_MAIN:,} calls, α = {A_MAIN})",
        pct=("share_clients", "share_dollars"), money=("capped",), save="m5_9c_lead")
B.table(RS, "9c · Top reason at the first alert", pct=("share",), money=("capped",))

# ── leavers vs stayers on change features ────────────────────────────
BEHAV = [f for f in ["dep_bal_vs_norm", "dep_d6", "ntxn_all_out__d6", "ntxn_all_in__d6", "amt_all_out__d6", "amt_all_in__d6",
                     "n_fi_ach_out__d6", "n_fi_out__d6", "ncpty_all_out__d6", "ncpty_all_in__d6", "active_days_out__d6",
                     "active_days_in__d6"] if f in MF.columns][:12]
EB = topd(sub(MAX_EVENT_CUST, SEED + 1).select("grp", "rel", *[F.col(f).cast("double").alias(f) for f in BEHAV]))
BV = EB.groupby(["grp", "rel"])[BEHAV].mean()
fig, axs = plt.subplots(int(np.ceil(len(BEHAV) / 4)), 4, figsize=(16, 3.1 * int(np.ceil(len(BEHAV) / 4))))
for a_, f in zip(np.ravel(axs), BEHAV):
    for g_, col in (("leaver", "#c00000"), ("stayer", "#1f4e79")):
        a_.plot(rels, BV.loc[g_, f].reindex(rels), marker="o", ms=3, color=col, label=g_)
    a_.axhline(0, color="grey", lw=0.6); a_.set_title(label(f)[:50], fontsize=9); a_.tick_params(labelsize=8)
for a_ in np.ravel(axs)[len(BEHAV):]: a_.axis("off")
np.ravel(axs)[0].legend(fontsize=8); fig.suptitle("Mean change (log ratio) by month before the money left — money-still-here months")
savefig(fig, "m5_9d_leavers_vs_stayers")
LS = pd.DataFrame([{"feature": label(f), "month": k, "leavers": BV.loc["leaver", f].get(k), "stayers": BV.loc["stayer", f].get(k)}
                   for f in BEHAV for k in (-12, -6, -3, -1)]).pivot(index="feature", columns="month", values=["leavers", "stayers"])
LS.columns = [f"{a} {b}" for a, b in LS.columns]; LS = LS.reset_index()
R.update(ls=LS)
figdata("m5_9d_leavers_vs_stayers_by_month", BV.reset_index())
figdata("m5_9c_lead", LEAD); figdata("m5_9c_reasons", RS)
B.table(LS, "9d · Leavers vs stayers: mean change 12, 6, 3 and 1 months before", save="m5_9d_leavers_vs_stayers")
B.kv([("leavers / stayers traced", f"{R['n_ev_leavers']:,} / {R['n_ev_stayers']:,}"),
      ("payment change features firing 6+ months out (size-matched)", R["n_pay_onset6"]),
      ("first alerted 3+ months before the money left", f"{R['lead3']:.1%} of clients, {R['lead3_usd']:.1%} of dollars"),
      ("sustained alert 3+ months before the money left", f"{R['lead3_sus']:.1%} of reached clients, {R['lead3_sus_usd']:.1%} of dollars"),
      ("reached only by one-month blips", f"{R['single_only']:.1%} of reached clients"),
      ("first alerts whose top reason is a drain already in progress", f"{R['late_share']:.1%}")], "9e · Summary")
B.show(t0)

## §10 Example clients for RM managers

Chosen **by rule** from the test months (1,000 calls, α = 0.5, primary model). The alert date is the **sustained
alert**: the first of ≥ 2 consecutive months on the list (earlier one-month blips are mentioned, not counted).
Departing examples must pass the §7 check (verdict *departed* or *moved to another bank*) and have **12 steady
months** before the alert (balance ≥ 10% of normal every month); a reached client whose money arrived recently gets
its own example, *new money that left*.

For each client: the **balance in dollars** against its normal balance, the months on the list, the model's reasons,
and a panel for each **change signal that fired** (§9a) with the stayers' threshold; ▼/▲ in the table marks a month
beyond it. The data behind every example is in `figdata/m5_10_examples_long.csv`. The report shows power ids when
`SHOW_PWR_IDS` is on (one-way hashes otherwise); full ids are always in `m5_10_examples_FULL_IDS.csv`.

In [ ]:
# =====================================================================
# 10 · EXAMPLE CLIENTS FOR RM MANAGERS                             [OUTPUT BLOCK 10]
# =====================================================================
t0 = time.time(); B = Block("§10 · Example clients")
def play(feat):
    f = feat.replace("miss·", "")
    if f.startswith("pr_"):
        return ("Money is moving through payment processors, brokerages or other banks. Ask how the client now pays and "
                "collects and where it invests cash; offer PNC payables, receivables and liquidity products before the "
                "processor or another bank's platform replaces them.")
    if f.startswith("nx"):
        return ("Payments are shifting toward a different kind of counterparty (industry mix). Ask what changed in the "
                "business — new suppliers, investments, an acquisition — and the cash plans behind it.")
    if f.startswith(("nb_", "cp_", "cf_")):
        return ("The client's trading partners are changing: PNC counterparties leaving or partners moving banks. Check "
                "whether the client's ecosystem is moving and offer to keep its payments with those partners at PNC.")
    if re.search(r"(n_fi|^fi_|_fi_|selfpay|top_fi|^share_|n_rails)", f):
        return ("Competitive threat: money appears to be moving to another bank. Compare pricing, earnings credit "
                "and services with the likely competitor; offer a retention package.")
    if f.startswith("dep_"):
        return ("Balance drawdown: deposits are falling ahead of the payments. Ask about cash plans; offer sweep, "
                "money-market or liquidity options to keep the operating balance here.")
    return ("Business activity is shrinking: fewer payments received and fewer trading partners. Check the client's "
            "business; offer credit, working-capital or receivables support.")

OKV = {"departed", "moved to another bank"}
rk = f"rank_{MAIN}_{A_MAIN}"
Qx = Q[["cust_pwr_id", "tm", rk, "p", "bal", "bal_norm", "capped", "ab_t", "exit_type", "y", "r1", "r2", "r3"]]
onl = Qx[(Qx[rk] <= K_MAIN) & ((Qx.ab_t.isna()) | (Qx["tm"] < Qx.ab_t))]
_on = set(zip(onl.cust_pwr_id, onl["tm"]))
sus = onl[[(c_, t_ + 1) in _on for c_, t_ in zip(onl.cust_pwr_id, onl["tm"])]].sort_values("tm").drop_duplicates("cust_pwr_id")
lst = sus[["cust_pwr_id", "tm", "bal", "bal_norm", "r1"]].rename(columns={"tm": "T_alert", "bal": "bal_at_alert", "bal_norm": "norm_at_alert"})
dp = pool[["cust_pwr_id", "ab_t", "exit_type", "capped", "bal_norm"]].merge(lst, on="cust_pwr_id", how="left")
dp["lead"] = dp.ab_t - dp.T_alert; dp["pct_at_alert"] = dp.bal_at_alert / dp.norm_at_alert
hit = dp[dp.T_alert.notna()]

HIST = {}
def steady(ids_):
    """True if the client's balance stayed ≥ 10% of normal in each of the STEADY_M months before its alert."""
    new = [c for c in ids_ if c not in HIST]
    if new:
        h = topd(FEAT.filter(F.col("cust_pwr_id").isin(new)).select("cust_pwr_id", "t", "bal"))
        for c in new: HIST[c] = h[h.cust_pwr_id == c].set_index("t").bal
    out = {}
    for c in ids_:
        ta = dp.loc[dp.cust_pwr_id == c, "T_alert"]; nm = dp.loc[dp.cust_pwr_id == c, "norm_at_alert"]
        if not len(ta) or pd.isna(ta.iloc[0]): out[c] = True; continue
        w = HIST[c].loc[int(ta.iloc[0]) - STEADY_M:int(ta.iloc[0]) - 1]
        out[c] = len(w) >= STEADY_M and bool((w >= 0.10 * float(nm.iloc[0])).all())
    return out
chosen, EXS = set(), []
def pick(df, kind, sort="bal_norm", need_steady=True, want_steady=True):
    d = df[~df.cust_pwr_id.isin(chosen)].sort_values(sort, ascending=False).head(25)
    if not len(d): return
    v = audit(list(d.cust_pwr_id)).set_index("cust_pwr_id").verdict
    ok = [c for c in d.cust_pwr_id if v.get(c) in OKV]
    if need_steady and ok:
        st = steady(ok); ok = [c for c in ok if st[c] == want_steady]
    if ok: chosen.add(ok[0]); EXS.append((kind, ok[0]))
pick(hit[hit.lead >= 4], "Early catch")
pick(hit[(hit.pct_at_alert >= 0.9) & hit.r1.map(lambda r: isinstance(r, str) and not r.startswith("dep_"))], "Payments moved before the balance")
pick(hit[hit.exit_type == "balance exit"], "Drained without closing")
pick(hit[hit.lead <= 2], "Late catch")
pick(hit, "New money that left", want_steady=False)
never = set(SC.loc[SC.ab_t.isna(), "cust_pwr_id"]) - set(SC.loc[SC.y == 1, "cust_pwr_id"])
fal = sus[sus.cust_pwr_id.isin(never)].merge(Qx.groupby("cust_pwr_id").p.max().rename("pmax"), on="cust_pwr_id").sort_values("pmax", ascending=False)
if len(fal): chosen.add(fal.cust_pwr_id.iloc[0]); EXS.append(("False alarm (most confident wrong call)", fal.cust_pwr_id.iloc[0]))
pick(dp[dp.T_alert.isna() & ~dp.cust_pwr_id.isin(set(onl.cust_pwr_id))], "Missed", sort="capped")
ids = [c for _, c in EXS]

# ── change signals with a stayers' threshold (from §9a) ─────────────
tim_on = TIM.dropna(subset=["onset_month"])
THR = {f: (st_rows[f].quantile(0.10) if d_ == "falls" else st_rows[f].quantile(0.90), d_)
       for f, d_ in zip(tim_on.feature, tim_on.leavers) if f in FEAT.columns}
PAN = topd(FEAT.filter(F.col("cust_pwr_id").isin(ids)).select("cust_pwr_id", "t", "bal", "bal_norm", "live", "ab_t",
                                                              *[F.col(f).cast("double").alias(f) for f in THR]))
fmtv = lambda v: "—" if pd.isna(v) else f"{v:+.2f}"
def beyond(v, f):
    th, d_ = THR[f]; return pd.notna(v) and ((v < th) if d_ == "falls" else (v > th))
EXOUT, full_ids, LONG = [], [], []
for n, (kind, c) in enumerate(EXS, 1):
    pan = PAN[PAN.cust_pwr_id == c].sort_values("t"); sc = Qx[Qx.cust_pwr_id == c].sort_values("tm")
    e = int(pan.ab_t.dropna().iloc[0]) if pan.ab_t.notna().any() else None
    s_row = sus[sus.cust_pwr_id == c]
    anchor = s_row.iloc[0] if len(s_row) else (sc.sort_values(rk).iloc[0] if len(sc) else None)
    ta = int(anchor["tm"]) if anchor is not None else None
    lo_t = int(min(sc["tm"].min() if len(sc) else pan.t.max(), (e - 12) if e is not None else 10 ** 6, (ta - 6) if ta is not None else 10 ** 6))
    hi_t = int(e + 1) if e is not None else int(sc["tm"].max() + 1 if len(sc) else pan.t.max())
    pan = pan[(pan.t >= max(lo_t, 0)) & (pan.t <= hi_t)].set_index("t")
    ref = float(anchor.bal_norm) if anchor is not None else float(pan.bal_norm.dropna().median())
    alerts = sc[sc[rk] <= K_MAIN]
    blips = [ym_of(t_) for t_ in alerts["tm"] if ta is not None and t_ < ta]
    fired = []
    for f in THR:
        v = pan[f]; m_ = v[[beyond(x, f) for x in v]].index
        m_ = m_[m_ < e] if e is not None else m_
        if len(m_): fired.append((int(m_.min()), f))
    fired.sort()
    reasons = [r for r in (anchor[["r1", "r2", "r3"]].tolist() if anchor is not None and "r1" in anchor else []) if isinstance(r, str)]
    if anchor is not None and not reasons:
        rr = sc[sc["tm"] == ta]; reasons = [r for r in rr[["r1", "r2", "r3"]].iloc[0].tolist() if isinstance(r, str)] if len(rr) else []
    trig = list(dict.fromkeys([f for f in reasons if f in THR] + [f for _, f in fired]))[:N_TRIG]
    rows = []
    for t_, r in pan.iterrows():
        s_ = sc[sc["tm"] == t_]
        row = {"month": ym_of(t_), "vs_exit": (int(t_ - e) if e is not None else None), "balance": r.bal, "normal": ref,
               "balance_pct_normal": r.bal / ref if ref else np.nan, "score": float(s_.p.iloc[0]) if len(s_) else np.nan,
               "rank": f"{int(s_[rk].iloc[0]):,}" if len(s_) else "—", "on_list": bool(len(s_) and s_[rk].iloc[0] <= K_MAIN)}
        for f in trig:
            v = r.get(f, np.nan); h_ = beyond(v, f)
            row[LABEL[f][:45]] = fmtv(v) + ((" ▼" if THR[f][1] == "falls" else " ▲") if h_ else "")
            LONG.append({"example": n, "kind": kind, "client": cid(c), "month": ym_of(t_), "vs_exit": row["vs_exit"], "variable": f,
                         "label": LABEL[f], "value": v, "threshold": THR[f][0], "direction": THR[f][1], "fired": bool(h_),
                         "reason_at_alert": f in reasons})
        for var, val in (("balance", r.bal), ("normal", ref), ("score", row["score"])):
            LONG.append({"example": n, "kind": kind, "client": cid(c), "month": ym_of(t_), "vs_exit": row["vs_exit"], "variable": var,
                         "label": var, "value": val, "threshold": np.nan, "direction": None,
                         "fired": row["on_list"] if var == "score" else None, "reason_at_alert": False})
        row["reasons"] = " · ".join(LABEL.get(x, x)[:50] for x in s_[["r1", "r2", "r3"]].iloc[0]) if len(s_) else ""
        rows.append(row)
    TAB = pd.DataFrame(rows)
    trig_at = "; ".join(f"{LABEL[f][:55]} = {fmtv(pan[f].get(ta))} (stayers' threshold {THR[f][0]:+.2f})" for f in trig) if ta is not None else ""
    fired_txt = "; ".join(f"{LABEL[f][:55]} (from {m - e:+d})" for m, f in fired[:5]) if e is not None else \
                "; ".join(f"{LABEL[f][:55]} (from {ym_of(m)})" for m, f in fired[:5])
    rsn = " · ".join(LABEL.get(x, x)[:55] for x in reasons)
    ex_type = dp.loc[dp.cust_pwr_id == c, "exit_type"].iloc[0] if (dp.cust_pwr_id == c).any() else None
    blip_txt = f" (earlier one-month appearances: {', '.join(blips)})" if blips else ""
    if kind.startswith("False"):
        text = (f"On the list from {ym_of(ta)} for {len(alerts)} month(s), score up to {sc.p.max():.1%}, and stayed. Reasons: {rsn}. "
                + (f"Change signals beyond the stayers' threshold: {fired_txt}. " if fired_txt else "")
                + f"At {K_MAIN:,} calls about {1 - float(ACC[(ACC.model == MAIN) & (ACC.alpha == A_MAIN) & (ACC.calls == K_MAIN)].precision_mean.iloc[0]):.0%} "
                "of calls look like this one.")
    elif kind == "Missed":
        text = (f"The money left in {ym_of(e)} ({ex_type}); the client never reached the top {K_MAIN:,}"
                + (f" (best rank {int(anchor[rk]):,} in {ym_of(ta)}; reasons then: {rsn})." if anchor is not None else ".")
                + (f" Change signals that fired: {fired_txt}." if fired_txt else " No change signal crossed the stayers' threshold."))
    else:
        text = (f"Sustained alert from {ym_of(ta)}, {int(e - ta)} month(s) before the money left ({ex_type}, {ym_of(e)}){blip_txt}. "
                f"Balance then ${anchor.bal:,.0f} against a ${anchor.bal_norm:,.0f} normal ({anchor.bal / anchor.bal_norm:.0%}). "
                f"Model's reasons: {rsn}. " + (f"Change signals at the alert: {trig_at}. " if trig_at else "")
                + (f"Signals that fired, in order: {fired_txt}. " if fired_txt else "")
                + (f"How an RM could have intervened: {play(reasons[0])}" if reasons else ""))
        if kind == "New money that left":
            text = "Balance arrived within the 12 months before the alert. " + text
    xs = [ym_of(t_) for t_ in pan.index]
    fig, axs = plt.subplots(1 + len(trig), 1, figsize=(11, 2.6 + 1.7 * len(trig)), sharex=True, squeeze=False)
    axs = axs[:, 0]
    axs[0].plot(xs, pan.bal / 1e6, marker="o", color="#1f4e79", label="balance ($m)")
    axs[0].axhline(ref / 1e6, color="grey", ls="--", lw=1, label="normal balance")
    for t_ in alerts["tm"]:
        if t_ in pan.index:
            for ax_ in axs: ax_.axvline(ym_of(t_), color="#c55a11", alpha=0.25, lw=9)
    if e is not None and e in pan.index:
        for ax_ in axs: ax_.axvline(ym_of(e), color="#c00000", ls="--", lw=1.2)
    axs[0].set_title(f"Example {n} · {kind} · client {disp(c)}  (orange = on the list, red dashed = money left)")
    axs[0].legend(loc="lower left", fontsize=8); axs[0].set_ylabel("$m")
    for ax_, f in zip(axs[1:], trig):
        v = pan[f]; th, d_ = THR[f]
        ax_.plot(xs, v.values, marker="o", ms=3, color="#333"); ax_.axhline(th, color="#c00000", lw=0.8, ls=":")
        hm = v[[beyond(x, f) for x in v]]
        ax_.plot([ym_of(t_) for t_ in hm.index], hm.values, "o", color="#c00000", ms=6)
        ax_.set_title(LABEL[f][:80] + f" — leavers {d_}; red dots = beyond the stayers' threshold", fontsize=9)
    axs[-1].tick_params(axis="x", rotation=90)
    savefig(fig, f"m5_10_example_{n}")
    EXOUT.append(dict(n=n, kind=kind, client=cid(c), shown_id=disp(c), normal=ref, text=text, table=TAB, chart=f"m5_10_example_{n}"))
    full_ids.append({"example": n, "kind": kind, "display_id": cid(c), "cust_pwr_id": c})
    B.note(f"<b>Example {n} · {kind} · client {disp(c)}</b> — {_html.escape(text)}")
    B.table(TAB, f"Example {n}: month by month (▼/▲ = beyond the stayers' threshold)", pct=("balance_pct_normal", "score"),
            money=("balance", "normal"))
pd.DataFrame(full_ids).to_csv(OUT / "m5_10_examples_FULL_IDS.csv", index=False)
figdata("m5_10_examples_long", pd.DataFrame(LONG))
figdata("m5_10_thresholds", pd.DataFrame([{"feature": f, "label": LABEL[f], "threshold": th, "leavers": d_} for f, (th, d_) in THR.items()]))
R.update(examples=EXOUT)
if len(EXS) < 7: B.check("examples", "INFO", f"only {len(EXS)} of 7 rules found a client")
B.show(t0)

## §10b Where the money goes; the current list

**Destination of a departure:** rises in the exit month and the three before, against the client's own months −9…−4, of
at least 10% of the normal balance, checked in this order: payments to the client's **own name** at another bank (from
`pkg_cust_fi_month`; that bank is the destination); outflows to **money-movement processors or brokerages** (§3b; the money
left through an intermediary and the bank behind it is not visible — a rise in payments at a bank is then usually the
processor's own bank); payments to **another bank's customers** (that bank).

**Naming the bank in advance:** for each departure, at one and three months before the exit, the candidates are the banks the
client paid in the last 12 months plus the 15 largest competitor banks. A small ranking model scores each candidate from
data up to that month only (dollar shares and their change, own-name dollars, counterparties, new and standing links, the
bank's overall growth, and the earlier departures' destinations). It is trained on departures up to the first test month
and checked on later ones against three baselines: the client's current top bank, its top own-name bank, and the most
common destination.

**The current list:** the top 1,000 of the last scored month, with reasons, the suggested play and the likely destination
banks (refitted on every labelled departure). Full ids are in `m5_10b_current_list_FULL_IDS.csv`.

In [ ]:
# =====================================================================
# 10b · WHERE THE MONEY GOES: DESTINATION BANK · THE CURRENT LIST   [OUTPUT BLOCK 10b]
# =====================================================================
t0 = time.time(); B = Block("§10b · Where the money goes; the current list")
import traceback
def pk_py(x):
    """Python twin of pk_expr: the metrics-side power id of a deposit cust_pwr_id."""
    if x is None or (isinstance(x, float) and np.isnan(x)): return None
    s_ = str(x).strip()
    if not s_: return None
    return re.sub(r"^0+(?=.)", "", s_) if KEY == "strip0" else s_
TRAIN_E = t_of(TEST_FIRST)                                  # departures up to the first test month train; later ones test
OBS_END = END_T - max(CONFIRM_M, BAL_EXIT_HOLD) + 1
DL = ex[~ex.cust_pwr_id.isin(EXCL) & (ex.ab_t <= OBS_END)][["cust_pwr_id", "ab_t", "exit_type", "norm"]].copy()
DL["e"] = DL.ab_t.astype(int); DL["pwr"] = DL.cust_pwr_id.map(pk_py)
DL["period"] = np.where(DL.e <= TRAIN_E, "before the test period", "test period and after")

# ── the current list: last scored month, ranked as in the queue ──────
src_ = FW if len(FW) else SC
CUR_T = int(src_.tm.max())
cur = src_[src_.tm == CUR_T].copy(); cur["norm"] = cur.bal_norm.clip(lower=1.0)
cur["score"] = cur.p * cur.norm ** A_MAIN
cur = cur.sort_values(["score", "cust_pwr_id"], ascending=[False, True]).head(K_MAIN).reset_index(drop=True)
cur["rank"] = np.arange(1, len(cur) + 1); cur["pwr"] = cur.cust_pwr_id.map(pk_py)
LST = cur[["rank", "cust_pwr_id", "p", "bal", "bal_norm"]].rename(columns={"p": "probability_6m", "bal": "balance", "bal_norm": "normal"})
short = lambda s_, n: s_ if len(s_) <= n else s_[:n - 1] + "…"
LST["reasons"] = [" · ".join(LABEL.get(x, x) for x in r_ if isinstance(x, str)) for r_ in cur[["r1", "r2", "r3"]].values]
LST["play"] = [play(r_) if isinstance(r_, str) else "" for r_ in cur.r1]

DEST_ERR, DEST_ACC, DEST_SRC, EVID, DTOP, VIA = None, None, None, None, None, None
try:
    if not table_exists(FI_TABLE): raise RuntimeError(f"{FI_TABLE} is not readable from Spark")
    fic = spark.table(FI_TABLE).columns
    miss = [c for c in ("customer_pwr_id", "dir", "fi", "amount", "month") if c not in fic]
    if miss: raise RuntimeError(f"{FI_TABLE} lacks {miss}")
    fl = lambda c: F.coalesce(F.col(c).cast("int"), F.lit(0)) if c in fic else F.lit(0)
    FIB = (spark.table(FI_TABLE).filter((F.col("dir") == "out") & F.col("month").isin(WANT))
             .select(pk_expr("customer_pwr_id", KEY).alias("pwr"), t_col("month").alias("t"),
                     F.upper(F.trim(F.col("fi").cast("string"))).alias("fi"),
                     (F.coalesce(F.col("is_pnc").cast("boolean"), F.lit(False)) if "is_pnc" in fic else F.lit(False)).alias("is_pnc"),
                     F.col("amount").cast("double").alias("amt"),
                     (F.col("selfpay_amt").cast("double") if "selfpay_amt" in fic else F.lit(0.0)).alias("selfpay"),
                     (F.col("n_cpty").cast("double") if "n_cpty" in fic else F.lit(0.0)).alias("n_cpty"),
                     fl("standing").alias("standing"), fl("active").alias("active"), fl("in_prev").alias("in_prev"))
             .filter(~F.col("is_pnc") & F.col("fi").isNotNull() & (F.col("fi") != "") & ~F.col("fi").rlike(r"^PNC\b")))
    IDS = sorted(set(DL.pwr.dropna()) | set(cur.pwr.dropna()))
    ids_df = spark.createDataFrame([(x,) for x in IDS], "pwr string")
    FIQ = topd(FIB.join(F.broadcast(ids_df), "pwr").groupBy("pwr", "t", "fi").agg(
        F.sum("amt").alias("amt"), F.sum("selfpay").alias("selfpay"), F.sum("n_cpty").alias("n_cpty"),
        F.max("standing").alias("standing"), F.max(((F.col("active") == 1) & (F.col("in_prev") == 0)).cast("int")).alias("new")))
    for c in ("amt", "selfpay", "n_cpty"): FIQ[c] = FIQ[c].fillna(0.0)
    GLB = topd(FIB.groupBy("t", "fi").agg(F.sum("amt").alias("g_amt")))
    FIQ = FIQ.sort_values(["pwr", "t", "fi"]).reset_index(drop=True)
    BYP = {k: g for k, g in FIQ.groupby("pwr")}

    # ── label: the bank whose dollars rose most around the exit, vs the client's own months −9…−4 ──
    def bank_rise(pwr, e):
        """(bank with the largest own-name rise, rise, bank with the largest payments rise, rise)."""
        h = BYP.get(pwr)
        lo = max(0, e - 9); n0 = (e - 4) - lo + 1
        if h is None or n0 < 3: return None, np.nan, None, np.nan
        w1 = h[(h.t >= e - 3) & (h.t <= e)].groupby("fi")[["amt", "selfpay"]].sum()
        if not len(w1): return None, np.nan, None, np.nan
        w0 = h[(h.t >= lo) & (h.t <= e - 4)].groupby("fi")[["amt", "selfpay"]].sum() * (4.0 / n0)
        xs = w1.sub(w0.reindex(w1.index).fillna(0.0)).sort_index()
        return xs.selfpay.idxmax(), float(xs.selfpay.max()), xs.amt.idxmax(), float(xs.amt.max())
    BR = pd.DataFrame([bank_rise(p_, e_) for p_, e_ in zip(DL.pwr, DL.e)], index=DL.index, columns=["sb", "sx", "pb", "px"])

    # ── no bank visible: did outflows to processors / brokerages / bank-named payees rise instead? ──
    ids2 = spark.createDataFrame([(x,) for x in DL.cust_pwr_id.astype(str)], "cust_pwr_id string")
    PRQ = topd(FEAT.join(F.broadcast(ids2), "cust_pwr_id").select("cust_pwr_id", "t", *[f"raw_{c}" for c in RAWP]))
    PRB = {k: g.set_index("t") for k, g in PRQ.groupby("cust_pwr_id")}
    def via_of(c, e, norm):
        h = PRB.get(c)
        if h is None or not (norm > 0): return None
        lo = max(0, e - 9); n0 = (e - 4) - lo + 1
        if n0 < 3: return None
        best = None
        for g in VIA_GROUPS:
            col = f"raw_pr_{g}_amt_out"
            x = h[col].loc[e - 3:e].sum() - h[col].loc[lo:e - 4].sum() * 4.0 / n0
            if x >= DEST_MIN * norm and (best is None or x > best[1]): best = (g, x)
        return best[0] if best else None
    DL["via"] = [via_of(c, e_, n_) for c, e_, n_ in zip(DL.cust_pwr_id, DL.e, DL.norm)]
    # order of evidence: own-name account at a bank › processors / brokerages (the bank behind them is not visible, and a rise in
    # "payments" at a bank is then usually the processor's own bank) › payments to a bank's customers
    thr = DEST_MIN * DL.norm
    own_ok = BR.sx >= thr; via_ok = ~own_ok & DL.via.notna(); pay_ok = ~own_ok & ~via_ok & (BR.px >= thr)
    DL["dest"] = np.where(own_ok, BR.sb, np.where(pay_ok, BR.pb, None))
    DL["dest_src"] = np.select([own_ok, via_ok, pay_ok], ["own-name account", "processors", "payments"], "none")
    DL["dest_excess"] = np.where(own_ok, BR.sx, np.where(pay_ok, BR.px, np.nan))
    DL["evidence"] = np.select([own_ok, via_ok, pay_ok], ["own-name account at another bank", "processors / brokerages rose (bank not visible)",
                                                         "payments to another bank's customers rose"], "not identified")
    EVID = (DL.groupby(["period", "evidence"]).agg(departures=("e", "size"), normal_balance=("norm", "sum")).reset_index())
    EVID["share_of_dollars"] = EVID.normal_balance / EVID.groupby("period").normal_balance.transform("sum")
    VIA = (DL[DL.dest_src == "processors"].assign(group=lambda d: d.via.map(GRPT))
             .groupby("group").agg(departures=("e", "size"), normal_balance=("norm", "sum")).reset_index())
    tp = DL[DL.e > TRAIN_E]
    DTOP = (tp[tp.dest.notna()].groupby("dest").agg(departures=("e", "size"), normal_balance=("norm", "sum"))
              .sort_values("normal_balance", ascending=False).head(15).reset_index())

    # ── candidates and features at T = exit − lead (data up to T only) ──
    TOPG = list(GLB[GLB.t <= TRAIN_E].groupby("fi").g_amt.sum().sort_values(ascending=False).head(DEST_TOPG).index)
    TOPG_ALL = list(GLB.groupby("fi").g_amt.sum().sort_values(ascending=False).head(DEST_TOPG).index)
    gp = GLB.pivot_table(index="t", columns="fi", values="g_amt", aggfunc="sum").reindex(range(END_T + 1)).fillna(0.0)
    g3 = gp.rolling(3, min_periods=1).sum(); g6 = gp.shift(3).rolling(6, min_periods=1).sum().fillna(0.0)
    GF = pd.DataFrame({"glob_share": g3.div(g3.sum(axis=1).replace(0, np.nan), axis=0).stack(),
                       "glob_grow": (np.log1p(g3 / 3) - np.log1p(g6 / 6)).stack()})
    GF.index.names = ["T", "fi"]; GF = GF.reset_index()
    lab_d = DL[DL.dest.notna()]
    if len(lab_d) < 30: raise RuntimeError(f"only {len(lab_d)} departures with an identified destination bank")
    pc_ = (lab_d.groupby(["e", "dest"]).size().unstack(fill_value=0).reindex(range(END_T + 2), fill_value=0).cumsum().shift(1).fillna(0))
    pr_s = pc_.div(pc_.sum(axis=1).replace(0, np.nan), axis=0).stack()
    pr_s.index.names = ["T", "fi"]; PRIOR = pr_s.rename("prior").reset_index()
    Qs = [(r.pwr, r.e - L, L, r.e, r.dest, r.dest_src, "train" if r.e <= TRAIN_E else "test")
          for r in lab_d.itertuples() for L in DEST_LEADS if r.e - L >= 3]
    Qs += [(p_, CUR_T, 0, None, None, None, "list") for p_ in cur.pwr]
    QD = pd.DataFrame(Qs, columns=["pwr", "T", "lead", "e", "dest", "dest_src", "split"]); QD["qid"] = np.arange(len(QD))
    AGG = {"a3": "sum", "a12": "sum", "s3": "sum", "s12": "sum", "s6": "sum", "n3": "sum", "n12": "sum", "act12": "sum", "new3": "max", "std3": "max"}
    parts = []
    for i0 in range(0, len(QD), 2000):
        j = QD.iloc[i0:i0 + 2000][["qid", "pwr", "T"]].merge(FIQ, on="pwr")
        j = j[(j.t >= j["T"] - 11) & (j.t <= j["T"])]
        r_ = j.t - j["T"]
        j = j.assign(a3=j.amt * (r_ >= -2), a12=j.amt * (r_ <= -3), s3=j.selfpay * (r_ >= -2), s12=j.selfpay * (r_ <= -3),
                     s6=j.selfpay * (r_ >= -5), n3=j.n_cpty * (r_ >= -2), n12=j.n_cpty * (r_ <= -3), act12=(j.amt > 0).astype(int),
                     new3=j.new * (r_ >= -2), std3=j.standing * (r_ >= -2))
        parts.append(j.groupby(["qid", "fi"]).agg(AGG).reset_index())
    CD = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["qid", "fi"] + list(AGG))
    CD = CD[(CD.a3 + CD.a12) > 0]
    TG = pd.concat([QD.loc[QD.split != "list", ["qid"]].merge(pd.DataFrame({"fi": TOPG}), how="cross"),
                    QD.loc[QD.split == "list", ["qid"]].merge(pd.DataFrame({"fi": TOPG_ALL}), how="cross")], ignore_index=True)
    CD = CD.merge(TG, on=["qid", "fi"], how="outer").fillna({k: 0.0 for k in AGG})
    CD = CD.merge(QD, on="qid", how="left")
    q_ = CD.groupby("qid")
    tot3, tot12, ts6 = q_.a3.transform("sum"), q_.a12.transform("sum"), q_.s6.transform("sum")
    CD["sh3"] = np.where(tot3 > 0, CD.a3 / tot3.where(tot3 > 0, 1), np.nan)
    CD["sh12"] = np.where(tot12 > 0, CD.a12 / tot12.where(tot12 > 0, 1), np.nan)
    CD["dsh"] = CD.sh3 - CD.sh12
    CD["la3"], CD["ls3"] = np.log1p(CD.a3.clip(lower=0)), np.log1p(CD.s3.clip(lower=0))
    CD["dls"] = np.log1p(CD.s3.clip(lower=0) / 3) - np.log1p(CD.s12.clip(lower=0) / 9)
    CD["ssh6"] = np.where(ts6 > 0, CD.s6 / ts6.where(ts6 > 0, 1), 0.0)
    CD["ln3"], CD["dn"] = np.log1p(CD.n3.clip(lower=0)), CD.n3 / 3 - CD.n12 / 9
    CD["rank3"] = np.where(CD.a3 > 0, q_.a3.rank(ascending=False, method="min"), 99)
    CD["top_self"] = ((CD.s6 > 0) & (CD.s6 == q_.s6.transform("max"))).astype(int)
    CD["in_hist"] = ((CD.a3 + CD.a12) > 0).astype(int)
    CD = CD.merge(GF, on=["T", "fi"], how="left").merge(PRIOR, on=["T", "fi"], how="left")
    CD["prior"] = CD.prior.fillna(0.0)
    CD["y"] = (CD.fi == CD.dest).astype(int)
    DFEATS = ["sh3", "sh12", "dsh", "la3", "ls3", "dls", "ssh6", "ln3", "dn", "act12", "new3", "std3", "rank3", "top_self",
              "in_hist", "glob_share", "glob_grow", "prior"]

    def dest_fit(D):
        """Pointwise ranking model (small trees; ridge logistic without a tree library); 20% of clients for early stopping."""
        hv = (pd.util.hash_pandas_object(D.pwr.astype(str), index=False).to_numpy() % 1000) < 200
        Xa, ya, Xv, yv = D.loc[~hv, DFEATS].to_numpy(float), D.y[~hv].to_numpy(), D.loc[hv, DFEATS].to_numpy(float), D.y[hv].to_numpy()
        if GBM == "lightgbm":
            import lightgbm as lgb
            pr = dict(objective="binary", learning_rate=0.05, num_leaves=15, max_depth=4, min_data_in_leaf=20, feature_fraction=0.8,
                      bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=SEED, num_threads=os.cpu_count())
            dt = lgb.Dataset(Xa, ya); dv = lgb.Dataset(Xv, yv, reference=dt)
            try:    m = lgb.train(pr, dt, 2000, valid_sets=[dv], callbacks=[lgb.early_stopping(100, verbose=False)])
            except (AttributeError, TypeError): m = lgb.train(pr, dt, 2000, valid_sets=[dv], early_stopping_rounds=100, verbose_eval=False)
            return lambda Z: m.predict(Z, num_iteration=m.best_iteration or None)
        if GBM == "xgboost":
            import xgboost as xgb
            pr = dict(objective="binary:logistic", eval_metric="logloss", eta=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
                      min_child_weight=5, reg_lambda=1.0, tree_method="hist", nthread=os.cpu_count(), seed=SEED)
            m = xgb.train(pr, xgb.DMatrix(Xa, label=ya, missing=np.nan), 2000,
                          evals=[(xgb.DMatrix(Xv, label=yv, missing=np.nan), "v")], early_stopping_rounds=100, verbose_eval=False)
            n_ = int(m.best_iteration) + 1
            def _p(Z):
                d = xgb.DMatrix(Z, missing=np.nan)
                try: return m.predict(d, iteration_range=(0, n_))
                except TypeError: return m.predict(d, ntree_limit=n_)
            return _p
        if GBM == "sklearn":
            from sklearn.ensemble import HistGradientBoostingClassifier
            m = HistGradientBoostingClassifier(learning_rate=0.05, max_depth=4, max_iter=2000, min_samples_leaf=20, early_stopping=True,
                                               validation_fraction=0.15, n_iter_no_change=50, random_state=SEED)
            m.fit(D[DFEATS].to_numpy(float), D.y.to_numpy())
            return lambda Z: m.predict_proba(Z)[:, 1]
        Xf = D[DFEATS].to_numpy(float); mu = np.nanmean(Xf, 0); sd = np.nanstd(Xf, 0); sd[~(sd > 1e-9)] = 1.0
        zz = lambda Z: np.nan_to_num((Z - mu) / sd).astype(np.float32)
        bb, _ = logit_newton(zz(Xf), D.y.to_numpy(float), np.ones(len(D)), 1.0)
        return lambda Z: lin_score(bb, zz(Z))

    def first_by(D, cols, k=1):
        """Top-k candidates per query by columns in order (ties: bank name), as {qid: [banks]}."""
        cols = [cols] if isinstance(cols, str) else list(cols)
        s_ = D.sort_values(["qid"] + cols + ["fi"], ascending=[True] + [False] * len(cols) + [True])
        return s_.groupby("qid").fi.apply(lambda v: list(v.head(k))).to_dict()
    TRD, TED = CD[CD.split == "train"], CD[CD.split == "test"]
    if TRD.y.sum() < 30: raise RuntimeError(f"only {int(TRD.y.sum())} departures with a destination before the test period")
    pred = dest_fit(TRD)
    TED = TED.assign(score=pred(TED[DFEATS].to_numpy(float)))
    glob_dest = lab_d[lab_d.e <= TRAIN_E].dest.value_counts().index[0]
    m1, m3 = first_by(TED, "score"), first_by(TED, "score", 3)
    btop = first_by(TED, ["a3", "a12"])                         # the client's current top bank (last 3 months, then last 12)
    bself = first_by(TED, ["s6", "a3", "a12"])                  # its top own-name bank (last 6 months), else the top bank
    qq = QD[QD.split == "test"].copy()
    qq["cands"] = qq.qid.map(TED.groupby("qid").fi.apply(set))
    qq["in_cands"] = [d in (c or set()) for d, c in zip(qq.dest, qq.cands)]
    qq["model_top1"] = [m1.get(q, [None])[0] == d for q, d in zip(qq.qid, qq.dest)]
    qq["model_top3"] = [d in m3.get(q, []) for q, d in zip(qq.qid, qq.dest)]
    qq["current_top_bank"] = [btop.get(q, [None])[0] == d for q, d in zip(qq.qid, qq.dest)]
    qq["top_own_name_bank"] = [bself.get(q, [None])[0] == d for q, d in zip(qq.qid, qq.dest)]
    qq["most_common_destination"] = qq.dest == glob_dest
    MET_D = ["in_cands", "model_top1", "model_top3", "current_top_bank", "top_own_name_bank", "most_common_destination"]
    DEST_ACC = qq.groupby("lead")[MET_D].mean().reset_index().merge(qq.groupby("lead").size().rename("departures").reset_index(), on="lead")
    DEST_SRC = qq.groupby(["lead", "dest_src"])[["model_top1", "current_top_bank", "top_own_name_bank"]].mean().reset_index().merge(
        qq.groupby(["lead", "dest_src"]).size().rename("departures").reset_index(), on=["lead", "dest_src"])
    # ── final model on every labelled departure → likely destinations for the current list ──
    predA = dest_fit(CD[CD.split != "list"])
    LC = CD[CD.split == "list"].assign(score=lambda d: predA(d[DFEATS].to_numpy(float)))
    LC["share"] = LC.score / LC.groupby("qid").score.transform("sum")
    LC = LC.sort_values(["qid", "score", "fi"], ascending=[True, False, True])
    top3 = LC.groupby("qid").head(3).assign(k=lambda d: d.groupby("qid").cumcount() + 1)
    wide = top3.pivot(index="qid", columns="k", values=["fi", "share"])
    wide.columns = [f"{'bank' if a == 'fi' else 'bank_share'}_{b}" for a, b in wide.columns]
    own_ = LC[LC.s6 > 0].sort_values(["qid", "s6"], ascending=[True, False]).groupby("qid").head(1).set_index("qid")
    LST["qid"] = QD.loc[QD.split == "list", "qid"].to_numpy()       # one query per list row, same order
    LST = LST.merge(wide.reset_index(), on="qid", how="left")
    LST["own_name_accounts_at"] = LST.qid.map(own_.fi); LST["own_name_6m"] = LST.qid.map(own_.s6)
    LST = LST.drop(columns="qid")
    a1 = DEST_ACC.set_index("lead")
    R.update(dest_acc=DEST_ACC, dest_src=DEST_SRC, dest_evid=EVID, dest_via=VIA, dest_top=DTOP,
             dest_top1_l1=float(a1.model_top1.get(1, np.nan)), dest_top1_l3=float(a1.model_top1.get(3, np.nan)),
             dest_base_l3=float(a1.current_top_bank.get(3, np.nan)), dest_base_l1=float(a1.current_top_bank.get(1, np.nan)),
             dest_top3_l1=float(a1.model_top3.get(1, np.nan)),
             dest_cov=float(tp.loc[tp.dest.notna(), "norm"].sum() / max(tp.norm.sum(), 1.0)),
             dest_cov_self=float(tp.loc[tp.dest_src == "own-name account", "norm"].sum() / max(tp.norm.sum(), 1.0)),
             dest_via_share=float(tp.loc[tp.dest_src == "processors", "norm"].sum() / max(tp.norm.sum(), 1.0)),
             dest_n_train=int(TRD.y.sum()), dest_n_test=int(qq[["pwr", "e"]].drop_duplicates().shape[0]), dest_glob=glob_dest)
    DL.to_csv(OUT / "m5_10b_departures_destination_FULL_IDS.csv", index=False)
except Exception as e_:
    DEST_ERR = f"{type(e_).__name__}: {e_}"
    B.check("destination bank", "FIX", f"skipped — {_html.escape(DEST_ERR)}")
    B.note(f"<pre style='font-size:10px;color:#777'>{_html.escape(traceback.format_exc()[-2500:])}</pre>")
R["dest_err"] = DEST_ERR

# ── the current list, written with full ids ──────────────────────────
LST.to_csv(OUT / "m5_10b_current_list_FULL_IDS.csv", index=False)
show_ = LST.head(N_LIST_SHOW).copy()
show_.insert(1, "client", show_.cust_pwr_id.map(disp))
if "bank_1" in show_.columns:
    show_["likely_destination"] = [("no clear destination: " if not (r_[3] >= 0.25) else "")
                                   + " · ".join(f"{b} {s_:.0%}" for b, s_ in zip(r_[:3], r_[3:]) if isinstance(b, str) and s_ >= 0.05)
                                   for r_ in show_[[c for c in ("bank_1", "bank_2", "bank_3", "bank_share_1", "bank_share_2", "bank_share_3")
                                                    if c in show_.columns]].values] if "bank_share_3" in show_.columns else show_.bank_1
show_["reasons"] = [" · ".join(short(x, 70) for x in r_.split(" · ")) for r_ in show_.reasons]
show_ = show_[[c for c in ("rank", "client", "probability_6m", "normal", "likely_destination", "own_name_accounts_at", "reasons") if c in show_.columns]]
R.update(cur_month=ym_of(CUR_T), cur_list=show_, cur_n=len(LST), cur_normal=float(LST.normal.sum()),
         cur_p_mean=float(LST.probability_6m.mean()) if len(LST) else np.nan)
if EVID is not None:
    B.table(EVID, "10b-1 · Departures by what shows where the money went (§7 exclusions removed)", pct=("share_of_dollars",),
            money=("normal_balance",), save="m5_10b1_evidence")
    if VIA is not None and len(VIA): B.table(VIA, "10b-2 · No bank visible: which outflows rose", money=("normal_balance",))
    B.table(DTOP, "10b-3 · Most common destinations (test period and after)", money=("normal_balance",), save="m5_10b3_destinations")
if DEST_ACC is not None:
    B.table(DEST_ACC, "10b-4 · Naming the destination bank, test-period departures (share right; in_cands = the ceiling)",
            pct=tuple(MET_D), save="m5_10b4_dest_accuracy")
    B.table(DEST_SRC, "10b-5 · By how the destination was identified", pct=("model_top1", "current_top_bank", "top_own_name_bank"))
B.table(show_, f"10b-6 · Current list, {ym_of(CUR_T)}: top {N_LIST_SHOW} of {len(LST):,} (full list in m5_10b_current_list_FULL_IDS.csv)",
        pct=("probability_6m",), money=("normal",))
B.show(t0)

## §11–12 Prediction scorecard and the HTML report

Model 4's sections, plus what the network features add (§8.12), the 2026 months (§8.13), where the money goes (§8.14) and
the current list (§8.15). Section 1 ("What we found") is drafted from this run's numbers; review the wording before
circulating.

In [ ]:
# =====================================================================
# 12 · PREDICTIONS · HTML REPORT                                    [OUTPUT BLOCK 12]
# =====================================================================
t0 = time.time(); B = Block("§12 · Predictions and report")
_v = lambda mn, a, K: float(VAL[(VAL.model == mn) & (VAL.alpha == a) & (VAL.calls == K)].dollars_reached_year.iloc[0])
_g = lambda k, d=np.nan: R.get(k, d) if R.get(k) is not None else d
PRED = [
    ("P1", "Pooled AUC: full tree ≥ model-4-features tree + 0.005", 0.45,
     lambda: (R["auc_gbm"] - R["auc_gbm_base"] >= 0.005) if R["ablation"] else None, f"{_g('auc_gbm') - _g('auc_gbm_base'):+.4f}"),
    ("P2", "Dollars reached (1,000 calls, α = 0.5): full tree ≥ 1.03 × model-4-features tree", 0.40,
     lambda: (R["main_reached"] >= 1.03 * R["base_reached"]) if R["ablation"] else None, f"{_g('main_reached') / max(_g('base_reached', 1.0), 1.0):.3f}×"),
    ("P3", "At least one network feature in the top 25 (tree SHAP)", 0.75, lambda: R["n_net_top25"] >= 1, f"{_g('n_net_top25', 0)}"),
    ("P4", "A processor or NAICS-mix feature alone: AUC ≥ 0.55", 0.45, lambda: R["best_pn_auc"] >= 0.55, f"{_g('best_pn_auc'):.3f}"),
    ("P5", "2026 precision within ±30% of the test months' at the same follow-up", 0.60,
     lambda: 0.7 <= R["fwd_prec_ratio"] <= 1.3, f"{_g('fwd_prec_ratio'):.2f}×"),
    ("P6", "Destination bank top-1 ≥ 50%, one month before the exit", 0.50, lambda: R["dest_top1_l1"] >= 0.5, f"{_g('dest_top1_l1'):.1%}"),
    ("P7", "Destination model ≥ current top bank + 5 points (top-1), three months before", 0.45,
     lambda: R["dest_top1_l3"] - R["dest_base_l3"] >= 0.05, f"{_g('dest_top1_l3') - _g('dest_base_l3'):+.1%}"),
    ("P8", "≥ 40% of departing normal balance (test period) has an identified destination bank", 0.40,
     lambda: R["dest_cov"] >= 0.40, f"{_g('dest_cov'):.1%}")]
rows = []
for pid, txt, conf, fn, obs in PRED:
    try:
        hit = fn(); hit = None if hit is None or (isinstance(hit, float) and np.isnan(hit)) else bool(hit)
    except Exception: hit = None
    try: obs = obs if isinstance(obs, str) else str(obs)
    except Exception: obs = "n/a"
    if "nan" in obs: hit, obs = None, "—"
    rows.append({"id": pid, "prediction": txt, "confidence": conf, "observed": obs,
                 "result": "HELD" if hit else "FAILED" if hit is False else "n/a",
                 "brier": (conf - float(hit)) ** 2 if hit is not None else np.nan})
PS = pd.DataFrame(rows); R["preds"] = PS
B.table(PS, "12a · Pre-registered predictions", pct=("confidence",), save="m5_12a_predictions")

def img(name):
    return f"<img src='data:image/png;base64,{base64.b64encode(open(CHARTS[name], 'rb').read()).decode()}' style='max-width:100%'>"
def tab(df, pct=(), money=(), rows=60):
    return fmt_df(df.head(rows), pct, money).to_html(index=False, border=0, classes="t", escape=True)
usd = lambda v: f"${v / 1e9:,.2f}bn" if abs(v) >= 1e9 else f"${v / 1e6:,.1f}m" if abs(v) >= 1e6 else f"${v:,.0f}"
HVd = R["headline"]
S = [f"""<h1>Deposit attrition early warning — v13 model 5</h1>
<p class='sub'>Payment Knowledge Graph (PKG) · Treasury Management · DSI · test months {R['test_months'][0]} … {R['test_months'][-1]} ·
primary model: {R['primary']}{f' ({R["engine"]})' if R['engine'] else ''}</p>"""]
def _fnum(df, col, row_match, key="dollars"):
    try: return float(df.loc[df[key] == row_match, col].iloc[0])
    except Exception: return np.nan
_early = R["tim"][(R["tim"].kind == "early signal") & R["tim"].onset_month.notna() & R["tim"].set.str.startswith("payment")].sort_values("onset_month")
_early_txt = "; ".join(f"{l.split(' — ')[0][:60]} {d} (from {int(m)} months)" for l, d, m in
                       zip(_early.label.head(3), _early.leavers.head(3), _early.onset_month.head(3))) or "no payment change crossed the bar"
_n_in = int(_early.feature.head(3).str.contains("_in_|_in$|_in__").sum()) if len(_early) else 0
_side = "mostly on the receiving side" if _n_in >= 2 else "mostly on the paying side" if len(_early) and _n_in <= 1 else ""
def _ls(feat, col):
    try: return float(R["ls"].loc[R["ls"].feature == label(feat), col].iloc[0])
    except Exception: return np.nan
_bn3 = _ls("dep_bal_vs_norm", "leavers -3") - _ls("dep_bal_vs_norm", "stayers -3")
_out1 = _ls("amt_all_out__d6", "leavers -1")
_bal_txt = ("The balance itself stays close to stayers' level until the final month" if pd.notna(_bn3) and abs(_bn3) < 0.1
            else "The balance falls below stayers' level in the months before the exit" if pd.notna(_bn3) else "")
_bal_txt += (", when the money goes out as a payment." if pd.notna(_out1) and _out1 > 0.5 else "." if _bal_txt else "")
_fo = R["folds"]; _ahead = int((_fo.auc_tree > _fo.auc_simple).sum()) if R["engine"] else 0
_better = R["engine"] and R["auc_gbm"] > R["auc_lin"]
_bv = R.get("ls")
_cap = float(HVd.reached_per_year.iloc[-1]); _top5 = float(HVd.reached_per_year.iloc[2]) if len(HVd) > 2 else np.nan
_lo = min(_cap, _top5) if pd.notna(_top5) else _cap
_ac = R["acc"]; _pr = float(_ac[(_ac.model == MAIN) & (_ac.alpha == A_MAIN) & (_ac.calls == K_MAIN)].precision_mean.iloc[0])
_prs = float(_ac[(_ac.model == "simple") & (_ac.alpha == A_MAIN) & (_ac.calls == K_MAIN)].precision_mean.iloc[0])
_vc = R["audit"].verdict.value_counts()
def _pct(v, d=0): return "—" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.{d}%}"
_ni = R.get("net_imp"); _ns = R.get("net_sig")
_sh = lambda s_, n=90: s_ if len(s_) <= n else s_[:n - 1] + "…"
_top_net = "; ".join(_sh(l) for l in _ni.label.head(3)) if _ni is not None and len(_ni) else "none"
_pn = _ns[_ns.set.isin(["processors", "NAICS mix"])] if _ns is not None and len(_ns) else None
_pn_txt = ("; ".join(f"{_sh(r.label)} ({r.signal}, {r.leavers_beyond:.0%} of leavers beyond the stayers' threshold against "
                     f"{r.stayers_beyond:.0%} of stayers)" for r in _pn.head(2).itertuples()) if _pn is not None and len(_pn) else "")
_abl_better = R.get("ablation") and R["auc_gbm"] > R["auc_gbm_base"]
NEW_FIND = ""
if R.get("ablation"):
    NEW_FIND += f"""<li><b>The network features {"add a little" if _abl_better and R['diff_base_ci'][0] > 0 else "add little" if _abl_better else "do not add"}
to the model.</b> With them the tree's AUC is {R['auc_gbm']:.3f} against {R['auc_gbm_base']:.3f} on model 4's features
(ahead in {int((R['folds'].auc_tree > R['folds'].auc_tree_base).sum())} of {len(R['folds'])} test months); the dollars reached at {K_MAIN:,}
calls change by {usd(R['diff_base'])} a year (95% {usd(R['diff_base_ci'][0])} to {usd(R['diff_base_ci'][1])}). They carry
{R['net_share']:.0%} of the model's attribution; the strongest are {_top_net}.{(" Processors and industry mix on their own: " + _pn_txt + ".") if _pn_txt else ""}</li>"""
if R.get("fwd_tab") is not None:
    NEW_FIND += f"""<li><b>The model holds on the 2026 months so far.</b> Scored each month with models that saw only the outcomes known then,
precision at {K_MAIN:,} calls is {R['fwd_prec']:.1%} against {R['fwd_test_prec']:.1%} for the test months cut to the same follow-up
({R['fwd_prec_ratio']:.2f}×), AUC {R['fwd_auc']:.3f} against {R['fwd_test_auc']:.3f}. Exits are observed through {R['obs_end']}.</li>"""
if R.get("dest_acc") is not None:
    NEW_FIND += f"""<li><b>Where the money goes.</b> For {_pct(R['dest_cov'])} of the departing normal balance in the test period a destination bank is visible
({_pct(R['dest_cov_self'])} through the client's own accounts at that bank); for another {_pct(R['dest_via_share'])} the outflows to
money-movement processors or brokerages rose instead, so the bank behind them is not visible. One month before the exit the model names the right bank
first for {_pct(R['dest_top1_l1'])} of departures (top three: {_pct(R['dest_top3_l1'])}), against {_pct(R['dest_base_l1'])} for the client's current
top bank; three months before, {_pct(R['dest_top1_l3'])} against {_pct(R['dest_base_l3'])}.</li>"""
FINDINGS = f"""
<ol class='find'>
<li><b>About {R['exits_per_year']:,.0f} Treasury Management clients a year move their money out, holding {usd(R['at_risk_year'])} of normal balance.</b>
{R['bal_exit_share']:.0%} of those dollars leave through <i>balance exits</i>: the client drains its balance and leaves the accounts open, which a
closure-based measure never counts. The dollars are concentrated: the largest single client is {R['top_client_share']:.0%} of the total.</li>
<li><b>Payment behaviour changes before the balance moves.</b> Against stayers of the same size, the first changes to cross the bar are
{_side + " — " if _side else ""}{_early_txt}. {_bal_txt}
Leavers' scores rise towards the exit (gap to stayers 6 months out = {R['gap_ratio']:.0%} of the gap 1 month out).</li>
<li><b>The tree model {"ranks leavers better than" if _better else "does not beat"} the simple model</b>: AUC {R['auc_gbm']:.3f} against {R['auc_lin']:.3f},
ahead in {_ahead} of {len(_fo)} test months; precision at {K_MAIN:,} calls a month {_pr:.1%} against {_prs:.1%}. The dollar difference
({usd(R['diff'])} a year, 95% {usd(R['diff_ci'][0])} to {usd(R['diff_ci'][1])}) is {"significant" if R['diff_ci'][0] > 0 else "not yet proven"}.</li>
<li><b>{K_MAIN:,} calls a month reach {usd(R['main_reached'])} a year of departing balance ({R['main_share']:.0%} of it)</b>, with {usd(_lo)} as the
cautious figure (largest five clients removed, or each client's credit capped at ${CAP_CREDIT / 1e6:,.0f}m). What that saves depends on the save
rate, which only a pilot can measure: at 5% it is {usd(_lo * 0.05)}–{usd(R['main_reached'] * 0.05)} a year, at 20% {usd(_lo * 0.20)}–{usd(R['main_reached'] * 0.20)}.</li>
<li><b>How early the list warns.</b> {R['lead3_sus_usd']:.0%} of the dollars reached had a sustained alert (two
consecutive months on the list) three or more months before the money left ({R['lead3_sus']:.0%} of the clients reached);
{R['single_only']:.0%} of reached clients appeared on the list only in one-month blips.</li>
<li><b>The largest departures were checked in the data.</b> Of {R['n_audited']} checked: {_vc.get('departed', 0)} departed,
{_vc.get('moved to another bank', 0)} moved their money to their own accounts at another bank, {_vc.get('still transacting', 0)} still transact here,
{_vc.get('moved within PNC', 0)} moved within PNC and {_vc.get('came back', 0)} came back. The last two groups are not losses; all need an RM's confirmation.</li>
{NEW_FIND}
</ol>"""
S.append(f"""<section><h2>1 · What we found</h2><div class='box warn'><b>Draft, generated from this run.</b> Review the wording and the
example clients with the RM leads before circulating.</div>{FINDINGS}
<h3>Headline numbers</h3>{tab(pd.DataFrame([
 ["clients whose money left, per year (after the §7 check)", f"{R['exits_per_year']:,.0f}"],
 ["their normal balance, per year", usd(R['at_risk_year'])], ["…share leaving through balance exits (accounts left open)", f"{R['bal_exit_share']:.1%}"],
 ["pooled AUC, simple / tree", f"{R['auc_lin']:.3f} / {R['auc_gbm']:.3f}" if R['engine'] else f"{R['auc_lin']:.3f} / —"],
 [f"dollars reached per year, {K_MAIN:,} calls, α = {A_MAIN}", f"{usd(R['main_reached'])} (95% {usd(R['main_ci'][0])} – {usd(R['main_ci'][1])})"],
 ["…without the largest 5 departing clients", usd(_top5) if pd.notna(_top5) else "—"],
 [f"…with each client's credit capped at ${CAP_CREDIT / 1e6:,.0f}m", usd(_cap)],
 ["tree − simple, dollars reached per year", f"{usd(R['diff'])} (95% {usd(R['diff_ci'][0])} – {usd(R['diff_ci'][1])})" if R['engine'] else "—"],
 ["sustained alert 3+ months before the money left", f"{R['lead3_sus']:.1%} of reached clients, {R['lead3_sus_usd']:.1%} of dollars"],
 ["reached only by one-month blips", f"{R['single_only']:.1%} of reached clients"],
 ["leavers' score gap 6 months out ÷ 1 month out", f"{R['gap_ratio']:.2f}"],
 ["pooled AUC, tree on all features / on model 4's features", f"{R['auc_gbm']:.3f} / {R['auc_gbm_base']:.3f}" if R.get('ablation') else "—"],
 ["all features − model 4's features, dollars reached per year", f"{usd(R['diff_base'])} (95% {usd(R['diff_base_ci'][0])} – {usd(R['diff_base_ci'][1])})" if R.get('ablation') else "—"],
 ["2026 months: precision ÷ test months at the same follow-up", f"{R['fwd_prec_ratio']:.2f}×" if R.get('fwd_tab') is not None else "—"],
 ["departing balance with an identified destination bank (test period)", _pct(R.get('dest_cov'), 1) if R.get('dest_acc') is not None else "—"],
 ["destination bank named first, 1 / 3 months before the exit", f"{_pct(R.get('dest_top1_l1'))} / {_pct(R.get('dest_top1_l3'))}" if R.get('dest_acc') is not None else "—"]],
 columns=["measure", "value"]))}</section>""")
S.append("""<section><h2>2 · The question</h2><p>Which Treasury Management deposit clients will move their money out in the
next six months, early enough for a relationship manager to act; how much of that money a fixed number of calls can
reach; how much a stronger model adds; whether the client's counterparties (other banks' customers, PNC customers, payment processors,
the industries it pays) add warning; whether the model holds on the most recent months; and which bank the money goes to.
Model 5 keeps model 4's design (which corrected model 1's focus on which <i>kind</i> of client leaves) and adds the payment network.</p></section>""")
S.append(f"""<section><h2>3 · Data and preparation</h2>
<h3>Sources</h3>{tab(pd.DataFrame([[DEP_TABLE, "daily balances and status per account × leg"], [MET_TABLE, f"{R['n_metrics']} monthly payment metrics per customer"],
 [PAIR_TABLE, "customer × counterparty × month: who the client pays and is paid by"], [CPTY_TABLE, "external counterparties' monthly metrics"],
 [DIM_TABLE, "power id, family and NAICS of every PNC customer"], [FI_TABLE, "customer × bank × month: dollars to other banks, own-name accounts"]]
 + ([[CPF_TABLE, "counterparty features v1.1"]] if R.get('cpf_used') else []), columns=["table", "content"]))}
<h3>Pipeline</h3><ol><li>Legs summed per account-day; balances rolled to customer-day, then month; both status vocabularies mapped to live / closed.</li>
<li>A complete month grid per customer: a month with no rows is non-live with zero balance.</li>
<li>Payment metrics joined on the power id ({R['key_match']:.1%} of deposit customers found); the largest entity per power id and month.</li>
<li>Rows kept: <b>money still here</b> only. Payment levels become gaps to size peers (same month, same decile of normal
balance); changes vs months −6…−4 and vs the last 3 months; three size anchors.</li>
<li>{R['n_dupes']} near-duplicate features dropped (|r| &gt; {CORR_MAX}); one missing-share column per feature set in the simple model;
the tree model reads missing values natively.</li></ol>
<h3>Silent months</h3><p>No payment row after the client's first row = zero activity (dollars and counts 0; shares empty).
{R['pay_seen']:.1%} of live client-months have payment history; {R['pay_silent']:.1%} are silent.</p>
<h3>Checks</h3>{tab(pd.DataFrame(CHECKS))}</section>""")
S.append(f"""<section><h2>4 · Definitions</h2><h3>The target, and why</h3>
<div class='box'><b>The money leaves within the next 6 months</b>: the earlier of a <b>full exit</b> (every account non-live,
confirmed by {CONFIRM_M} observed non-live months) and a <b>balance exit</b> (balance below {BAL_EXIT_FRAC:.0%} of normal for
{BAL_EXIT_HOLD} months while accounts stay open).</div><ul>
<li><b>It is what costs money:</b> {R['bal_exit_share']:.0%} of departing normal balance leaves through balance exits, which a closure-only target never sees.</li>
<li><b>It is earlier:</b> the drain usually precedes the formal closure.</li>
<li><b>Full exits are absorbing:</b> {R['p_return']:.2%} of clients non-live for {CONFIRM_M}+ months came back.</li>
<li><b>Six months</b> leaves time to act; months without six confirmed months ahead ({R['last_label']} onwards) are unlabelled.</li></ul>
<h3>Terms</h3>{tab(pd.DataFrame([
 ["normal balance", f"median balance over up to {NORM_WIN} prior live months (needs {NORM_MIN_OBS})"],
 ["money still here", f"live, normal ≥ ${MSH_FLOOR:,.0f}, current ≥ {MSH_FRAC:.0%} of normal — the only rows trained on, scored and called"],
 ["size peers", "same month, same decile of normal balance"], ["capped credit", "lesser of current and normal balance at the alert"],
 ["α", "score = probability × normal balance^α"], ["dollars reached", "capped balance of departing clients at first alert, counted once, annualised"],
 ["savings", "dollars reached × assumed save rate (1–20%); no call cost"]], columns=["term", "meaning"]))}</section>""")
S.append(f"""<section><h2>5 · Training and test sets</h2><p>Rolling monthly folds: each test month's models see only outcomes known then
(origins ≤ test month − 6); {HOLD_FRAC:.0%} of clients are held out (all their months) to choose the ridge penalty
and tree depth on the first fold, stop the trees early and calibrate their probabilities on every fold. Stayers are sampled at {NEG_RATE:.0%} and weighted back.</p>
{tab(R['tune'])}{tab(R['folds'], pct=('mean_p_simple', 'mean_p_tree', 'actual_rate'))}</section>""")
S.append(f"""<section><h2>6 · Feature sets</h2><p>{R['n_feats']} features after de-duplication ({R['net_kept']} from the payment network); no identity columns.
Network columns are cached under {_html.escape(str(R['net_cache']))} (signature {R['net_signature']}).</p>
{tab(R['net_fam'], pct=('median_coverage', 'max_coverage'))}{tab(R['net_proc'], pct=('client-months paying it (live)',))}
{tab(R['net_naics'], pct=('share of labelled dollars',))}
{tab(R['feature_sets'], pct=('median_coverage',))}{tab(R['forms'])}</section>""")
S.append(f"""<section><h2>7 · Methods</h2><h3>Simple model</h3><p>Ridge logistic regression (L2 = {R['L2']:g}, chosen on validation) on
standardised, clipped features plus one missing-share per feature set; damped Newton.</p>
<h3>Tree model</h3><p>{('Gradient-boosted trees (' + R['engine'] + f'), depth {R["depth"]}, learning rate {GBM_ETA}, early stopping on the validation months; exact TreeSHAP gives the reason codes.') if R['engine'] else 'Not run: no tree library installed.'}</p>
<h3>Validation</h3><p>Backward-looking features; labels need six confirmed months; each fold trains only on outcomes known before its test month;
{HOLD_FRAC:.0%} of clients are held out inside the training window for tuning and early stopping; held-out clients from the {CAL_RECENT} most recent
labelled months calibrate the tree (isotonic, capped at {P_CAP:.0%}); test months are scored in full.</p>
<h3>From a score to dollars</h3><p>Money-still-here clients ranked by p × normal^α; top K called; each departing client credited once, at its first alert, with its capped balance.</p>
<h3>Uncertainty and concentration</h3><p>Per-month accuracy; bootstrap over departing clients for the dollars and for the tree-vs-simple difference; dollars shown without the largest clients and after the §7 check.</p></section>""")
_im = lambda n: img(n) if n in CHARTS else ""
NEWSEC = ""
if R.get("ablation"):
    NEWSEC += f"""<h3>8.12 What the network features add</h3><p>Two trees in the same folds on the same rows: one on model 4's features, one
on all features. Near-duplicates were resolved in favour of model 4's features, so the comparison is fair.</p>{_im('m5_6b_ablation')}
{tab(R['abl_t'])}{tab(R['net_fam_imp'], pct=('share', 'share_simple'))}
<h4>Top network features</h4>{tab(R['net_imp'][['rank', 'feature', 'label', 'set', 'form', 'share', 'alone_auc', 'leavers_sit']], pct=('share',))}
<h4>Network signals on their own (size-matched)</h4><p>Share of leavers' and stayers' months beyond the stayers' 10th or 90th percentile of the same
size decile, test months. Lift above 1.5 is a usable warning; the processor rows test the idea that clients route money to other banks through processors.</p>
{tab(R['net_sig'], pct=('leavers_beyond', 'stayers_beyond', 'coverage', 'shap_share'))}"""
if R.get("fwd_tab") is not None:
    NEWSEC += f"""<h3>8.13 The 2026 months</h3><p>Each month after {R['last_label']} is scored by models fitted only on outcomes known at that month.
Exits are observed through {R['obs_end']}, so each month has a shorter follow-up than six months; the comparison is against the test months
cut to the same follow-up.</p>{_im('m5_8b_forward')}{tab(R['fwd_tab'], pct=('rate', 'test_rate', 'precision', 'test_precision', 'recall_dollars', 'test_recall_dollars'))}
{tab(R['fwd_folds'], pct=('mean_p_tree',))}"""
if R.get("dest_evid") is not None:
    NEWSEC += f"""<h3>8.14 Where the money goes</h3><p>Rises in the exit month and the three before, against the client's own months −9…−4, of at least
{DEST_MIN:.0%} of the normal balance, checked in this order: payments to the client's <b>own name</b> at another bank (that bank is the
destination); outflows to <b>money-movement processors or brokerages</b> (the bank behind them is not visible, and a rise in payments at a bank is
then usually the processor's own bank); payments to <b>another bank's customers</b> (that bank).</p>
{tab(R['dest_evid'], pct=('share_of_dollars',), money=('normal_balance',))}{tab(R['dest_via'], money=('normal_balance',)) if R.get('dest_via') is not None and len(R['dest_via']) else ""}
<h4>Most common destinations</h4>{tab(R['dest_top'], money=('normal_balance',))}"""
if R.get("dest_acc") is not None:
    NEWSEC += f"""<h4>Naming the bank in advance</h4><p>A ranking model over the client's banks of the last 12 months and the {DEST_TOPG} largest
competitor banks, trained on departures up to {TEST_FIRST} and checked on later ones, one and three months before the money left. Baselines:
the client's current top bank, its top own-name bank, and the most common destination ({_html.escape(str(R['dest_glob']))}).
<i>in_cands</i> is the share whose destination was among the candidates (the ceiling).</p>
{tab(R['dest_acc'], pct=('in_cands', 'model_top1', 'model_top3', 'current_top_bank', 'top_own_name_bank', 'most_common_destination'))}
{tab(R['dest_src'], pct=('model_top1', 'current_top_bank', 'top_own_name_bank'))}"""
elif R.get("dest_err"):
    NEWSEC += f"<h3>8.14 Where the money goes</h3><p>Not available in this run: {_html.escape(R['dest_err'])}</p>"
NEWSEC += f"""<h3>8.15 The current list ({R['cur_month']})</h3><p>The top {K_MAIN:,} money-still-here clients by p × normal^{A_MAIN} in the last
scored month ({R['cur_n']:,} clients, {usd(R['cur_normal'])} normal balance, mean probability {_pct(R['cur_p_mean'], 1)}). The top {N_LIST_SHOW} are shown;
the full list with power ids, reasons, the suggested play and the likely destination banks is in m5_10b_current_list_FULL_IDS.csv.
Bank shares are the model's split among candidate banks if the money leaves.</p>
{tab(R['cur_list'], pct=('probability_6m',), money=('normal',), rows=N_LIST_SHOW)}"""
S.append(f"""<section><h2>8 · Results</h2>
<h3>8.1 How much is at risk</h3><p>{R['exits_per_year']:,.0f} clients a year move their money out, with {usd(R['at_risk_year'])} of normal balance
({R['bal_exit_share']:.0%} through balance exits). The largest single client is {R['top_client_share']:.1%} of it.</p>{tab(R['at_risk_tab'], money=('normal_balance', 'per_year'))}
<h3>8.2 Checking the largest departures</h3>{tab(R['audit_counts'], money=('normal',))}{tab(R['audit'], pct=('balance_after_vs_normal', 'siblings_absorbed', 'activity_after_vs_before', 'own_name_out_vs_drop'), money=('normal', 'test_pool_capped'))}
<h3>8.3 Which features predict</h3>{img('m5_6_importance')}{tab(R['form_imp'], pct=('share', 'share_simple'))}
{tab(R['imp'].head(25), pct=('share', 'share_simple'))}{tab(R['fam_imp'], pct=('share', 'share_simple'))}
<h3>8.4 Simple vs tree model</h3>{tab(R['cmp'], pct=tuple(c for c in R['cmp'].columns if c.startswith('precision_')), money=tuple(c for c in R['cmp'].columns if c in ('simple', 'tree')))}
<h3>8.5 Accuracy against calls</h3>{img('m5_8_accuracy_vs_calls')}{tab(R['acc'], pct=('precision_mean', 'precision_min', 'precision_max', 'recall_clients', 'recall_dollars'))}
<h3>8.6 Accuracy by test month</h3>{img('m5_8_by_month')}{tab(R['bym'], pct=tuple(c for c in R['bym'].columns if c.startswith('precision_')))}
<h3>8.7 Calibration</h3>{img('m5_8_calibration')}{tab(R['cal'], pct=('mean_predicted', 'actual'))}<h4>Simple model</h4>{tab(R['cal_simple'], pct=('mean_predicted', 'actual'))}
<h3>8.8 When signals fire</h3><p>Event time describes leavers after the fact; queue time is what an RM experiences.</p>
{img('m5_9a_onset_heatmap')}{tab(R['tim'], pct=('coverage',))}<h4>Model view</h4>{img('m5_9b_total_gap')}{tab(R['model_view'])}
<p>Largest drivers 6 months out: {', '.join(R['lead6'])}. 1 month out: {', '.join(R['lead1'])}.</p>
<h4>Queue time</h4><p>First alert of any kind:</p>{tab(R['lead'], pct=('share_clients', 'share_dollars'), money=('capped',))}
<p>Sustained alert (first of ≥ 2 consecutive months on the list):</p>{tab(R['lead_sus'], pct=('share_clients', 'share_dollars'), money=('capped',))}{tab(R['reason_sets'], pct=('share',), money=('capped',))}
<h3>8.9 Savings against calls</h3>{img('m5_8_savings_vs_calls')}{tab(HVd, pct=('share',), money=('reached_per_year', 'departing_per_year'))}
{tab(R['val'], pct=('share_of_departing', 'share_of_dollars'), money=tuple(['dollars_reached_year'] + [f'save_{int(s * 100)}pct_year' for s in SAVE_RATES]))}
<h3>8.10 Leavers and stayers</h3>{img('m5_9d_leavers_vs_stayers')}{tab(R['ls'])}
<h3>8.11 Examples for RM managers</h3><p>Chosen by rule; alert = sustained alert; departing examples passed the §7 check and had 12 steady
months before the alert (except <i>new money that left</i>). Balances in dollars; change signals with ▼/▲ where beyond the stayers' threshold.
Review with the client's RM before wider use.</p>""" +
"".join(f"<h4>Example {e['n']} · {e['kind']} · client {e.get('shown_id', e['client'])}</h4><p>{_html.escape(e['text'])}</p>{img(e['chart'])}"
        f"{tab(e['table'], pct=('balance_pct_normal', 'score'), money=('balance', 'normal'))}" for e in R["examples"]) + NEWSEC + "</section>")
S.append(f"""<section><h2>9 · Predictions written before the run</h2>{tab(R['preds'], pct=('confidence',))}</section>
<section><h2>10 · Limitations</h2><ul><li><b>The save rate is unmeasured</b>; savings are illustrations until a pilot with a holdout.</li>
<li><b>The §7 check is a rule, not a review</b>; flagged clients need an RM's confirmation, so the headline keeps them and the exclusion is shown as a sensitivity row.</li>
<li><b>Identity is current-state:</b> a customer's power id and relationship are today's.</li>
<li><b>Few test months</b> ({R['n_test']}): the data start in 2024 and every label needs six confirmed months.</li>
<li><b>Event-time onsets describe leavers after the fact</b>; they are not detection leads.</li>
<li><b>Industry is known only for PNC counterparties</b> (NAICS from the customer dimension); external counterparties are grouped by name, so
processor dollars outside PNC are a lower bound.</li>
<li><b>The destination bank is a rule on observed payments</b>: money moved by wire to an account under another name, or through a processor,
shows no bank; processors also rotate among their own banks.</li>
<li><b>The 2026 check is partial</b>: the latest months have one to five months of follow-up, and the last {len(R.get('fwd_pending', [])) if R.get('fwd_pending') is not None else 0}
months none yet.</li></ul></section>
<section><h2>11 · Next steps</h2><ul><li>RM review of the largest departures and the example clients.</li>
<li>A pilot with a randomised holdout to measure the save rate; start from the current list (§8.15), with the likely destination bank as talking point.</li>
<li>Keep the network features only if the dollars-reached interval (§8.12) stays above zero on the next refresh.</li>
<li>Rerun the 2026 check as follow-up accumulates; the cache makes a rerun cheap.</li></ul></section>
<section><h2>Appendix · Parameters</h2>{tab(pd.DataFrame([[k, str(v)] for k, v in [("H", H), ("CONFIRM_M", CONFIRM_M), ("NORM_WIN / MIN_OBS", f"{NORM_WIN} / {NORM_MIN_OBS}"),
 ("money still here", f"{MSH_FRAC} × normal, ≥ ${MSH_FLOOR:,}"), ("balance exit", f"< {BAL_EXIT_FRAC} × normal for {BAL_EXIT_HOLD} months"),
 ("NEG_RATE", NEG_RATE), ("L2 chosen", R['L2']), ("tree depth chosen", R['depth']), ("CORR_MAX", CORR_MAX), ("CALLS", CALLS),
 ("ALPHAS", ALPHAS), ("SAVE_RATES", SAVE_RATES), ("LIFT_BAR", LIFT_BAR), ("N_AUDIT", N_AUDIT), ("tree depth, model-4 features", R.get('depth_base')),
 ("NET_VERSION / signature", f"{NET_VERSION} / {R['net_signature']}"), ("NAICS_TOPK", NAICS_TOPK), ("NB_BASE_M / KNOWN_LAG", f"{NB_BASE_M} / {KNOWN_LAG}"),
 ("DEST_MIN / DEST_LEADS / DEST_TOPG", f"{DEST_MIN} / {DEST_LEADS} / {DEST_TOPG}")]], columns=["parameter", "value"]))}</section>""")
CSS = """body{font-family:Arial,Helvetica,sans-serif;max-width:1180px;margin:24px auto;padding:0 18px;color:#222;line-height:1.45}
h1{font-size:26px;margin-bottom:4px}.sub{color:#666;margin-top:0}h2{border-bottom:2px solid #1f4e79;padding-bottom:4px;margin-top:34px}
h3{color:#1f4e79;margin-top:22px}table.t{border-collapse:collapse;font-size:12px;margin:8px 0;display:block;overflow-x:auto}
table.t th{background:#1f4e79;color:#fff;padding:4px 8px;text-align:left}table.t td{padding:3px 8px;border-bottom:1px solid #e3e3e3}
ol.find li{margin:8px 0}.box{background:#eef4fa;border-left:4px solid #1f4e79;padding:10px 14px;margin:10px 0}.box.warn{background:#fff4e5;border-color:#e65100}"""
REPORT = OUT / "PKG_Attrition_v13_m5_report.html"
REPORT.write_text(f"<!doctype html><html><head><meta charset='utf-8'><title>PKG attrition v13 · model 5</title><style>{CSS}</style></head>"
                  f"<body>{''.join(S)}<p style='color:#999;font-size:11px'>Internal — PNC Treasury Management, DSI</p></body></html>",
                  encoding="utf-8")
B.kv([("report", str(REPORT.resolve())), ("size", f"{REPORT.stat().st_size / 1e6:,.1f} MB"), ("CSVs and charts", str(OUT.resolve())), ("figure data", str(FIG.resolve())), ("results bundle", "m5_results.pkl"),
      ("total wall time", f"{(time.time() - T0) / 60:,.1f} min")], "12b · Written")
import pickle
R["sus_ids"] = None
with open(OUT / "m5_results.pkl", "wb") as fh:          # everything the report uses, for rebuilding it later
    pickle.dump({"R": {k: v for k, v in R.items() if k != "excl_ids"}, "checks": CHECKS,
                 "charts": {k: str(v) for k, v in CHARTS.items()}}, fh)
for d_ in (X, FEAT, MF, AMP, CLI, CMAP, DIMP): d_.unpersist()
B.show(t0)